# Phase 2 running notebook: IADrift checks on development images (SR ×4)

All checks use **development images only** and the **correct (oracle) label**. No test image is used. HR is used only for scoring, except in Check 5's representation-error diagnostics, which fit HR on purpose and are diagnostics, never methods.

| Check | Question | Training in this notebook? |
|---|---|---|
| 1 | Post-processing controls: is the fusion gain from the prior or from measurement consistency? | no (cached outputs) |
| 2 | θ-only adaptation from CSGM (2,000 steps): before vs after | no (cached runs from notebook 05) |
| 3 | Generator diagnostics: random samples at several CFG values, ε statistics, patch-grid spectrum | no (sampling only) |
| 4 | ε sphere projection for CSGM vs the saved no-projection runs | yes, CSGM (cached after the first run) |
| 5 | Representation error: CSGM fitted to HR (R1) and in-range SR (R3) | yes, CSGM (cached after the first run) |

**LPIPS** (AlexNet backbone, `lpips` 0.1.4, CPU torch) was added on 2026-10-04 to every table: lower is better, 0 = identical. Images are clipped to [-1, 1] before LPIPS.

---
# Check 1: Post-processing controls

**Development images only, correct (oracle) label only. No optimization is run in this notebook**: every reconstruction is loaded from caches written by `notebooks/02_iadrift_dev_tuning.ipynb`, and only cheap post-processing is applied on top.

## Why this notebook exists

Notebook 02 found that **IADrift + fusion** (bicubic below 0.0625 and above 0.125 cycles/pixel, IADrift in the mid band 0.0625–0.125) beats bicubic by +0.18 dB on 4 development images. It also found a warning sign: **CSGM + back-projection** has a *higher* mid-band SNR (12.6 dB) than any raw IADrift output (≤ 11.1 dB).

So the fusion gain has two possible explanations:
- **(H1) the generator prior:** IADrift's mid band is better because the adapted generator puts the right content there;
- **(H2) measurement consistency alone:** any image that reproduces y exactly gets a good mid band, because the mid band is (mostly) determined by y. Then a prior-free method would do just as well.

These are hypotheses; the controls below are designed to tell them apart.

The **test-set section** is at the end, before the summary, and is clearly separated: it only loads cached test results and must never be used to tune anything.


**Figures.** Every side-by-side figure includes a plain **bicubic** upsampling of the image's 64×64 measurement as a reference panel (added 2026-10-06).

---
# Model overview: the drift generator and what we did with it

**The generator (`pixel_B_sota`) in one paragraph.** It produces a 256×256×3 image from **four inputs in one forward pass**: there is no iterative denoising as in diffusion models. "Drift" refers to how it was **trained** (a drift loss that pulls generated images toward real images of the class and pushes them from unconditional ones, measured in the features of pretrained encoders, a MAE-ResNet and ConvNeXt); those encoders are not part of the generator and play no role in our experiments.
- **Conditioning (global).** The class label c (1 of 1,000), the 32 discrete **noise labels** z (each 0–63) and the CFG scale w (we use 1.0) are each turned into a 768-d vector by embedding tables (and, for w, a small MLP) and **summed** into one conditioning vector `cond`. It acts on the image twice: as 16 extra **conditioning tokens** that every patch can attend to, and through **AdaLN** (per-block scale, shift and gate computed from `cond`) in all 12 transformer blocks and the output layer.
- **Noise (local).** The noise image ε (256×256×3 Gaussian) is cut into 256 patches of 16×16×3 values; each patch becomes one token through one linear layer (plus a position code). There is no separate encoder network.
- **Transformer.** 272 tokens (16 conditioning + 256 patch) × 768 channels pass through 12 blocks (self-attention with 12 heads, QK-norm and RoPE; SwiGLU MLP; RMSNorm; AdaLN from `cond`).
- **Output.** A final linear layer maps each patch token to its 16×16×3 pixels; the 16 conditioning tokens are dropped and the 256 tiles are placed side by side: x = f_θ(ε, z, c) in [−1, 1]. θ is all 134M weights.
- **Why it matters for us.** ε is local (one patch → one tile; tiles that don't agree give the 16-pixel seams), while z is a global code that decides the scene (Check 8).

**What we did (Figure 2).** Super-resolution ×4: given only the 64×64 measurement y = A(HR) (antialiased bicubic downsampling), find an image whose downsampled version matches y, by minimizing the LR loss ‖y − A f_θ(ε, z, c)‖² with the correct class label. **CSGM** optimizes ε with the generator frozen; **IADrift** (IAGAN-style) also adapts θ; the noise-label search, soft z and swaps choose or relax z; **back-projection** makes any output reproduce y exactly; the **ensemble** averages several reconstructions before BP.

In [ ]:
from pathlib import Path

from IPython.display import Image
Image(filename=str(Path.home() / "work" / "IADrift" / "figures" / "model_diagram_generator.png"))

In [ ]:
from pathlib import Path

from IPython.display import Image
Image(filename=str(Path.home() / "work" / "IADrift" / "figures" / "model_diagram_methods.png"))

## What exactly was done

**Images.** The same 4 development images as notebook 02: 8117 (streetcar), 3939 (bonnet), 403 (crane), 347 (spoonbill). `check_dev` refuses anything in the test split.

**Inputs (all loaded from disk, nothing re-optimized):**
- `y`: the stored 64×64 LR measurement (`imageNet_x4.npy`), and `HR`: the 256×256 original rebuilt with `load_hr256` (used **only for scoring**).
- **bicubic** = `bicubic_up(y)`.
- **CSGM** = cached x̂ from `csgm_dev_5000steps_lr0.02/` (frozen generator, Adam lr 0.02, 5,000 steps, 2 restarts). Of the 2 restarts we keep the one with the **lowest final LR loss** (never HR), exactly as in notebook 02 and the test benchmark.
- **IADrift** = cached step-1,000 output from `iadrift_dev_tuning_v1/` for each of the 6 settings θ lr ∈ {1e-6, 3e-6, 1e-5} × ε lr ∈ {0, 1e-3}. The main setting is **θ lr 1e-5, ε lr 1e-3** (the one where fusion was best in notebook 02). **Only step-1,000 outputs are cached**, so the early-stopped (step-50) IADrift is *not* covered here.

**Post-processing operators:**
1. **BP (back-projection):** $x_{BP} = x + A^T (A A^T)^{-1} (y - A x)$, the closest image to $x$ that reproduces $y$ exactly. Same code as notebook 02 (matrix-free, conjugate gradients). Check: LR loss of every BP output ≈ 0.
2. **fusion:** `fuse_with_bicubic(x, bicubic)` from `scripts/freq_metrics.py`: bicubic in [0, 0.0625) and [0.125, ∞), $x$ in [0.0625, 0.125). Band edges unchanged from notebook 02 (still chosen after looking at development data).
3. **BP + fusion:** BP first, then fusion: `fuse_with_bicubic(BP(x), bicubic)`. So the output is bicubic everywhere except the mid band, which comes from the measurement-consistent version of $x$.

**The new controls** (the rest are recomputed for a like-for-like table):

| Row | What it tests |
|---|---|
| **bicubic + BP** | A prior-free, measurement-consistent image. If H2 holds, its mid band should be as good as IADrift's. |
| **bicubic + BP + fusion** | The prior-free counterpart of "IADrift + fusion": same recipe, but the mid band comes from bicubic + BP instead of any generator. This is the key control. |
| **CSGM + BP + fusion** | Mid band from the frozen-generator prior after enforcing the measurement. |
| **IADrift + BP + fusion** | Mid band from the adapted generator after enforcing the measurement. |

**Reading the result.**
- If **IADrift + BP + fusion > bicubic + BP + fusion** and **> CSGM + BP + fusion** consistently, the prior adds something in the mid band (supports H1).
- If they are equal or the prior-free control wins, the fusion gain is measurement consistency (supports H2), and "IADrift + fusion" is not evidence for IADrift.

**Metrics** (same as before): HR PSNR in dB with data range 2 (unclipped = main, clipped to [-1, 1] also shown), LR loss ‖y − A x‖², and band SNR (low / mid / high / very high, `freq_metrics.band_snr_db`). The mid band is the one that matters here. All images are numpy arrays already produced by the eager path, so there's no bf16 compilation issue in this notebook.

**Consistency checks** against notebook 02's caches: recomputed *IADrift + BP* and *IADrift + fusion* are compared to the arrays stored there.

**Caveats.** 4 images only, no significance tests are meaningful at n = 4; per-image values are shown so you can see consistency. Fusion band edges were picked on these same images.

**Output.** Metrics are written to `~/work/datasets/ImageNetV2/iadrift_dev_postproc_controls_v1/metrics.json`.

## Setup

In [ ]:
import os
import sys
from pathlib import Path

# No θ-gradients are computed in this notebook (Check 2 only loads cached runs), so the
# `platform` allocator is deliberately NOT set. Under it, CSGM optimization follows a
# different numerical path and does not reproduce the saved CSGM runs (verified 2026-10-04;
# Check 4 re-tests this before using the saved runs as the no-projection baseline).
SCRIPTS = Path.home() / "work" / "IADrift" / "scripts"
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

import io
import json
from time import perf_counter

import lpips
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Markdown, display
from jax.scipy.sparse.linalg import cg

# Project helpers (importing run_sr4_csgm_test also loads the generator, used from Check 3 on).
from freq_metrics import (BANDS, LR_NYQUIST, RADIUS, WINDOW, band_filter, band_snr_db, fuse_with_bicubic,
                          radial_snr_db)
from run_sr4_csgm_test import (
    A_sr4, DATA_DIR, FROZEN_CONFIG, TRACE_EVERY, atomic_write_bytes, bicubic_up, generate, initialize, jax, jnp,
    load_hr256, load_index, load_split, measurement_loss, model, optax, params, psnr,
)
from run_sr4_iadrift_dev import CSGM_CACHE, check_dev
from run_sr4_iadrift_smoke import class_name

BAND_NAMES = [name for name, *_ in BANDS]
MID_LO, MID_HI = 0.0625, LR_NYQUIST
torch.set_num_threads(4)
LPIPS_NET = lpips.LPIPS(net="alex", verbose=False).eval()
%matplotlib inline
print("Devices:", jax.devices(), "| torch", torch.__version__, "(CPU, LPIPS only)")

## Back-projection and metrics

Identical to notebook 02, plus clipped PSNR and LPIPS (lower is better).

In [ ]:
_A_transpose = jax.linear_transpose(A_sr4, jnp.zeros((1, 256, 256, 3), jnp.float32))


@jax.jit
def _backproject(x, y):
    def AT(v):
        return _A_transpose(v)[0]
    u, _ = cg(lambda v: A_sr4(AT(v)), y - A_sr4(x), tol=1e-6, maxiter=500)
    return x + AT(u)


def backproject(x, y):
    # x: [256, 256, 3] numpy; y: [1, 64, 64, 3]. Returns the back-projected image.
    return np.asarray(_backproject(jnp.asarray(x, jnp.float32)[None], y)[0])


def lr_loss(x, y):
    return float(jnp.sum((A_sr4(jnp.asarray(x, jnp.float32)[None]) - y) ** 2))


def _to_torch(x):
    return torch.from_numpy(np.ascontiguousarray(np.clip(x, -1, 1).transpose(2, 0, 1)[None])).float()


def lpips_dist(x, ref):
    # LPIPS (AlexNet) between two [256, 256, 3] images in [-1, 1]; lower is better.
    with torch.no_grad():
        return float(LPIPS_NET(_to_torch(x), _to_torch(ref)))


def evaluate(x, hr, y):
    return {"psnr": psnr(hr, x), "psnr_clipped": psnr(hr, np.clip(x, -1, 1)), "lpips": lpips_dist(x, hr),
            "lr_loss": lr_loss(x, y), "band_snr_db": band_snr_db(x, hr)}

## Load cached reconstructions and apply the post-processing

For every image: pick the CSGM restart by lowest LR loss, load all 6 IADrift settings, then build every row of the table. The consistency checks against notebook 02's cached arrays are printed at the end.

In [ ]:
DEV_IMAGES = [8117, 3939, 403, 347]
CONFIGS = [(lr_theta, lr_eps) for lr_theta in (1e-6, 3e-6, 1e-5) for lr_eps in (0.0, 1e-3)]
MAIN_CFG = (1e-5, 1e-3)
TUNING_DIR = DATA_DIR / "iadrift_dev_tuning_v1"
OUT_DIR = DATA_DIR / "iadrift_dev_postproc_controls_v1"

index_rows = load_index()
labels = np.array([int(row["label"]) for row in index_rows])
train_indices, test_indices = load_split(labels)
check_dev(DEV_IMAGES, train_indices, test_indices)  # refuses test images
lr_all = np.load(DATA_DIR / "imageNet_x4.npy", mmap_mode="r")


def load_csgm(i, label):
    # Cached oracle-label CSGM restarts; keep the lowest final LR loss. Never re-runs CSGM.
    restarts = []
    for r in range(FROZEN_CONFIG["num_restarts"]):
        path = CSGM_CACHE / f"img{i:05d}_oracle_r{r}.npz"
        assert path.exists(), f"missing cache {path}"
        with np.load(path) as data:
            m = json.loads(str(data["metrics_json"]))
            assert m["class_label"] == label
            restarts.append((m["final_loss"], r, data["x_hat"]))
    loss, r, x_hat = min(restarts, key=lambda t: (t[0], t[1]))
    return x_hat, r


def with_post(name, x, bic, y):
    # name, name + BP, name + fusion, name + BP + fusion
    x_bp = backproject(x, y)
    return {name: x, f"{name} + BP": x_bp,
            f"{name} + fusion": fuse_with_bicubic(x, bic),
            f"{name} + BP + fusion": fuse_with_bicubic(x_bp, bic)}


images, checks = {}, []
for i in DEV_IMAGES:
    label = int(labels[i])
    hr = load_hr256(DATA_DIR / index_rows[i]["relative_path"])
    y = jnp.asarray(lr_all[i:i + 1], jnp.float32)
    bic = bicubic_up(lr_all[i])
    csgm, restart = load_csgm(i, label)
    outputs = {}
    outputs.update(with_post("bicubic", bic, bic, y))
    del outputs["bicubic + fusion"]  # identical to bicubic by construction
    outputs.update(with_post("CSGM", csgm, bic, y))
    iad = {}
    for cfg in CONFIGS:
        with np.load(TUNING_DIR / f"img{i:05d}_lrt{cfg[0]:g}_lre{cfg[1]:g}.npz") as data:
            cached = {k: data[k] for k in ("IADrift", "IADrift + BP", "IADrift + fusion")}
        iad[cfg] = with_post("IADrift", cached["IADrift"], bic, y)
        for k in ("IADrift + BP", "IADrift + fusion"):
            checks.append((i, cfg, k, float(np.max(np.abs(iad[cfg][k] - cached[k])))))
    images[i] = {
        "label": label, "hr": hr, "y": y, "csgm_restart": restart, "outputs": outputs, "iadrift": iad,
        "metrics": {n: evaluate(x, hr, y) for n, x in outputs.items()},
        "iad_metrics": {cfg: {n: evaluate(x, hr, y) for n, x in o.items()} for cfg, o in iad.items()},
    }
    print(f"image {i} ({class_name(label)}): CSGM restart r{restart} (lowest LR loss)")

worst = max(checks, key=lambda c: c[3])
print(f"\nConsistency with notebook 02 caches: max |recomputed - cached| over {len(checks)} arrays = {worst[3]:.2e} "
      f"(image {worst[0]}, setting {worst[1]}, {worst[2]})")
bp_losses = [m["lr_loss"] for im in images.values() for d in [im["metrics"], *im["iad_metrics"].values()]
             for n, m in d.items() if n.endswith("+ BP")]
print(f"LR loss after BP: max {max(bp_losses):.2e} over {len(bp_losses)} outputs (should be ~0)")

## Results 1: main table (mean over the 4 development images, IADrift setting θ lr 1e-5, ε lr 1e-3)

- **PSNR** (unclipped, main) and **clipped** are in dB; **Δ bic.** is the mean paired difference to bicubic; **> bic.** counts images where the method beats bicubic.
- **LR loss** = ‖y − A x‖² (measurement fit; ≈ 0 after BP; fused outputs are not exactly consistent because the low/high bands come from bicubic).
- **Band SNR** in dB, higher is better. **Mid** (0.0625–0.125) is the band the fusion takes from the method, so it's the column that decides H1 vs H2.
- Rows in **bold** are the new controls.

In [ ]:
NEW = {"bicubic + BP", "bicubic + BP + fusion", "CSGM + BP + fusion", "IADrift + BP + fusion"}
ROW_ORDER = ["bicubic", "bicubic + BP", "bicubic + BP + fusion",
             "CSGM", "CSGM + BP", "CSGM + fusion", "CSGM + BP + fusion",
             "IADrift", "IADrift + BP", "IADrift + fusion", "IADrift + BP + fusion"]


def per_image(name, cfg=MAIN_CFG):
    return [images[i]["iad_metrics"][cfg][name] if name.startswith("IADrift") else images[i]["metrics"][name]
            for i in DEV_IMAGES]


bic_psnr = np.array([m["psnr"] for m in per_image("bicubic")])


def table(rows, cfg=MAIN_CFG):
    lines = ["| Method | PSNR | clipped | Δ bic. | > bic. | LPIPS ↓ | LR loss | " + " | ".join(f"SNR {b}" for b in BAND_NAMES) + " |",
             "|---|---:|---:|---:|---:|---:|---:|" + "---:|" * len(BAND_NAMES)]
    for name in rows:
        ms = per_image(name, cfg)
        p = np.array([m["psnr"] for m in ms])
        label = f"**{name}**" if name in NEW else name
        lines.append(f"| {label} | {p.mean():.2f} | {np.mean([m['psnr_clipped'] for m in ms]):.2f} | "
                     f"{np.mean(p - bic_psnr):+.2f} | {int(np.sum(p > bic_psnr + 1e-9))}/{len(p)} | "
                     f"{np.mean([m['lpips'] for m in ms]):.3f} | {np.mean([m['lr_loss'] for m in ms]):.2f} | "
                     + " | ".join(f"{np.mean([m['band_snr_db'][b] for m in ms]):.2f}" for b in BAND_NAMES) + " |")
    return "\n".join(lines)


display(Markdown(table(ROW_ORDER)))

## Results 2: per image

HR PSNR (dB), mid-band SNR (dB) and LPIPS (lower is better) for each image. Consistency across the 4 images matters more than the mean.

In [ ]:
GETTERS = {"psnr": ("HR PSNR (dB)", lambda m: m["psnr"], "{:.2f}"),
           "mid": ("Mid-band SNR (dB)", lambda m: m["band_snr_db"]["mid"], "{:.2f}"),
           "lpips": ("LPIPS (lower is better)", lambda m: m["lpips"], "{:.3f}")}
for key, (title, get, fmt) in GETTERS.items():
    lines = ["| Method | " + " | ".join(f"{i} ({class_name(images[i]['label'])})" for i in DEV_IMAGES) + " | mean |",
             "|---|" + "---:|" * (len(DEV_IMAGES) + 1)]
    for name in ROW_ORDER:
        vals = [get(m) for m in per_image(name)]
        label = f"**{name}**" if name in NEW else name
        lines.append(f"| {label} | " + " | ".join(fmt.format(v) for v in vals) + f" | {fmt.format(np.mean(vals))} |")
    display(Markdown(f"**{title}**\n\n" + "\n".join(lines)))

## Results 3: the decisive paired comparisons

Each row is a paired difference A − B per image, then the mean and how many of the 4 images favour A (higher PSNR / SNR, **lower** LPIPS). Same IADrift setting as above.

In [ ]:
PAIRS = [
    ("IADrift + BP + fusion", "bicubic + BP + fusion"),
    ("IADrift + BP + fusion", "CSGM + BP + fusion"),
    ("CSGM + BP + fusion", "bicubic + BP + fusion"),
    ("IADrift + fusion", "bicubic + BP + fusion"),
    ("IADrift + BP + fusion", "IADrift + fusion"),
    ("bicubic + BP + fusion", "bicubic"),
    ("IADrift + BP", "bicubic + BP"),
    ("IADrift", "CSGM"),
]
for key, (title, get, fmt) in GETTERS.items():
    sign = -1 if key == "lpips" else 1  # LPIPS: lower is better
    lines = ["| A − B | " + " | ".join(str(i) for i in DEV_IMAGES) + " | mean | A better |",
             "|---|" + "---:|" * (len(DEV_IMAGES) + 2)]
    for a, b in PAIRS:
        d = np.array([get(ma) - get(mb) for ma, mb in zip(per_image(a), per_image(b))])
        f = "{:+.3f}" if key == "lpips" else "{:+.2f}"
        lines.append(f"| {a} − {b} | " + " | ".join(f.format(v) for v in d) + f" | {f.format(d.mean())} | {int(np.sum(sign * d > 0))}/4 |")
    display(Markdown(f"**{title}: differences**\n\n" + "\n".join(lines)))

## Results 4: all six IADrift settings

The same IADrift rows for every (θ lr, ε lr) setting, to check the conclusion doesn't depend on the main setting. Baselines (bicubic, CSGM) don't depend on the setting; see the main table.

In [ ]:
IAD_ROWS = ["IADrift", "IADrift + BP", "IADrift + fusion", "IADrift + BP + fusion"]
lines = ["| θ lr | ε lr | Method | PSNR | Δ bic. | > bic. | LPIPS ↓ | SNR mid | SNR high |",
         "|---:|---:|---|---:|---:|---:|---:|---:|---:|"]
for cfg in CONFIGS:
    for name in IAD_ROWS:
        ms = per_image(name, cfg)
        p = np.array([m["psnr"] for m in ms])
        lines.append(f"| {cfg[0]:g} | {cfg[1]:g} | {name} | {p.mean():.2f} | {np.mean(p - bic_psnr):+.2f} | "
                     f"{int(np.sum(p > bic_psnr))}/4 | {np.mean([m['lpips'] for m in ms]):.3f} | "
                     f"{np.mean([m['band_snr_db']['mid'] for m in ms]):.2f} | "
                     f"{np.mean([m['band_snr_db']['high'] for m in ms]):.2f} |")
for ref_name in ("bicubic", "bicubic + BP", "bicubic + BP + fusion", "CSGM"):
    ref = per_image(ref_name)
    p = np.array([m["psnr"] for m in ref])
    lines.append(f"| — | — | **{ref_name}** (reference) | {p.mean():.2f} | {np.mean(p - bic_psnr):+.2f} | "
                 f"{int(np.sum(p > bic_psnr))}/4 | {np.mean([m['lpips'] for m in ref]):.3f} | "
                 f"{np.mean([m['band_snr_db']['mid'] for m in ref]):.2f} | {np.mean([m['band_snr_db']['high'] for m in ref]):.2f} |")
display(Markdown("\n".join(lines)))

## Results 5: radial SNR curves

Mean over the 4 images of SNR per radial frequency bin (`radial_snr_db`), for the measurement-consistent (BP) versions of each method plus plain bicubic. The shaded region is the mid band that fusion takes from the method; the dashed line is the LR Nyquist limit (0.125).

In [ ]:
CURVES = ["bicubic", "bicubic + BP", "CSGM + BP", "IADrift + BP", "IADrift"]
fig, ax = plt.subplots(figsize=(9, 4.5))
for k, name in enumerate(CURVES):
    curves = []
    for i in DEV_IMAGES:
        x = images[i]["iadrift"][MAIN_CFG][name] if name.startswith("IADrift") else images[i]["outputs"][name]
        f, s = radial_snr_db(x, images[i]["hr"])
        curves.append(s)
    ax.plot(f, np.mean(curves, axis=0), color=f"C{k}", label=name, linestyle=":" if name == "IADrift" else "-")
ax.axvspan(MID_LO, MID_HI, color="gray", alpha=0.15, label="mid band (fused)")
ax.axvline(LR_NYQUIST, color="k", linestyle="--", linewidth=1)
ax.set(xlabel="radial frequency (cycles per HR pixel)", ylabel="SNR (dB)", xlim=(0, 0.3),
       title=f"Mean radial SNR, 4 development images (IADrift θ lr {MAIN_CFG[0]:g}, ε lr {MAIN_CFG[1]:g})")
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
plt.show()

## Results 6: what the mid band looks like

For each image: the HR content in the mid band (0.0625–0.125, via `band_filter`), and the mid-band content each measurement-consistent method contributes to its fused output. Titles give that method's mid-band SNR. Shown with a common gray scale per image (mid-band content is signed, so 0 = mid gray).

In [ ]:
for i in DEV_IMAGES:
    im = images[i]
    panels = [("HR (truth)", im["hr"], None),
              ("bicubic", im["outputs"]["bicubic"], im["metrics"]["bicubic"]),
              ("bicubic + BP", im["outputs"]["bicubic + BP"], im["metrics"]["bicubic + BP"]),
              ("CSGM + BP", im["outputs"]["CSGM + BP"], im["metrics"]["CSGM + BP"]),
              ("IADrift + BP", im["iadrift"][MAIN_CFG]["IADrift + BP"], im["iad_metrics"][MAIN_CFG]["IADrift + BP"])]
    bands = [band_filter(x, MID_LO, MID_HI).mean(axis=-1) for _, x, _ in panels]
    scale = 3 * np.std(bands[0])
    fig, axes = plt.subplots(1, len(panels), figsize=(4 * len(panels), 4.3))
    for ax, (title, _, m), band in zip(axes, panels, bands):
        ax.imshow(band, cmap="gray", vmin=-scale, vmax=scale)
        sub = "" if m is None else f"\nmid SNR {m['band_snr_db']['mid']:.2f} dB"
        ax.set_title(f"{title}{sub}", fontsize=9)
        ax.axis("off")
    fig.suptitle(f"Development image {i} ({class_name(im['label'])}): mid-band content (RGB mean)", fontsize=10, y=1.04)
    fig.tight_layout()
    plt.show()

## Save metrics

In [ ]:
record = {
    "dev_images": DEV_IMAGES, "main_setting": list(MAIN_CFG),
    "csgm_restart": {str(i): images[i]["csgm_restart"] for i in DEV_IMAGES},
    "metrics": {str(i): {**images[i]["metrics"],
                         **{f"{n} (lrt {c[0]:g}, lre {c[1]:g})": m for c, d in images[i]["iad_metrics"].items() for n, m in d.items()}}
                for i in DEV_IMAGES},
}
OUT_DIR.mkdir(exist_ok=True)
atomic_write_bytes(OUT_DIR / "metrics.json", json.dumps(record, indent=1).encode())
print("Wrote", OUT_DIR / "metrics.json")

## Check 1 conclusions (4 development images, correct label, IADrift step-1,000 outputs; small sample)

**Observations**

1. **The fusion gain is measurement consistency, not the generator prior (supports H2, not H1).**
   - The prior-free control **bicubic + BP + fusion** (25.31 dB) beats **IADrift + BP + fusion** (25.15) and **CSGM + BP + fusion** (25.23) on **4/4 images**.
   - The order is the same in the mid band: bicubic + BP 14.3–14.4 dB > CSGM + BP 12.6 > IADrift + BP 11.4–11.5. IADrift's mid band is worse than the prior-free one on every image (by 0.6 to 4.8 dB).
   - This holds for **all 6 IADrift settings**: the best IADrift + BP + fusion is 25.16 dB, below 25.31.
2. **Plain bicubic + BP is the best method in this notebook: 25.43 dB, +0.49 dB over bicubic, 4/4 images.** Fusion slightly *hurts* it (−0.12 dB), because the plain bicubic low band replaces a better, measurement-consistent one.
3. **Plain bicubic is not measurement-consistent** (LR loss 9.5 under our antialiased `A_sr4`). BP alone adds +5.3 dB in the mid band and +19 dB in the low band to it.
4. **More adaptation makes IADrift's mid band worse once BP is applied.** CSGM + BP (the step-0 starting point) has 12.55 dB; IADrift + BP after 1,000 steps has 11.3–11.7 dB at every setting. Without BP, IADrift improves the mid band over CSGM (5.4 → 11.0 dB), which is what made it look good in notebook 02; BP gives CSGM that improvement and more.
5. **For IADrift, BP before fusion adds a little** (+0.03 dB, 4/4) and the fused outputs remain roughly consistent (LR loss about 5).
6. Checks passed: recomputed IADrift + BP / + fusion match notebook 02's cached arrays exactly (max difference 0), and every BP output has LR loss ≤ 6e-11.

**Interpretation (hypotheses, not tested here)**
- Under SR ×4 with this operator, the measured bands are best filled by the minimum-norm correction of a smooth image. The generator's content in the mid band is less accurate than "no content + exact measurement fit". IADrift's apparent mid-band strength in notebook 02 was mostly its good measurement fit.
- The mid band isn't fully determined even after BP (bicubic + BP is 14 dB, not about 50 like the low band), presumably because the antialiasing filter in `A_sr4` attenuates frequencies near the LR Nyquist limit. That attenuated part is where a prior *could* help; here none does.

**Not covered**
- Early-stopped IADrift (step 50–100): not cached. Notebook 02 shows IADrift's raw mid band rises with steps, so earlier outputs should be no better in the mid band, but that is an inference.
- Wrong-label conditions, more images, LPIPS / perceptual quality.

**Implications**
- "IADrift + fusion" should **not** be presented as evidence for IADrift; at most as a post-processing variant, always next to **bicubic + BP (+ fusion)**.
- **bicubic + BP** is a stronger, prior-free PSNR baseline than plain bicubic and should probably be added to every comparison (dev and test) — it's cheap.
- IADrift's case then rests on **perceptual quality and visual detail** (LPIPS, examples), not PSNR. That makes LPIPS more important.
- Whether IADrift's **high-band** content can be made useful (weight anchor, early stop) remains the open question for run 3.

**LPIPS addendum (added 2026-10-04; lower is better)**
- **On LPIPS the generator prior does help:** CSGM (0.406) beats bicubic (0.484), and **CSGM + BP (0.328) is the best of all Check 1 methods**, better than bicubic + BP (0.429) on **4/4 images**. PSNR ranks them the other way round (21.86 vs 25.43 dB).
- **IADrift makes LPIPS worse than its CSGM start:** 0.470 vs 0.406, worse on 4/4 images (by +0.002 to +0.124). BP doesn't change that (0.467).
- **All fused outputs look like bicubic to LPIPS** (0.46 for every fused variant, paired differences ≤ 0.005). Fusion is a PSNR-only device.


---
# Check 2: θ-only adaptation from CSGM (2,000 steps)

**Question.** Take the CSGM-best solution of each image, **freeze its noise ε**, and optimize **only the generator weights θ** for 2,000 steps. How does quality change from before (CSGM best) to after (CSGM + θ)?

This is IADrift with the ε learning rate set to exactly zero, run twice as long as in notebook 02.

**What exactly was done** (the runs themselves are in `notebooks/05_theta_only_dev.ipynb`; this section only loads and analyses their cached outputs, with no GPU training):
- **Images:** the same 4 development images as above, correct (oracle) label only.
- **Before = CSGM best:** the cached CSGM restart with the lowest final LR loss (never HR). Its ε and `noise_labels` are the starting point.
- **After = CSGM + θ:**
  - θ starts from the pretrained weights (fresh copy per run).
  - ε, `noise_labels` and the class label are **constants**; no ε optimizer exists.
  - Loss ‖y − A f_θ(ε_CSGM)‖², Adam on all 241 leaves (134M parameters), **2,000 steps**.
  - Two learning rates, each run separately from the same start: **θ lr 1e-6** and **θ lr 1e-5**.
- **Monitoring:** every 50 steps, LR loss, HR PSNR and band SNR through the compiled generator (evaluation only; slightly offset from eager because of bf16).
- **Final metrics:** eager generator output at step 2,000, scored with the same `evaluate` as above (including LPIPS). The HR-best step on the curves uses HR, so it's an **oracle diagnostic, not a stopping rule**.
- **Cache:** `~/work/datasets/ImageNetV2/theta_only_dev_v1/img#####_lrt{lr}_steps2000.npz`.

In [ ]:
# Check 2: θ-only adaptation from CSGM. Loads the cached runs from notebook 05 (no training here).
THETA_DIR = DATA_DIR / "theta_only_dev_v1"
THETA_LRS = [1e-6, 1e-5]
THETA_STEPS = 2_000

theta_runs = {}
for lr_theta in THETA_LRS:
    for i in DEV_IMAGES:
        path = THETA_DIR / f"img{i:05d}_lrt{lr_theta:g}_steps{THETA_STEPS}.npz"
        assert path.exists(), f"missing {path}: run notebooks/05_theta_only_dev.ipynb first"
        with np.load(path) as data:
            result, x = json.loads(str(data["result_json"])), data["x"]
        assert result["csgm_restart"] == images[i]["csgm_restart"]  # same CSGM start as above
        theta_runs[(lr_theta, i)] = {"trace": result["trace"], "x": x,
                                     "metrics": evaluate(x, images[i]["hr"], images[i]["y"])}

# Before/after table (eager, step 2,000).
THETA_ROWS = [("bicubic", None), ("bicubic + BP", None), ("CSGM", None)] + [("CSGM + θ", lr) for lr in THETA_LRS]


def theta_metrics(name, lr):
    return [theta_runs[(lr, i)]["metrics"] if lr else images[i]["metrics"][name] for i in DEV_IMAGES]


csgm_p = np.array([images[i]["metrics"]["CSGM"]["psnr"] for i in DEV_IMAGES])
lines = ["| Method | PSNR | clipped | Δ CSGM | > CSGM | LPIPS ↓ | LR loss | " + " | ".join(f"SNR {b}" for b in BAND_NAMES) + " |",
         "|---|---:|---:|---:|---:|---:|---:|" + "---:|" * len(BAND_NAMES)]
for name, lr in THETA_ROWS:
    ms = theta_metrics(name, lr)
    p = np.array([m["psnr"] for m in ms])
    label = f"**{name} (lr {lr:g})**" if lr else ("**CSGM best**" if name == "CSGM" else name)
    lines.append(f"| {label} | {p.mean():.2f} | {np.mean([m['psnr_clipped'] for m in ms]):.2f} | "
                 f"{np.mean(p - csgm_p):+.2f} | {int(np.sum(p > csgm_p + 1e-9))}/4 | "
                 f"{np.mean([m['lpips'] for m in ms]):.3f} | {np.mean([m['lr_loss'] for m in ms]):.2f} | "
                 + " | ".join(f"{np.mean([m['band_snr_db'][b] for m in ms]):.2f}" for b in BAND_NAMES) + " |")
display(Markdown("**Before vs after, mean over 4 development images (eager, step 2,000)**\n\n" + "\n".join(lines)))

lines = ["| Method | " + " | ".join(f"{i} ({class_name(images[i]['label'])})" for i in DEV_IMAGES) + " |",
         "|---|" + "---:|" * len(DEV_IMAGES)]
for name, lr in THETA_ROWS:
    label = f"{name} (lr {lr:g})" if lr else ("CSGM best" if name == "CSGM" else name)
    lines.append(f"| {label} | " + " | ".join(f"{m['psnr']:.2f} / {m['lpips']:.3f} / {m['lr_loss']:.1f}"
                                            for m in theta_metrics(name, lr)) + " |")
display(Markdown("**Per image: HR PSNR dB / LPIPS / LR loss**\n\n" + "\n".join(lines)))

# Curves over the 2,000 steps (compiled path; mean over images).
panels = [("lr_loss", "LR loss (log)"), ("psnr", "HR PSNR (dB)")] + [(b, f"SNR {b} (dB)") for b in BAND_NAMES]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for (key, title), ax in zip(panels, axes.flat):
    for k, lr in enumerate(THETA_LRS):
        traces = [theta_runs[(lr, i)]["trace"] for i in DEV_IMAGES]
        ax.plot(traces[0]["step"], np.mean([t[key] for t in traces], axis=0), color=f"C{k}", label=f"θ lr {lr:g}")
    if key == "lr_loss":
        ax.set_yscale("log")
    else:
        for base, color in (("bicubic", "k"), ("bicubic + BP", "gray")):
            ms = [images[i]["metrics"][base] for i in DEV_IMAGES]
            ax.axhline(np.mean([m["psnr"] if key == "psnr" else m["band_snr_db"][key] for m in ms]),
                       color=color, linestyle="--", linewidth=1, label=base)
    ax.set(title=title, xlabel="θ-only step")
    ax.grid(alpha=0.3)
axes.flat[1].legend(fontsize=8)
fig.suptitle("θ-only adaptation from CSGM best (step 0), mean over 4 development images", fontsize=11)
fig.tight_layout()
plt.show()

lines = ["| θ lr | PSNR step 0 | HR-best step (oracle) | PSNR there | PSNR step 2,000 | LR loss step 0 → 2,000 |",
         "|---:|---:|---:|---:|---:|---:|"]
for lr in THETA_LRS:
    traces = [theta_runs[(lr, i)]["trace"] for i in DEV_IMAGES]
    mp, ml = np.mean([t["psnr"] for t in traces], axis=0), np.mean([t["lr_loss"] for t in traces], axis=0)
    k = int(np.argmax(mp))
    lines.append(f"| {lr:g} | {mp[0]:.2f} | {traces[0]['step'][k]} | {mp[k]:.2f} | {mp[-1]:.2f} | {ml[0]:.1f} → {ml[-1]:.2f} |")
display(Markdown("**Mean curve summary (compiled path)**\n\n" + "\n".join(lines)))

# Side by side: full images (top) and an 80x80 detail crop (bottom). Helpers reused by Checks 3-5.
CROP = 80


def detail_crop(hr):
    # Crop with the most HR gradient energy; chosen on HR for display only.
    g = np.abs(np.diff(hr.mean(-1), axis=0))[:, :-1] + np.abs(np.diff(hr.mean(-1), axis=1))[:-1, :]
    pos = [(r, c) for r in range(0, 256 - CROP, 16) for c in range(0, 256 - CROP, 16)]
    return max(pos, key=lambda p: g[p[0]:p[0] + CROP, p[1]:p[1] + CROP].sum())


to_img = lambda x: (np.clip(x, -1, 1) + 1) / 2


def side_by_side(reference, ref_title, shown, suptitle=None):
    # shown: [(title, image, metric dict or None)]; top row full images, bottom row the same detail crop.
    # A plain bicubic panel (bicubic upsampling of A_sr4(reference), i.e. of the figure's LR measurement) is
    # added as the first panel unless the figure already shows plain bicubic.
    if not any(t.split("\n")[0].strip() == "bicubic" for t, _, _ in shown):
        y_ref = A_sr4(jnp.asarray(reference, jnp.float32)[None])
        bic_ref = bicubic_up(np.asarray(y_ref[0]))
        shown = [("bicubic", bic_ref, evaluate(bic_ref, reference, y_ref))] + list(shown)
    r, c = detail_crop(reference)
    fig, axes = plt.subplots(2, len(shown) + 1, figsize=(4 * (len(shown) + 1), 8.6))
    axes[0, 0].imshow(to_img(reference))
    axes[0, 0].add_patch(plt.Rectangle((c, r), CROP, CROP, fill=False, color="yellow", linewidth=1.5))
    axes[0, 0].set_title(ref_title, fontsize=9)
    axes[1, 0].imshow(to_img(reference[r:r + CROP, c:c + CROP]), interpolation="nearest")
    axes[1, 0].set_title("reference (crop)", fontsize=9)
    for k, (name, x, m) in enumerate(shown, start=1):
        sub = "" if m is None else (f"\nPSNR {m['psnr']:.2f} dB | LPIPS {m['lpips']:.3f} | LR loss {m['lr_loss']:.1f}\nSNR "
                                    + "/".join(f"{m['band_snr_db'][b]:.1f}" for b in BAND_NAMES))
        axes[0, k].imshow(to_img(x))
        axes[0, k].set_title(f"{name}{sub}", fontsize=8)
        axes[1, k].imshow(to_img(x[r:r + CROP, c:c + CROP]), interpolation="nearest")
        axes[1, k].set_title(f"{name} (crop)", fontsize=9)
    for ax in axes.flat:
        ax.axis("off")
    if suptitle:
        fig.suptitle(suptitle, fontsize=10)
    fig.subplots_adjust(left=0.01, right=0.99, bottom=0.01, top=0.86 if suptitle else 0.9, wspace=0.04, hspace=0.2)
    plt.show()


for i in DEV_IMAGES:
    im = images[i]
    shown = [("bicubic", im["outputs"]["bicubic"], im["metrics"]["bicubic"]),
             ("CSGM best", im["outputs"]["CSGM"], im["metrics"]["CSGM"])]
    shown += [(f"CSGM + θ (lr {lr:g})", theta_runs[(lr, i)]["x"], theta_runs[(lr, i)]["metrics"]) for lr in THETA_LRS]
    side_by_side(im["hr"], f"Original HR, development image {i}\n{class_name(im['label'])}", shown)

### Check 2 conclusions (4 development images, correct label; small sample)

1. **θ-only adaptation lowers HR PSNR on every image at both learning rates** (eager, step 2,000): θ lr 1e-6: 21.06 → **18.78 dB**; θ lr 1e-5: 21.06 → **18.18 dB** (0/4 improve). For reference, bicubic 24.94 and bicubic + BP 25.43.
2. **LPIPS gets worse too:** 0.406 → 0.454 (lr 1e-6) and → 0.481 (lr 1e-5). Image 8117 (0.395 → 0.525) and 403 (0.573 → 0.713) are hit hardest. So the "sharper" look of adapted outputs isn't perceptually closer to the truth by this measure.
3. **Measurement fit improves a lot,** which is what is optimized: LR loss 50.5 → 7.8 (lr 1e-6) and → 1.1 (lr 1e-5).
4. **Same trade-off as in Check 1 and notebook 02, now attributable to θ alone.** Measured bands improve (low 20.7 → 28–34 dB, mid 5.4 → 9.7–11.3 dB) but stay below bicubic + BP (56.8 / 14.4). Invented bands get worse (high −2.6 → −4.2 dB, very high −6.0 → about −10 dB).
5. **Over the steps:** PSNR peaks at step 50 (oracle-chosen: +0.04 dB at lr 1e-6, +0.28 dB at lr 1e-5) and then declines. Going from 1,000 steps (notebook 02) to 2,000 makes lr 1e-6 worse (19.9 → 18.8 dB).
6. **The numbers match notebook 02's ε-lr-0 runs** (lr 1e-5: 18.29 at 1,000 steps vs 18.18 here): ε contributes almost nothing; θ drives the adaptation.

**Visual impression** (subjective): CSGM + θ follows the original's large-scale structure and colours more closely and partly removes CSGM's block artifacts, but adds dense streaks and colour noise that aren't in the original.

---
# Check 3: Generator diagnostics (no optimization)

**Question.** Where could the blocky (CSGM) and "oil painting" (IADrift) textures come from? Three cheap diagnostics, no optimization:

**3a. Random samples at several CFG values.** Is the texture already in the generator's own samples, especially at the CFG 1.0 we use?
- For each of the 4 development classes, 2 random draws (seed 2031, folded with the class and draw index): ε ~ N(0, I) of shape (1, 256, 256, 3) and random `noise_labels` (32 values in 0–63).
- The **same ε and `noise_labels`** are generated at CFG 1.0, 1.5 and 2.5, so only CFG changes. Upstream evaluates CFG 1.0–3.5 for generation.
- Shown: full images and a 96×96 centre crop (nearest-neighbour enlarged).

**3b. ε statistics.** Does CSGM push ε off the Gaussian distribution the generator was trained on?
- The generator cuts ε into 256 patches of 16×16×3 = 768 values (its tokens) with one linear layer. For ε ~ N(0, I), every patch norm / √768 ≈ 1 with standard deviation ≈ 1/√(2·768) ≈ 0.026.
- Compared, per image: the initial ε (`initialize`, the CSGM start) and the cached CSGM-best ε. Statistics: RMS, mean, per-patch norm / √768 (mean, std, min, max), mean |patch mean|, and relative change ‖ε_CSGM − ε₀‖ / ‖ε₀‖.
- Radial power spectrum of ε (normalized so white Gaussian noise = 1 at every frequency), and per-patch norm maps next to the CSGM output.

**3c. Image power spectra and a patch-grid score.** Do outputs carry periodic structure at the generator's 16-pixel patch grid?
- Mean 2D power spectrum (Hann window, mean removed, log scale) for: the 4 HR images, the 8 random samples at CFG 1.0, CSGM best, IADrift (θ lr 1e-5, ε lr 1e-3, 1,000 steps, from Check 1) and CSGM + θ (lr 1e-5, 2,000 steps, from Check 2).
- **Patch-grid score:** a 16-pixel periodic pattern puts energy on the spectral lines at multiples of 1/16 cycles/pixel. Score = mean power on the lines k/16 (k = 1…7, both axes) divided by the mean power on the neighbouring lines (k/16 ± 2/256). About 1 = no grid; > 1 = grid structure.

In [ ]:
# Check 3a: random samples at several CFG values (no optimization).
SAMPLE_SEED = 2031
CFG_SCALES = [1.0, 1.5, 2.5]
SAMPLE_DRAWS = 2
SAMPLE_CROP = 96


def generate_cfg(eps, label, noise_labels, cfg):
    cond = model.apply({"params": params}, jnp.asarray(label, dtype=jnp.int32).reshape(1), cfg, noise_labels,
                       method=model.c_cfg_noise_to_cond)
    x = model.apply({"params": params}, eps, cond, True, method=model.generate_image)
    return np.asarray(x.astype(jnp.float32))[0]


samples = {}
for i in DEV_IMAGES:
    label = images[i]["label"]
    for d in range(SAMPLE_DRAWS):
        key = jax.random.fold_in(jax.random.fold_in(jax.random.PRNGKey(SAMPLE_SEED), label), d)
        k_eps, k_nl = jax.random.split(key)
        eps = jax.random.normal(k_eps, (1, 256, 256, 3), dtype=jnp.float32)
        nl = jax.random.randint(k_nl, (1, model.noise_coords), 0, model.noise_classes, dtype=jnp.int32)
        for cfg in CFG_SCALES:
            samples[(i, d, cfg)] = generate_cfg(eps, label, nl, cfg)

c0 = (256 - SAMPLE_CROP) // 2
for i in DEV_IMAGES:
    n = len(CFG_SCALES)
    fig, axes = plt.subplots(SAMPLE_DRAWS, 2 * n, figsize=(3.1 * 2 * n, 3.4 * SAMPLE_DRAWS))
    for d in range(SAMPLE_DRAWS):
        for k, cfg in enumerate(CFG_SCALES):
            x = samples[(i, d, cfg)]
            axes[d, k].imshow(to_img(x))
            axes[d, k].add_patch(plt.Rectangle((c0, c0), SAMPLE_CROP, SAMPLE_CROP, fill=False, color="yellow", linewidth=1))
            axes[d, k].set_title(f"draw {d}, CFG {cfg:g}", fontsize=9)
            axes[d, n + k].imshow(to_img(x[c0:c0 + SAMPLE_CROP, c0:c0 + SAMPLE_CROP]), interpolation="nearest")
            axes[d, n + k].set_title(f"draw {d}, CFG {cfg:g} (crop)", fontsize=9)
    for ax in axes.flat:
        ax.axis("off")
    fig.suptitle(f"Random samples, class {images[i]['label']} ({class_name(images[i]['label'])}): no optimization", fontsize=10)
    fig.subplots_adjust(left=0.01, right=0.99, bottom=0.01, top=0.88, wspace=0.04, hspace=0.15)
    plt.show()

### Check 3b: ε statistics



In [ ]:
# Check 3b: ε statistics, initial vs CSGM best.
GAUSS_PATCH_STD = 1 / np.sqrt(2 * 768)


def patch_view(e):
    # [.., 256, 256, 3] -> [256 patches, 768], in the generator's patch order (16x16 grid of 16x16x3).
    e = np.asarray(e, np.float64).reshape(16, 16, 16, 16, 3)
    return e.transpose(0, 2, 1, 3, 4).reshape(256, 768)


def eps_stats(e, e0=None):
    p = patch_view(e)
    pn = np.linalg.norm(p, axis=1) / np.sqrt(768)
    s = {"rms": float(np.sqrt(np.mean(p ** 2))), "mean": float(p.mean()),
         "patch_norm_mean": float(pn.mean()), "patch_norm_std": float(pn.std()),
         "patch_norm_min": float(pn.min()), "patch_norm_max": float(pn.max()),
         "patch_mean_abs": float(np.abs(p.mean(1)).mean())}
    if e0 is not None:
        s["rel_change"] = float(np.linalg.norm(p - patch_view(e0)) / np.linalg.norm(patch_view(e0)))
    return s, pn.reshape(16, 16)


def eps_spectrum(e, num_bins=32):
    # Radial power spectrum of ε, normalized so that white N(0, 1) noise is 1 at every frequency.
    e = np.asarray(e, np.float64).reshape(256, 256, 3)
    power = np.sum(np.abs(np.fft.fft2(e, axes=(0, 1))) ** 2, axis=-1) / (256 * 256 * 3)
    edges = np.linspace(0.0, 0.5, num_bins + 1)
    return (edges[:-1] + edges[1:]) / 2, np.array([power[(RADIUS >= lo) & (RADIUS < hi)].mean()
                                                    for lo, hi in zip(edges[:-1], edges[1:])])


eps_info = {}
for i in DEV_IMAGES:
    r = images[i]["csgm_restart"]
    with np.load(CSGM_CACHE / f"img{i:05d}_oracle_r{r}.npz") as data:
        eps_csgm = data["eps"]
    eps0 = np.asarray(initialize(i, r)[0])
    eps_info[i] = {"init": (eps0, *eps_stats(eps0)), "CSGM": (eps_csgm, *eps_stats(eps_csgm, eps0))}

KEYS = ["rms", "mean", "patch_norm_mean", "patch_norm_std", "patch_norm_min", "patch_norm_max", "patch_mean_abs", "rel_change"]
lines = ["| Image | ε | " + " | ".join(KEYS) + " |", "|---|---|" + "---:|" * len(KEYS)]
for i in DEV_IMAGES:
    for state in ("init", "CSGM"):
        s = eps_info[i][state][1]
        lines.append(f"| {i} | {state} | " + " | ".join(f"{s[k]:.3f}" if k in s else "—" for k in KEYS) + " |")
display(Markdown(f"**ε statistics** (Gaussian reference: patch_norm_mean ≈ 1, patch_norm_std ≈ {GAUSS_PATCH_STD:.3f})\n\n"
                 + "\n".join(lines)))

fig, ax = plt.subplots(figsize=(7, 4))
for state, color in (("init", "k"), ("CSGM", "C3")):
    curves = [eps_spectrum(eps_info[i][state][0]) for i in DEV_IMAGES]
    ax.plot(curves[0][0], np.mean([c[1] for c in curves], axis=0), color=color, label=f"ε {state}")
ax.axvline(LR_NYQUIST, color="gray", linestyle="--", linewidth=1, label="LR Nyquist (0.125)")
ax.set(yscale="log", xlabel="radial frequency (cycles per pixel)", ylabel="power (white noise = 1)",
       title="Radial power spectrum of ε, mean over 4 development images")
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
plt.show()

vmax = max(np.abs(eps_info[i]["CSGM"][2] - 1).max() for i in DEV_IMAGES)
fig, axes = plt.subplots(2, len(DEV_IMAGES), figsize=(4 * len(DEV_IMAGES), 8.3))
for k, i in enumerate(DEV_IMAGES):
    axes[0, k].imshow(to_img(images[i]["outputs"]["CSGM"]))
    axes[0, k].set_title(f"CSGM best, image {i}", fontsize=9)
    h = axes[1, k].imshow(eps_info[i]["CSGM"][2], cmap="coolwarm", vmin=1 - vmax, vmax=1 + vmax)
    axes[1, k].set_title("ε_CSGM per-patch norm / √768", fontsize=9)
for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])
fig.colorbar(h, ax=axes[1, :].tolist(), shrink=0.8)
plt.show()

### Check 3c: power spectra and patch-grid score



In [ ]:
# Check 3c: image power spectra and the patch-grid score.
def power_2d(x):
    x = np.asarray(x, np.float64)
    x = x - x.mean(axis=(0, 1))
    return np.sum(np.abs(np.fft.fft2(x * WINDOW, axes=(0, 1))) ** 2, axis=-1)


def grid_score(x):
    # Mean power on spectral lines at k/16 cycles/pixel (k = 1..7, both axes) / neighbouring lines (±2 bins).
    p = power_2d(x)
    on = np.mean([p[:, 16 * k].mean() + p[16 * k, :].mean() for k in range(1, 8)])
    off = np.mean([p[:, 16 * k + s].mean() + p[16 * k + s, :].mean() for k in range(1, 8) for s in (-2, 2)])
    return float(on / off)


GROUPS = {
    "HR (truth)": [images[i]["hr"] for i in DEV_IMAGES],
    "random samples, CFG 1.0": [samples[(i, d, 1.0)] for i in DEV_IMAGES for d in range(SAMPLE_DRAWS)],
    "CSGM best": [images[i]["outputs"]["CSGM"] for i in DEV_IMAGES],
    "IADrift (1,000 steps)": [images[i]["iadrift"][MAIN_CFG]["IADrift"] for i in DEV_IMAGES],
    "CSGM + θ (lr 1e-5, 2,000 steps)": [theta_runs[(1e-5, i)]["x"] for i in DEV_IMAGES],
    "bicubic": [images[i]["outputs"]["bicubic"] for i in DEV_IMAGES],
}
lines = ["| Images | n | patch-grid score (mean) | per image |", "|---|---:|---:|---|"]
for name, xs in GROUPS.items():
    scores = [grid_score(x) for x in xs]
    lines.append(f"| {ARM9[name]} | {len(xs)} | {np.mean(scores):.2f} | " + ", ".join(f"{s:.2f}" for s in scores) + " |")
display(Markdown("**Patch-grid score** (about 1 = no 16-pixel periodic structure)\n\n" + "\n".join(lines)))

shown = [g for g in GROUPS if g != "bicubic"]
fig, axes = plt.subplots(1, len(shown), figsize=(4 * len(shown), 4.2))
f = np.fft.fftshift(np.fft.fftfreq(256))
for ax, name in zip(axes, shown):
    p = np.fft.fftshift(np.mean([power_2d(x) for x in GROUPS[name]], axis=0))
    ax.imshow(np.log10(p + 1e-12), cmap="magma", extent=(f[0], f[-1], f[-1], f[0]), vmin=-1, vmax=6)
    ax.set_title(name, fontsize=9)
    ax.set_xlabel("cycles/pixel")
fig.suptitle("Mean 2D power spectrum (log10); bright lines at multiples of 1/16 = patch grid", fontsize=10)
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4.5))
edges = np.linspace(0, 0.5, 65)
for k, name in enumerate(GROUPS):
    p = np.mean([power_2d(x) for x in GROUPS[name]], axis=0)
    radial = [p[(RADIUS >= lo) & (RADIUS < hi)].mean() for lo, hi in zip(edges[:-1], edges[1:])]
    ax.plot((edges[:-1] + edges[1:]) / 2, radial, color="k" if k == 0 else f"C{k}", label=name,
            linewidth=2 if k == 0 else 1.2)
ax.axvline(LR_NYQUIST, color="gray", linestyle="--", linewidth=1)
ax.set(yscale="log", xlabel="radial frequency (cycles per pixel)", ylabel="mean power",
       title="Radial power spectrum of the images (not the error)")
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
plt.show()

### Check 3 conclusions (no optimization; observations unless marked)

**3a. The generator's own samples already contain the artifacts.**
- At CFG 1.0, 1.5 and 2.5 alike, random samples show visible **16×16 block structure** and occasional **"confetti" patches** (a single 16×16 patch of saturated colour noise), plus painterly texture inside patches. Higher CFG gives more contrast and saturation, not cleaner patches.
- So part of the texture in CSGM/IADrift outputs is **intrinsic to `pixel_B_sota`** and not created by our optimization. CFG 1.0 is not the cause.

**3b. CSGM moves ε far off the Gaussian shell.**
- Per-patch norm / √768: Gaussian ≈ 1.00 ± 0.026. After CSGM: mean **1.37–1.82**, std **0.23–0.57** (10–20× the Gaussian spread), max up to **4.8**. RMS rises from 1.0 to 1.4–1.9, and ε changes by 97–164 % of its starting norm.
- The spectrum of ε_CSGM is about **3× white noise at every frequency**, with an excess at the lowest frequencies and bumps at multiples of about 1/16 cycles/pixel. The change is broadband, not just in the bands the measurement controls, and it carries patch-grid structure.
- The patches with the largest norms don't line up obviously with the worst CSGM blocks (per-patch maps vs outputs); the inflation is spread over the whole image.

**3c. Patch-grid structure is measurable, and it's already in the samples.**
- Patch-grid score (≈ 1 = none): HR 0.97, bicubic 0.94; **random samples 1.54**, CSGM best 1.67, IADrift 1.74, CSGM + θ 1.74. The 2D spectra show a lattice of peaks at multiples of 1/16 in all generator outputs, and none in HR.
- Optimization increases the grid score only modestly over plain samples (1.54 → 1.67–1.74).

**Interpretation (hypotheses):** the 16-pixel patch grid comes from the architecture (linear patch embedding and unpatchify, 16×16 patches) and is present before any inversion. CSGM's off-distribution ε may make it somewhat worse (Check 4 tests whether forcing ε back helps).

---
# Check 4: ε sphere projection for CSGM

**Question.** Check 3b tests whether CSGM pushes ε off the Gaussian distribution. If it does, does **keeping ε on the Gaussian shell** during CSGM give better reconstructions? The idea follows PULSE, which projects StyleGAN latents onto the sphere of radius √d after every step.

**Baseline = the saved no-projection runs** (`csgm_dev_5000steps_lr0.02/`, as used in Checks 1–3): frozen generator, Adam lr 0.02 on ε, 5,000 steps, 2 restarts, keep the lowest final LR loss.

**Reproduction test first.** The projected runs use a new jitted step: `step = Adam update, then project`. With the projection set to the identity, this step must reproduce a saved run **bit for bit**, otherwise differences could come from a different compiled function (Check 2 setup note: CSGM trajectories change under a different numerical path). Test: development image 8117, restart 1, identity projection, compared with the saved run (final LR loss, ε, output, loss trace). The notebook stops if it isn't exact.

**Projection variants** (everything else identical to the baseline: the same `initialize(image, restart)` start, Adam lr 0.02, 5,000 steps, restarts 0 and 1, lowest final LR loss kept, never HR):
- **global sphere:** after every Adam step, ε ← ε · √d / ‖ε‖ with d = 196,608 (the whole ε).
- **per-patch sphere:** after every Adam step, each 16×16×3 patch of ε (aligned with the generator's patch grid, 768 values) is rescaled to norm √768. This is stronger: every token gets the norm a Gaussian patch would have.
- The starting ε is projected too (it's already almost on the sphere).
- Adam's moment estimates are left as they are; only ε is projected.

**Metrics.** As in Check 1 (eager output): PSNR (unclipped, clipped), LPIPS, LR loss, band SNR; also "+ BP" versions, and the ε statistics of Check 3b. Loss traces are the compiled LR loss every 50 steps.

**Cost.** 16 runs × about 75 s (first execution only). Cache: `~/work/datasets/ImageNetV2/csgm_eps_projection_dev_v1/`.

In [ ]:
# Check 4 (and 5): a generic CSGM runner with an optional ε projection after each Adam step.
PROJ_DIR = DATA_DIR / "csgm_eps_projection_dev_v1"
PROJ_DIR.mkdir(exist_ok=True)
CSGM_STEPS = FROZEN_CONFIG["num_steps"]
RESTARTS = list(range(FROZEN_CONFIG["num_restarts"]))


def project_none(eps):
    return eps


def project_global(eps):
    return eps * jnp.sqrt(float(eps.size)) / jnp.linalg.norm(eps)


def project_patch(eps):
    b = eps.reshape(1, 16, 16, 16, 16, 3)  # (batch, grid row, pixel row, grid col, pixel col, channel)
    return (b * jnp.sqrt(768.0) / jnp.sqrt(jnp.sum(b ** 2, axis=(2, 4, 5), keepdims=True))).reshape(eps.shape)


_compiled = {}


def csgm_step(loss_fn, project):
    if (loss_fn, project) not in _compiled:
        opt = optax.adam(FROZEN_CONFIG["learning_rate"])

        @jax.jit
        def step(eps, state, target, label, noise_labels):
            loss, g = jax.value_and_grad(loss_fn)(eps, target, label, noise_labels)
            updates, state = opt.update(g, state, eps)
            return project(optax.apply_updates(eps, updates)), state, loss

        _compiled[(loss_fn, project)] = (opt, step)
    return _compiled[(loss_fn, project)]


def run_csgm(path, loss_fn, project, target, label, eps0, noise_labels):
    """One CSGM run (same loop as run_sr4_csgm_test.reconstruct); cached as .npz. Returns (metrics, arrays)."""
    if path.exists():
        with np.load(path) as data:
            return json.loads(str(data["metrics_json"])), {k: data[k] for k in data.files if k != "metrics_json"}
    opt, step = csgm_step(loss_fn, project)
    eps = project(eps0)
    state = opt.init(eps)
    trace_steps, trace_loss = [], []
    start = perf_counter()
    for s in range(1, CSGM_STEPS + 1):
        eps, state, loss_before = step(eps, state, target, label, noise_labels)
        if s % TRACE_EVERY == 0:
            trace_steps.append(s - 1)  # loss_before is evaluated before update s
            trace_loss.append(float(loss_before))
    eps.block_until_ready()
    metrics = {"final_loss": float(loss_fn(eps, target, label, noise_labels)), "seconds": perf_counter() - start}
    arrays = {"eps": np.asarray(eps, dtype=np.float32), "noise_labels": np.asarray(noise_labels),
              "x_hat": np.asarray(jax.device_get(generate(eps, label, noise_labels)), dtype=np.float32)[0],
              "trace_steps": np.asarray(trace_steps, dtype=np.int32), "trace_loss": np.asarray(trace_loss, dtype=np.float32)}
    buffer = io.BytesIO()
    np.savez_compressed(buffer, metrics_json=json.dumps(metrics), **arrays)
    atomic_write_bytes(path, buffer.getvalue())
    return metrics, arrays


def best_restart(runs):
    # runs: {restart: (metrics, arrays)} -> (restart, metrics, arrays) with the lowest final loss (never HR).
    r = min(runs, key=lambda r: (runs[r][0]["final_loss"], r))
    return r, *runs[r]


# Reproduction test: identity projection must reproduce the saved no-projection run exactly.
i, r = 8117, 1
eps0, nl0 = initialize(i, r)
m_rep, a_rep = run_csgm(PROJ_DIR / f"repro_img{i:05d}_none_r{r}.npz", measurement_loss, project_none,
                        images[i]["y"], images[i]["label"], eps0, nl0)
with np.load(CSGM_CACHE / f"img{i:05d}_oracle_r{r}.npz") as data:
    saved_m, saved = json.loads(str(data["metrics_json"])), {k: data[k] for k in ("eps", "x_hat", "trace_loss")}
exact = (m_rep["final_loss"] == saved_m["final_loss"] and np.array_equal(a_rep["eps"], saved["eps"])
         and np.array_equal(a_rep["x_hat"], saved["x_hat"]) and np.array_equal(a_rep["trace_loss"], saved["trace_loss"]))
print(f"Reproduction test (image {i}, restart {r}): final LR loss {m_rep['final_loss']:.4f} vs saved "
      f"{saved_m['final_loss']:.4f}; ε, output and loss trace identical: {exact}")
assert exact, "The new CSGM step does not reproduce the saved run; projection comparisons would be confounded."

### Check 4 results



In [ ]:
# Check 4 runs: global and per-patch sphere projection, 4 images x 2 restarts each.
PROJ_VARIANTS = {"global sphere": project_global, "per-patch sphere": project_patch}

proj = {}
for name, project in PROJ_VARIANTS.items():
    slug = name.replace(" ", "_").replace("-", "")
    for i in DEV_IMAGES:
        restarts = {r: run_csgm(PROJ_DIR / f"img{i:05d}_{slug}_r{r}.npz", measurement_loss, project,
                                images[i]["y"], images[i]["label"], *initialize(i, r)) for r in RESTARTS}
        r, m, a = best_restart(restarts)
        proj[(name, i)] = {"restart": r, "final_losses": [restarts[k][0]["final_loss"] for k in RESTARTS],
                           "eps": a["eps"], "x": a["x_hat"], "trace": (a["trace_steps"], a["trace_loss"])}
        print(f"{name}, image {i}: final LR loss r0 {restarts[0][0]['final_loss']:.1f}, r1 {restarts[1][0]['final_loss']:.1f} "
              f"-> keep r{r} ({sum(restarts[k][0]['seconds'] for k in RESTARTS):.0f} s total)")

# The saved no-projection baseline, in the same structure.
for i in DEV_IMAGES:
    r = images[i]["csgm_restart"]
    with np.load(CSGM_CACHE / f"img{i:05d}_oracle_r{r}.npz") as data:
        proj[("none (saved)", i)] = {"restart": r, "eps": data["eps"], "x": data["x_hat"],
                                     "trace": (data["trace_steps"], data["trace_loss"])}

VARIANTS4 = ["none (saved)"] + list(PROJ_VARIANTS)
for (name, i), p in proj.items():
    p["metrics"] = evaluate(p["x"], images[i]["hr"], images[i]["y"])
    p["x_bp"] = backproject(p["x"], images[i]["y"])
    p["metrics_bp"] = evaluate(p["x_bp"], images[i]["hr"], images[i]["y"])
    p["eps_stats"] = eps_stats(p["eps"], np.asarray(initialize(i, p["restart"])[0]))[0]


def summary_table(rows, ref_rows=None, ref_name="ref"):
    # rows: [(label, [metric dict per image])]. Optional paired Δ columns against ref_rows.
    head = "| Method | PSNR | clipped | LPIPS ↓ | LR loss | " + " | ".join(f"SNR {b}" for b in BAND_NAMES) + " |"
    sep = "|---|---:|---:|---:|---:|" + "---:|" * len(BAND_NAMES)
    if ref_rows is not None:
        head += f" Δ PSNR vs {ref_name} | PSNR better | Δ LPIPS vs {ref_name} | LPIPS better |"
        sep += "---:|---:|---:|---:|"
    lines = [head, sep]
    for label, ms in rows:
        p, l = np.array([m["psnr"] for m in ms]), np.array([m["lpips"] for m in ms])
        line = (f"| {label} | {p.mean():.2f} | {np.mean([m['psnr_clipped'] for m in ms]):.2f} | {l.mean():.3f} | "
                f"{np.mean([m['lr_loss'] for m in ms]):.2f} | "
                + " | ".join(f"{np.mean([m['band_snr_db'][b] for m in ms]):.2f}" for b in BAND_NAMES) + " |")
        if ref_rows is not None:
            rp, rl = np.array([m["psnr"] for m in ref_rows]), np.array([m["lpips"] for m in ref_rows])
            line += (f" {np.mean(p - rp):+.2f} | {int(np.sum(p > rp + 1e-9))}/{len(p)} | "
                     f"{np.mean(l - rl):+.3f} | {int(np.sum(l < rl - 1e-9))}/{len(l)} |")
        lines.append(line)
    return "\n".join(lines)


ref = [proj[("none (saved)", i)]["metrics"] for i in DEV_IMAGES]
ref_bp = [proj[("none (saved)", i)]["metrics_bp"] for i in DEV_IMAGES]
rows = [("bicubic", [images[i]["metrics"]["bicubic"] for i in DEV_IMAGES]),
        ("bicubic + BP", [images[i]["metrics"]["bicubic + BP"] for i in DEV_IMAGES])]
rows += [(f"CSGM, {v}", [proj[(v, i)]["metrics"] for i in DEV_IMAGES]) for v in VARIANTS4]
display(Markdown("**CSGM output, mean over 4 development images** (Δ columns: paired vs CSGM without projection)\n\n"
                 + summary_table(rows, ref, "none")))
rows = [(f"CSGM + BP, {v}", [proj[(v, i)]["metrics_bp"] for i in DEV_IMAGES]) for v in VARIANTS4]
display(Markdown("**After back-projection** (Δ columns: paired vs CSGM + BP without projection)\n\n"
                 + summary_table(rows, ref_bp, "none + BP")))

lines = ["| Variant | " + " | ".join(f"{i} ({class_name(images[i]['label'])})" for i in DEV_IMAGES) + " |",
         "|---|" + "---:|" * len(DEV_IMAGES)]
for v in VARIANTS4:
    lines.append(f"| {v} | " + " | ".join(
        f"{proj[(v, i)]['metrics']['psnr']:.2f} / {proj[(v, i)]['metrics']['lpips']:.3f} / {proj[(v, i)]['metrics']['lr_loss']:.1f}"
        for i in DEV_IMAGES) + " |")
display(Markdown("**Per image: PSNR dB / LPIPS / LR loss (CSGM output, best restart)**\n\n" + "\n".join(lines)))

S_KEYS = ["rms", "patch_norm_mean", "patch_norm_std", "patch_norm_min", "patch_norm_max", "rel_change"]
lines = ["| Variant | " + " | ".join(S_KEYS) + " |", "|---|" + "---:|" * len(S_KEYS)]
for v in VARIANTS4:
    lines.append(f"| {v} | " + " | ".join(f"{np.mean([proj[(v, i)]['eps_stats'][k] for i in DEV_IMAGES]):.3f}"
                                          for k in S_KEYS) + " |")
display(Markdown(f"**ε statistics of the kept restart, mean over images** (Gaussian: patch_norm_std ≈ {GAUSS_PATCH_STD:.3f})\n\n"
                 + "\n".join(lines)))

fig, ax = plt.subplots(figsize=(8, 4.5))
for k, v in enumerate(VARIANTS4):
    steps = proj[(v, DEV_IMAGES[0])]["trace"][0]
    ax.plot(steps, np.mean([proj[(v, i)]["trace"][1] for i in DEV_IMAGES], axis=0), color=f"C{k}", label=v)
ax.set(yscale="log", xlabel="CSGM step", ylabel="LR loss (compiled)", title="CSGM loss, kept restart, mean over 4 images")
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
plt.show()

for i in DEV_IMAGES:
    shown = [("bicubic", images[i]["outputs"]["bicubic"], images[i]["metrics"]["bicubic"])]
    shown += [(f"CSGM, {v}", proj[(v, i)]["x"], proj[(v, i)]["metrics"]) for v in VARIANTS4]
    side_by_side(images[i]["hr"], f"Original HR, development image {i}\n{class_name(images[i]['label'])}", shown)

summary = {v: {str(i): {"restart": proj[(v, i)]["restart"], "metrics": proj[(v, i)]["metrics"],
                        "metrics_bp": proj[(v, i)]["metrics_bp"], "eps_stats": proj[(v, i)]["eps_stats"]}
               for i in DEV_IMAGES} for v in VARIANTS4}
atomic_write_bytes(PROJ_DIR / "summary.json", json.dumps(summary, indent=1).encode())

### Check 4 conclusions (4 development images, correct label; small sample)

1. **The reproduction test passed:** with the identity projection, the new step reproduced the saved run bit for bit (final LR loss 70.9893, identical ε, output and loss trace). Differences below are due to the projection.
2. **Sphere projection does not help CSGM on any measure that matters:**
   - PSNR: none 21.06, global sphere 20.74, per-patch sphere 20.73 dB (−0.32; better on 1/4 and 0/4 images).
   - **LPIPS gets worse:** 0.406 → 0.480 (global) and 0.477 (per-patch), worse on **4/4 images** for both.
   - Measurement fit gets worse: LR loss 50.5 → 67.7 (global) and 85.1 (per-patch); low and mid band SNR drop accordingly.
   - After back-projection the per-patch variant gains +0.13 dB PSNR (3/4 images), but LPIPS is still worse (+0.045; 1/4 images better).
3. **The projection does what it should:** per-patch norms are exactly 1 (global variant: still spread, std 0.32). So the constraint is enforced, and it costs fit without buying quality.
4. **Visually** (image 403 especially), the per-patch variant can lock into wrong content: a large textured bird region where the target has a smooth background, plus flat blocks. The global variant looks much like no projection.

**Interpretation (hypothesis):** inflated ε norms are not the main cause of the artifacts. Check 3a shows them in plain samples, and Check 5 (R1) shows that even fitting HR directly needs off-shell ε (patch-norm std 0.21–0.55). The generator seems to need off-distribution inputs to approach real images at all, so forcing ε onto the shell mainly removes the freedom CSGM uses to fit y.

**Decision input:** don't adopt the sphere projection as a CSGM or IADrift variant. It is a negative result.

---
# Check 5: Representation error (diagnostics, not methods)

**Question.** Is CSGM's poor quality a limit of the **generator** (it can't express these images), or of the **inversion** (SR ×4 is ill-posed and the optimizer finds bad inputs)? These two diagnostics separate them, following the representation-error argument of the IAGAN paper.

**R1: CSGM fitted to HR directly (A = identity).**
- Loss ‖HR − f(ε)‖² instead of ‖y − A f(ε)‖². **This uses HR on purpose:** it measures the best the frozen generator can do for each image, the representation error. It's a ceiling, not a method.
- Everything else as the CSGM baseline: `initialize(image, restart)`, Adam lr 0.02 (Adam is insensitive to the 16× larger loss scale), 5,000 steps, restarts 0 and 1, keep the lower final loss, no projection.
- Compared with bicubic and CSGM SR on the same images.

**R3: in-range SR ×4.**
- For each development image's class, draw a target from the generator itself: ε* ~ N(0, I), random `noise_labels*` (seed 2032, folded with the image index), x* = f(ε*), y* = A x*.
- The target is in the generator's range by construction, so representation error is zero and only ill-posedness plus optimization remain. The perfect solution (ε*) has LR loss 0.
- CSGM SR from y*, exactly as the baseline (`initialize(image, restart)`, Adam lr 0.02, 5,000 steps, 2 restarts, lowest final LR loss), in two conditions:
  - **noise labels known:** CSGM uses the true `noise_labels*` (only ε has to be found);
  - **noise labels random:** CSGM uses the random `noise_labels` from `initialize`, as in the real setting. `noise_labels` are discrete and never optimized.
- Scored against x* (not HR): PSNR, LPIPS, band SNR, LR loss, and ε error ‖ε − ε*‖ / ‖ε*‖. References: bicubic(y*) and bicubic + BP(y*).

**Reading the result.**
- If R1 gives a high PSNR and clean images while CSGM SR doesn't, the generator *can* express the image: the problem is the inversion (ill-posedness, off-distribution ε).
- If R1 itself is blocky or painterly with low PSNR, the texture is a representation limit of this generator.
- If CSGM can't recover in-range targets in R3 either (high LR loss, artifacts, large ε error), the inversion problem is in the optimization itself, independent of the data.

**Cost.** 8 runs (R1) + 16 runs (R3) × about 75 s (first execution only). Cache: `~/work/datasets/ImageNetV2/representation_error_dev_v1/`.

In [ ]:
# Check 5, R1: CSGM fitted to HR (A = identity). Diagnostic only: uses HR as the target.
REP_DIR = DATA_DIR / "representation_error_dev_v1"
REP_DIR.mkdir(exist_ok=True)


def hr_loss(eps, target, label, noise_labels):
    return jnp.sum((generate(eps, label, noise_labels).astype(jnp.float32) - target) ** 2)


r1 = {}
for i in DEV_IMAGES:
    im = images[i]
    target = jnp.asarray(im["hr"], jnp.float32)[None]
    restarts = {r: run_csgm(REP_DIR / f"R1_img{i:05d}_r{r}.npz", hr_loss, project_none, target, im["label"],
                            *initialize(i, r)) for r in RESTARTS}
    r, m, a = best_restart(restarts)
    r1[i] = {"restart": r, "x": a["x_hat"], "metrics": evaluate(a["x_hat"], im["hr"], im["y"]),
             "eps_stats": eps_stats(a["eps"], np.asarray(initialize(i, r)[0]))[0]}
    print(f"R1 image {i}: final HR loss r0 {restarts[0][0]['final_loss']:.1f}, r1 {restarts[1][0]['final_loss']:.1f} -> keep r{r}")

rows = [("bicubic (SR)", [images[i]["metrics"]["bicubic"] for i in DEV_IMAGES]),
        ("bicubic + BP (SR)", [images[i]["metrics"]["bicubic + BP"] for i in DEV_IMAGES]),
        ("CSGM SR (saved)", [images[i]["metrics"]["CSGM"] for i in DEV_IMAGES]),
        ("**R1: CSGM fitted to HR** (ceiling)", [r1[i]["metrics"] for i in DEV_IMAGES])]
display(Markdown("**R1: representation ceiling of the frozen generator, mean over 4 development images**\n\n"
                 + summary_table(rows)))
lines = ["| | " + " | ".join(f"{i} ({class_name(images[i]['label'])})" for i in DEV_IMAGES) + " |",
         "|---|" + "---:|" * len(DEV_IMAGES)]
for label, get in (("CSGM SR (saved)", lambda i: images[i]["metrics"]["CSGM"]), ("R1 fit to HR", lambda i: r1[i]["metrics"])):
    lines.append(f"| {label} | " + " | ".join(f"{get(i)['psnr']:.2f} / {get(i)['lpips']:.3f}" for i in DEV_IMAGES) + " |")
lines.append("| R1 ε patch_norm_std | " + " | ".join(f"{r1[i]['eps_stats']['patch_norm_std']:.3f}" for i in DEV_IMAGES) + " |")
display(Markdown("**Per image: PSNR dB / LPIPS**\n\n" + "\n".join(lines)))

for i in DEV_IMAGES:
    shown = [("bicubic", images[i]["outputs"]["bicubic"], images[i]["metrics"]["bicubic"]),
             ("CSGM SR (saved)", images[i]["outputs"]["CSGM"], images[i]["metrics"]["CSGM"]),
             ("R1: CSGM fitted to HR", r1[i]["x"], r1[i]["metrics"])]
    side_by_side(images[i]["hr"], f"Original HR, development image {i}\n{class_name(images[i]['label'])}", shown)

### Check 5 R3: in-range SR



In [ ]:
# Check 5, R3: in-range SR x4 (targets drawn from the generator itself).
INRANGE_SEED = 2032
R3_CONDITIONS = ("noise labels known", "noise labels random")

r3 = {}
for i in DEV_IMAGES:
    label = images[i]["label"]
    k_eps, k_nl = jax.random.split(jax.random.fold_in(jax.random.PRNGKey(INRANGE_SEED), i))
    eps_star = jax.random.normal(k_eps, (1, 256, 256, 3), dtype=jnp.float32)
    nl_star = jax.random.randint(k_nl, (1, model.noise_coords), 0, model.noise_classes, dtype=jnp.int32)
    x_star = np.asarray(jax.device_get(generate(eps_star, label, nl_star)), dtype=np.float32)[0]
    y_star = A_sr4(jnp.asarray(x_star)[None])
    bic = bicubic_up(np.asarray(y_star[0]))
    entry = {"x_star": x_star, "y_star": y_star, "label": label,
             "loss_at_truth": float(measurement_loss(eps_star, y_star, label, nl_star)),
             "outputs": {"bicubic": bic, "bicubic + BP": backproject(bic, y_star)}}
    for cond in R3_CONDITIONS:
        slug = cond.replace(" ", "_")
        restarts = {}
        for r in RESTARTS:
            eps0, nl0 = initialize(i, r)
            restarts[r] = run_csgm(REP_DIR / f"R3_img{i:05d}_{slug}_r{r}.npz", measurement_loss, project_none, y_star,
                                   label, eps0, nl_star if cond == "noise labels known" else nl0)
        r, m, a = best_restart(restarts)
        entry["outputs"][f"CSGM, {cond}"] = a["x_hat"]
        entry[f"eps_err, {cond}"] = float(np.linalg.norm(a["eps"] - np.asarray(eps_star)) / np.linalg.norm(np.asarray(eps_star)))
        print(f"R3 image {i}, {cond}: final LR loss r0 {restarts[0][0]['final_loss']:.2f}, "
              f"r1 {restarts[1][0]['final_loss']:.2f} -> keep r{r}")
    entry["metrics"] = {n: evaluate(x, x_star, y_star) for n, x in entry["outputs"].items()}
    r3[i] = entry
print("LR loss at the true ε* (eager):", [round(r3[i]["loss_at_truth"], 4) for i in DEV_IMAGES])

R3_ROWS = ["bicubic", "bicubic + BP"] + [f"CSGM, {c}" for c in R3_CONDITIONS]
rows = [(n, [r3[i]["metrics"][n] for i in DEV_IMAGES]) for n in R3_ROWS]
display(Markdown("**R3: in-range SR ×4, scored against the generated target x*** (mean over 4 targets)\n\n" + summary_table(rows)))
lines = ["| | " + " | ".join(f"target {i} (class {class_name(r3[i]['label'])})" for i in DEV_IMAGES) + " |",
         "|---|" + "---:|" * len(DEV_IMAGES)]
for n in R3_ROWS:
    lines.append(f"| {n} | " + " | ".join(f"{r3[i]['metrics'][n]['psnr']:.2f} / {r3[i]['metrics'][n]['lpips']:.3f} / "
                                        f"{r3[i]['metrics'][n]['lr_loss']:.1f}" for i in DEV_IMAGES) + " |")
for c in R3_CONDITIONS:
    lines.append(f"| ε error ‖ε − ε*‖/‖ε*‖, {c} | " + " | ".join(f"{r3[i][f'eps_err, {c}']:.3f}" for i in DEV_IMAGES) + " |")
display(Markdown("**Per target: PSNR dB / LPIPS / LR loss**\n\n" + "\n".join(lines)))

for i in DEV_IMAGES:
    shown = [(n, r3[i]["outputs"][n], r3[i]["metrics"][n]) for n in R3_ROWS if n != "bicubic + BP"]
    side_by_side(r3[i]["x_star"], f"In-range target x* = f(ε*)\nclass {class_name(r3[i]['label'])}", shown)

summary = {"R1": {str(i): {"restart": r1[i]["restart"], "metrics": r1[i]["metrics"], "eps_stats": r1[i]["eps_stats"]}
                  for i in DEV_IMAGES},
           "R3": {str(i): {"metrics": r3[i]["metrics"], **{k: v for k, v in r3[i].items() if k.startswith("eps_err")},
                           "loss_at_truth": r3[i]["loss_at_truth"]} for i in DEV_IMAGES}}
atomic_write_bytes(REP_DIR / "summary.json", json.dumps(summary, indent=1).encode())

### Check 5 conclusions (4 development images / 4 generated targets; diagnostics)

**R1: the frozen generator's representation error on real images is large.**
- Fitting HR **directly** (A = identity, 5,000 steps, best of 2 restarts) reaches only **24.44 dB**, below bicubic SR (24.94) and bicubic + BP (25.43), with LPIPS 0.332.
- Even the low band is poor (19.7 dB, vs 51.9 for CSGM + BP in Check 1), and the fit is **visibly blocky at the 16-pixel grid** (image 8117 crop).
- Caveat: this is the best this optimizer finds in 5,000 steps, an upper bound on the representation error rather than an exact value. ε again ends up off-shell (patch-norm std 0.21–0.55).
- **This is the IAGAN motivation in its strongest form:** the prior alone can't express these ImageNet images, even with full information.

**R3: in-range targets are recovered well, but only when the noise labels are right.**
- **Noise labels known:** CSGM reaches **25.93 dB and LPIPS 0.116** against the generated target, far better than bicubic (22.62 / 0.455) and bicubic + BP (23.03 / 0.402). Visually it's nearly indistinguishable from x* (crane target). The prior works when the target is in range.
- **Noise labels random** (the real setting): **21.89 dB, LPIPS 0.331**, LR loss 40 vs 19. Visually it **hallucinates the wrong content** (for example bird-feather texture filling the crane target's fence background).
- So the 32 discrete `noise_labels` (fixed at initialization, never optimized) control global content, and ε cannot make up for a wrong draw. A 4 dB / 0.2 LPIPS gap comes from noise labels alone.
- ε is not recovered even when the image is (‖ε − ε*‖ / ‖ε*‖ ≈ 1.7): many different ε give nearly the same image.
- Even in range with known noise labels, CSGM doesn't reach LR loss 0 (6–51): the optimization itself is imperfect.

**What this means for the ablation**
- **Representation error (R1)** explains why CSGM on real images sits about 3.4 dB below its own HR-fit ceiling and 3.9 dB below bicubic. It's the gap image adaptation (θ) is meant to close. But Checks 1–2 show that adapting θ through the SR loss alone moves the error into the invented bands instead of closing it.
- **Noise labels are an unexplored degree of freedom.** Since they decide content, they can be searched or relaxed: more restarts with different `noise_labels` chosen by LR loss (Check 7), or optimizing the continuous conditioning vector (class + noise-label embeddings, 768-d) jointly with ε (Check 9).
- Patch-grid artifacts are intrinsic to the generator (Check 3) and won't disappear with any inversion trick; they bound perceptual quality.

---
# Check 6: HR reconstruction without degradation: CSGM vs IAGAN-like (A = identity)

**Question.** If the full HR image is known (no degradation, A = identity), how well can each approach reconstruct it with the drift generator?
- **CSGM-like:** frozen generator, optimize ε only.
- **IAGAN-like:** start from the CSGM solution and optimize the generator weights θ **and** ε jointly.

This is the representation experiment of the IAGAN paper, here with a drift model instead of a GAN. It separates two causes of poor SR results: the generator *can't represent* the image (fixable by adaptation?) vs SR ×4 being *ill-posed* (adaptation fits whatever the loss allows).

**This is a diagnostic: both methods use HR as the target.** Development images only, correct (oracle) label.

**What exactly was done**
- **CSGM-HR** = Check 5 R1: ‖HR − f(ε)‖² over ε, frozen generator, Adam lr 0.02, 5,000 steps from `initialize(image, restart)`, 2 restarts, keep the lower final HR loss.
- **IAGAN-HR** (new): from that R1 solution (its ε and `noise_labels`; same class label), minimize ‖HR − f_θ(ε)‖² jointly over θ (all 241 leaves, 134M parameters, starting from the pretrained weights) and ε, with two Adam optimizers, **2,000 steps**:
  - θ lr **1e-6** and **1e-5**, ε lr 1e-3 (the same learning rates as the SR IADrift runs), each run separately from the same start.
- **Where it ran:** θ-gradients need the `platform` allocator, which must not be set in this notebook (Check 4 reproducibility), so the training ran in `scripts/run_hr_fit_iadrift_dev.py` (separate process, tmux). This section only loads its cache: `~/work/datasets/ImageNetV2/hr_fit_iadrift_dev_v1/`.
- **Monitoring** (in that script, compiled path): every 50 steps, PSNR, LPIPS, band SNR and LR loss against HR / y. Step 0 is the R1 solution.
- **Final metrics:** eager generator output at step 2,000, scored here with the same `evaluate` as all other checks, plus the Check 3c patch-grid score.
- **Context rows** (SR ×4, from earlier checks): bicubic, CSGM SR (saved) and IADrift SR (θ lr 1e-5, ε lr 1e-3, 1,000 steps, Check 1).

**Reading the result.**
- If IAGAN-HR reaches a high PSNR and low LPIPS **and** loses the patch-grid texture, adaptation can close the representation gap: the drift generator plus adaptation *can* express real images, and the SR failures (Checks 1–2) come from ill-posedness (unconstrained high bands).
- If IAGAN-HR keeps the blocky or painterly texture even with HR known, the artifacts are built into the architecture (16-pixel linear patch output), and no SR loss can be expected to remove them.

In [ ]:
# Check 6: load the IAGAN-like HR-fit runs (trained in scripts/run_hr_fit_iadrift_dev.py) and compare with CSGM-HR (R1).
HRFIT_DIR = DATA_DIR / "hr_fit_iadrift_dev_v1"
HRFIT_SETTINGS = [(1e-6, 1e-3), (1e-5, 1e-3)]
HRFIT_STEPS = 2_000

hrfit = {}
for lt, le in HRFIT_SETTINGS:
    for i in DEV_IMAGES:
        path = HRFIT_DIR / f"img{i:05d}_lrt{lt:g}_lre{le:g}_steps{HRFIT_STEPS}.npz"
        assert path.exists(), f"missing {path}: run scripts/run_hr_fit_iadrift_dev.py first"
        with np.load(path) as data:
            result, x = json.loads(str(data["result_json"])), data["x"]
        assert result["r1_restart"] == r1[i]["restart"]  # starts from the Check 5 R1 solution
        hrfit[(lt, i)] = {"trace": result["trace"], "x": x, "metrics": evaluate(x, images[i]["hr"], images[i]["y"])}

ROWS6 = [("bicubic (SR ×4)", lambda i: (images[i]["outputs"]["bicubic"], images[i]["metrics"]["bicubic"])),
         ("CSGM SR ×4 (saved)", lambda i: (images[i]["outputs"]["CSGM"], images[i]["metrics"]["CSGM"])),
         ("IADrift SR ×4 (Check 1)", lambda i: (images[i]["iadrift"][MAIN_CFG]["IADrift"],
                                                images[i]["iad_metrics"][MAIN_CFG]["IADrift"])),
         ("**CSGM-HR** (R1, ε only)", lambda i: (r1[i]["x"], r1[i]["metrics"]))]
ROWS6 += [(f"**IAGAN-HR** (θ lr {lt:g}, ε lr {le:g})", lambda i, lt=lt: (hrfit[(lt, i)]["x"], hrfit[(lt, i)]["metrics"]))
          for lt, le in HRFIT_SETTINGS]

display(Markdown("**Mean over 4 development images.** CSGM-HR and IAGAN-HR use HR as the target (diagnostic); "
                 "the SR rows are for context.\n\n" + summary_table([(n, [get(i)[1] for i in DEV_IMAGES]) for n, get in ROWS6])))

lines = ["| Method | patch-grid score (≈ 1 = none) | " + " | ".join(f"{i} ({class_name(images[i]['label'])}): PSNR / LPIPS" for i in DEV_IMAGES) + " |",
         "|---|---:|" + "---:|" * len(DEV_IMAGES)]
for n, get in ROWS6:
    lines.append(f"| {n} | {np.mean([grid_score(get(i)[0]) for i in DEV_IMAGES]):.2f} | "
                 + " | ".join(f"{get(i)[1]['psnr']:.2f} / {get(i)[1]['lpips']:.3f}" for i in DEV_IMAGES) + " |")
lines.append(f"| HR (truth) | {np.mean([grid_score(images[i]['hr']) for i in DEV_IMAGES]):.2f} | " + " | ".join("—" for _ in DEV_IMAGES) + " |")
display(Markdown("**Per image, and the Check 3c patch-grid score**\n\n" + "\n".join(lines)))

# Progress of the IAGAN-like fit (compiled path, mean over images).
CHECKPOINTS = [0, 50, 100, 250, 500, 1000, 2000]
lines = ["| θ lr | " + " | ".join(f"step {s}" for s in CHECKPOINTS) + " |", "|---:|" + "---:|" * len(CHECKPOINTS)]
for lt, _ in HRFIT_SETTINGS:
    traces = [hrfit[(lt, i)]["trace"] for i in DEV_IMAGES]
    idx = [traces[0]["step"].index(s) for s in CHECKPOINTS]
    mp, ml = np.mean([t["psnr"] for t in traces], axis=0), np.mean([t["lpips"] for t in traces], axis=0)
    lines.append(f"| {lt:g} | " + " | ".join(f"{mp[j]:.2f} dB / {ml[j]:.3f}" for j in idx) + " |")
display(Markdown("**IAGAN-HR over the steps: mean PSNR / LPIPS** (step 0 = the CSGM-HR solution)\n\n" + "\n".join(lines)))

panels = [("psnr", "PSNR vs HR (dB)"), ("lpips", "LPIPS vs HR (lower is better)")] + [(b, f"SNR {b} (dB)") for b in BAND_NAMES]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for (key, title), ax in zip(panels, axes.flat):
    for k, (lt, _) in enumerate(HRFIT_SETTINGS):
        traces = [hrfit[(lt, i)]["trace"] for i in DEV_IMAGES]
        ax.plot(traces[0]["step"], np.mean([t[key] for t in traces], axis=0), color=f"C{k}", label=f"IAGAN-HR, θ lr {lt:g}")
    for base, color, get in (("bicubic (SR ×4)", "k", lambda i: images[i]["metrics"]["bicubic"]),
                             ("CSGM SR ×4", "gray", lambda i: images[i]["metrics"]["CSGM"])):
        ms = [get(i) for i in DEV_IMAGES]
        ax.axhline(np.mean([m[key] if key in ("psnr", "lpips") else m["band_snr_db"][key] for m in ms]),
                   color=color, linestyle="--", linewidth=1, label=base)
    ax.set(title=title, xlabel="IAGAN-HR step (0 = CSGM-HR)")
    ax.grid(alpha=0.3)
axes.flat[0].legend(fontsize=8)
fig.suptitle("IAGAN-like fit to HR (A = identity), mean over 4 development images", fontsize=11)
fig.tight_layout()
plt.show()

for i in DEV_IMAGES:
    shown = [(n.replace("**", ""), *get(i)) for n, get in ROWS6[2:]]  # IADrift SR, CSGM-HR, IAGAN-HR x2
    side_by_side(images[i]["hr"], f"Original HR, development image {i}\n{class_name(images[i]['label'])}", shown)

### Check 6 conclusions (4 development images; diagnostic, HR is the target)

**1. With the image known, adaptation closes the representation gap almost completely.**

| HR fit (A = identity) | PSNR | LPIPS | patch-grid score |
|---|---:|---:|---:|
| CSGM-HR (ε only, 5,000 steps) | 24.44 dB | 0.332 | 1.06 |
| IAGAN-HR, θ lr 1e-6 (2,000 steps) | 28.29 dB | 0.189 | 0.99 |
| IAGAN-HR, θ lr 1e-5 (2,000 steps) | **40.09 dB** | **0.030** | 0.97 (HR: 0.97) |

- At θ lr 1e-5 every image reaches 38.5–42.3 dB and LPIPS 0.007–0.050: visually indistinguishable from HR, including fine texture such as the crane's feathers (image 403 crop).
- The gain grows steadily with steps (mean 24.6 → 31.0 dB at step 500 → 35.9 at 1,000 → 41.2 at 2,000; compiled-path curve, about 1 dB above eager at this level). It hasn't saturated at 2,000 steps.
- θ lr 1e-6 is too slow for 2,000 steps (28.3 dB) and still shows patch blocks in the crops.

**2. The adapted generator has no intrinsic patch-grid or oil-paint problem.** IAGAN-HR outputs have a patch-grid score equal to HR's (0.97) and no visible blocks. So the 16-pixel artifacts of Checks 1–3 are **not unavoidable for the architecture** once θ is adapted to a target that pins down every frequency.

**3. CSGM-HR's limit is representation.** ε alone stalls at 24.4 dB with visibly blurry, blocky fits (low-band SNR only 19.7 dB) — the same generator that, adapted, reaches 40 dB.

**What this means for SR ×4 (Checks 1–2)**
- The *same* adaptation (same optimizer, same learning rates) that reconstructs HR almost perfectly produces oil-paint texture under the SR loss. The difference is only the loss: with A = identity every frequency is constrained; with A_sr4 everything above 0.125 cycles/pixel is unconstrained, and θ's 134M parameters fill it with whatever reduces the LR loss.
- So the drift generator + IAGAN-style adaptation **can represent** these images; the SR failure is **ill-posedness**, not representation. This is the key difference from GAN-based IAGAN, where (hypothesis) the conv generator's inductive bias keeps adapted outputs natural even when the loss doesn't constrain the high frequencies.
- This points the mitigation work at the **unconstrained frequencies** (regularizing θ, e.g. a weight anchor, restricting which layers adapt, or early stopping), not at the representation capacity.

---
# Check 7: Noise-label search for CSGM SR ×4 (selection by LR loss only)

**Question.** Check 5 R3 showed that, for generated targets, the right `noise_labels` are worth about 4 dB and 0.2 LPIPS. These 32 discrete values (0–63) are drawn at random and never optimized. On **real** photos, does searching over noise labels, with the choice made only from y, give better CSGM reconstructions?

**What exactly was done** (runs in `scripts/run_noise_label_search_dev.py`, without the `platform` allocator; this section loads its cache `~/work/datasets/ImageNetV2/noise_label_search_dev_v1/`):
- **Images:** the same 4 development images, correct (oracle) label.
- **Candidates:** 16 per image = the 2 baseline ε starts (`initialize(image, restart)`, restart 0 and 1) × 8 noise-label sets.
  - Set 0 is the baseline's own noise labels, so the 2 baseline runs are among the candidates.
  - Sets 1–7 are new random draws (seed 2033, folded with image, restart and draw).
- **Screening:** every candidate runs **500** CSGM steps (same optimizer as the baseline: Adam lr 0.02 on ε, the runner's own jitted `train_step`).
- **Selection:** the **2** candidates with the lowest screening LR loss run the full **5,000** steps; the one with the lowest final LR loss is the result. **HR is never used for any choice.**
- **Reproduction test:** before the search, candidate (image 8117, restart 1, set 0) at 5,000 steps reproduced the saved baseline run bit for bit (final LR loss 70.9893).
- **Compute:** 16 × 500 + 2 × 5,000 = 18,000 steps per image, vs 10,000 for the baseline (2 × 5,000), so 1.8×.

**Analysis here.**
- Search result vs the saved baseline (CSGM best of 2 restarts), with and without back-projection: PSNR, LPIPS, LR loss, band SNR, paired per image.
- **Is LR loss a good selection signal?** For all 64 screening outputs (500 steps), HR PSNR and LPIPS against screening LR loss (HR used for analysis only), with Spearman rank correlations per image.
- Side-by-side images: HR, bicubic, CSGM baseline, CSGM noise-label search, and both + BP.

In [ ]:
# Check 7: noise-label search results (runs in scripts/run_noise_label_search_dev.py; loaded here).
from scipy.stats import spearmanr

NLS_DIR = DATA_DIR / "noise_label_search_dev_v1"
nls_summary = json.loads((NLS_DIR / "summary.json").read_text())


def load_run(path):
    with np.load(path) as data:
        return json.loads(str(data["metrics_json"])), {k: data[k] for k in data.files if k != "metrics_json"}


nls = {}
for i in DEV_IMAGES:
    s = nls_summary[str(i)]
    m, a = load_run(NLS_DIR / f"final_{'img%05d' % i}_{s['best']}.npz")
    x, y, hr = a["x_hat"], images[i]["y"], images[i]["hr"]
    x_bp = backproject(x, y)
    screen = []
    for key, loss in s["screen"].items():
        _, sa = load_run(NLS_DIR / f"screen_{'img%05d' % i}_{key}.npz")
        screen.append({"key": key, "lr_loss": loss, "psnr": psnr(hr, sa["x_hat"]), "lpips": lpips_dist(sa["x_hat"], hr),
                       "baseline_labels": key.endswith("_nl0")})
    nls[i] = {"best": s["best"], "final_loss": m["final_loss"], "x": x, "x_bp": x_bp,
              "metrics": evaluate(x, hr, y), "metrics_bp": evaluate(x_bp, hr, y), "screen": screen}
    print(f"image {i}: best {s['best']} (final LR loss {m['final_loss']:.1f}; baseline r{images[i]['csgm_restart']} "
          f"{images[i]['metrics']['CSGM']['lr_loss']:.1f} eager LR loss)")

base = [images[i]["metrics"]["CSGM"] for i in DEV_IMAGES]
base_bp = [images[i]["metrics"]["CSGM + BP"] for i in DEV_IMAGES]
rows = [("bicubic", [images[i]["metrics"]["bicubic"] for i in DEV_IMAGES]),
        ("bicubic + BP", [images[i]["metrics"]["bicubic + BP"] for i in DEV_IMAGES]),
        ("CSGM baseline (best of 2 restarts)", base),
        ("**CSGM noise-label search**", [nls[i]["metrics"] for i in DEV_IMAGES])]
display(Markdown("**CSGM output, mean over 4 development images** (Δ columns: paired vs the CSGM baseline)\n\n"
                 + summary_table(rows, base, "baseline")))
rows = [("CSGM baseline + BP", base_bp), ("**CSGM noise-label search + BP**", [nls[i]["metrics_bp"] for i in DEV_IMAGES])]
display(Markdown("**After back-projection** (Δ columns: paired vs CSGM baseline + BP)\n\n" + summary_table(rows, base_bp, "baseline + BP")))

lines = ["| | " + " | ".join(f"{i} ({class_name(images[i]['label'])})" for i in DEV_IMAGES) + " |",
         "|---|" + "---:|" * len(DEV_IMAGES)]
lines.append("| chosen candidate | " + " | ".join(nls[i]["best"] + (" (= baseline labels)" if nls[i]["best"].endswith("_nl0") else "")
                                         for i in DEV_IMAGES) + " |")
for label, get in (("baseline: PSNR / LPIPS / LR loss", lambda i: images[i]["metrics"]["CSGM"]),
                   ("search: PSNR / LPIPS / LR loss", lambda i: nls[i]["metrics"]),
                   ("baseline + BP: PSNR / LPIPS", lambda i: images[i]["metrics"]["CSGM + BP"]),
                   ("search + BP: PSNR / LPIPS", lambda i: nls[i]["metrics_bp"])):
    lines.append(f"| {label} | " + " | ".join(
        f"{get(i)['psnr']:.2f} / {get(i)['lpips']:.3f}" + ("" if "BP" in label else f" / {get(i)['lr_loss']:.1f}")
        for i in DEV_IMAGES) + " |")
display(Markdown("**Per image**\n\n" + "\n".join(lines)))

# Is LR loss a good selection signal? All 16 screening outputs per image (500 steps); HR used for analysis only.
lines = ["| Image | Spearman(LR loss, PSNR) | Spearman(LR loss, LPIPS) | HR-best candidate by PSNR | its LR-loss rank (of 16) |",
         "|---|---:|---:|---|---:|"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
for c, i in enumerate(DEV_IMAGES):
    sc = nls[i]["screen"]
    loss = np.array([s["lr_loss"] for s in sc])
    p, l = np.array([s["psnr"] for s in sc]), np.array([s["lpips"] for s in sc])
    rank = {s["key"]: r for r, s in enumerate(sorted(sc, key=lambda s: s["lr_loss"]), start=1)}
    best_hr = sc[int(np.argmax(p))]["key"]
    lines.append(f"| {i} | {spearmanr(loss, p)[0]:+.2f} | {spearmanr(loss, l)[0]:+.2f} | {best_hr} | {rank[best_hr]} |")
    for ax, v in zip(axes, (p, l)):
        base_mask = np.array([s["baseline_labels"] for s in sc])
        ax.scatter(loss[~base_mask], v[~base_mask], color=f"C{c}", s=22, label=f"{i}")
        ax.scatter(loss[base_mask], v[base_mask], color=f"C{c}", s=60, marker="*", edgecolor="k", linewidth=0.5)
for ax, name in zip(axes, ("HR PSNR (dB)", "LPIPS (lower is better)")):
    ax.set(xscale="log", xlabel="screening LR loss (500 steps)", ylabel=name, title=f"{name} vs LR loss, all 64 screening runs")
    ax.grid(alpha=0.3)
axes[0].legend(title="image (★ = baseline labels)", fontsize=8)
plt.show()
display(Markdown("**Does lower LR loss mean better HR quality?** (screening outputs; negative PSNR correlation / positive LPIPS correlation = LR loss is a good signal)\n\n" + "\n".join(lines)))

for i in DEV_IMAGES:
    shown = [("bicubic", images[i]["outputs"]["bicubic"], images[i]["metrics"]["bicubic"]),
             ("CSGM baseline", images[i]["outputs"]["CSGM"], images[i]["metrics"]["CSGM"]),
             (f"CSGM search ({nls[i]['best']})", nls[i]["x"], nls[i]["metrics"]),
             ("CSGM baseline + BP", images[i]["outputs"]["CSGM + BP"], images[i]["metrics"]["CSGM + BP"]),
             ("CSGM search + BP", nls[i]["x_bp"], nls[i]["metrics_bp"])]
    side_by_side(images[i]["hr"], f"Original HR, development image {i}\n{class_name(images[i]['label'])}", shown)

### Check 7 conclusions (4 development images, correct label; small sample)

**1. Searching over noise labels improves the measurement fit a lot, and HR quality modestly.**
- LR loss: 50.5 → **30.0** on average (3939: 34.7 → 11.5; 403: 79.6 → 25.4; 347: 16.6 → 12.3). On 8117 the search kept the baseline's own candidate (its best new draw finished at 74.6 vs 71.0).
- Output without BP: PSNR **+0.55 dB** (3/4 better, 1 unchanged), LPIPS **−0.040** (2/4 better, 1 unchanged, 1 worse: 347, 0.267 → 0.302).
- After BP: PSNR +0.24 dB (2/4), LPIPS −0.013 (1/4 better; 403 and 347 slightly worse). BP already gives CSGM much of the fit the search buys.
- Image 3939 shows the largest effect: LPIPS 0.388 → **0.242** (and 0.213 with BP, the best LPIPS on any image in this notebook). Visually the doll's face and dress become coherent and the block artifacts shrink.

**2. LR loss is a good selection signal.** Across all 64 screening runs (500 steps), lower LR loss goes with higher HR PSNR (Spearman −0.85 to −0.94 per image) and, less strongly, with lower LPIPS (+0.39 to +0.72). On 4/4 images the screening candidate with the best HR PSNR was also rank 1 by LR loss. Choosing by y alone picks what an HR oracle would have picked at screening time.

**3. Noise labels matter on real photos too,** but much less than on in-range targets (Check 5 R3: about 4 dB / 0.2 LPIPS there vs +0.55 dB / −0.04 here). On real photos the representation gap (Check 6) dominates.

**4. Costs and caveats.**
- 1.8× the baseline compute (18,000 vs 10,000 steps per image), and only 16 candidates.
- The 500-step screening ranking doesn't always survive to 5,000 steps (8117: the screening winner finished worse than the baseline), so the final choice by full-length LR loss matters.
- 4 images, mixed per-image LPIPS: a promising direction, not yet a robust improvement.

**Decision input:** the noise-label search is a cheap, principled improvement to CSGM (selection uses only y) and a natural starting point for IADrift. It's a candidate component of the frozen test method, ideally confirmed on more development images first.

---
# Check 8: what do the noise labels z control vs ε? (generator analysis)

**What was done** (run in `notebooks/06_noise_label_role_dev.ipynb` on 2026-10-05; this section only loads its cache `~/work/datasets/ImageNetV2/noise_label_role_dev_v1/`). No optimization, no HR, no test images; only class labels and the frozen generator (CFG 1.0, eager path).
- **Hypothesis:** ε (cut into the generator's 16×16 token patches) is a local input; the 32 noise labels z (summed into the conditioning vector that modulates every token) are a global code: z selects an image-wide mode, ε the arrangement and detail.
- **A:** for 8 classes (the 4 development classes + 4 drawn with seed 2034), 8 codes z × 8 noise images ε = 64 samples per class. Two-way variance decomposition (z, ε, interaction) of position-independent descriptors (colour statistics, pooled AlexNet features, power spectrum) and position-specific ones (16×16 thumbnail, edge layout).
- **B:** from a base sample per class, 12 single-patch ε resamples and 12 single-coordinate z changes; where the output changes (per 16×16 tile) and how much.
- **C:** per-tile errors of the cached Check 5 R3 reconstructions, known vs random z (details in notebook 06).

In [ ]:
# Check 8: load the notebook 06 results (generator-only analysis; no training here).
ROLE_DIR = DATA_DIR / "noise_label_role_dev_v1"
role = json.loads((ROLE_DIR / "summary.json").read_text())
CLASSES8 = role["classes"]
DESC8 = ["colour mean/std", "radial power spectrum", "pooled AlexNet features", "16×16 thumbnail (layout)",
         "edge layout (32×32 gradient)"]
lines = ["| Descriptor | z share | ε share | interaction |", "|---|---:|---:|---:|"]
for d in DESC8:
    s = {p: np.mean([role["anova_shares"][f"{c}|{d}"][p] for c in CLASSES8]) for p in ("z", "ε", "interaction")}
    lines.append(f"| {d} | {s['z']:.2f} | {s['ε']:.2f} | {s['interaction']:.2f} |")
display(Markdown(f"**A. Share of variation due to z, ε and their interaction, mean over {len(CLASSES8)} classes**\n\n" + "\n".join(lines)))

loc8 = {c: json.loads((ROLE_DIR / f"locality_class{c:03d}.json").read_text()) for c in CLASSES8}
ep8 = [r for c in CLASSES8 for r in loc8[c]["eps_patch"]]
zc8 = [r for c in CLASSES8 for r in loc8[c]["z_coord"]]
lines = ["| Perturbation | n | share in perturbed tile | share in 3×3 | normalized entropy (1 = uniform) | RMS change |",
         "|---|---:|---:|---:|---:|---:|",
         f"| one ε patch | {len(ep8)} | {np.mean([r['in_tile'] for r in ep8]):.3f} | {np.mean([r['in_3x3'] for r in ep8]):.3f} | "
         f"{np.mean([r['entropy'] for r in ep8]):.2f} | {np.mean([r['rms_change'] for r in ep8]):.3f} |",
         f"| one noise-label coordinate | {len(zc8)} | — | — | {np.mean([r['entropy'] for r in zc8]):.2f} | "
         f"{np.mean([r['rms_change'] for r in zc8]):.3f} |",
         "| uniform spread (reference) | — | 0.004 | 0.035 | 1.00 | — |"]
display(Markdown("**B. Locality of changes**\n\n" + "\n".join(lines)))

c = 134  # crane
with np.load(ROLE_DIR / f"grid_class{c:03d}.npz") as data:
    grid = data["x"].astype(np.float32)
fig, axes = plt.subplots(8, 8, figsize=(15, 15))
for kz in range(8):
    for je in range(8):
        axes[kz, je].imshow(to_img(grid[kz, je]))
        axes[kz, je].set_xticks([])
        axes[kz, je].set_yticks([])
    axes[kz, 0].set_ylabel(f"z{kz}", fontsize=9)
for je in range(8):
    axes[0, je].set_title(f"ε{je}", fontsize=9)
fig.suptitle(f"Class {c} ({class_name(c)}): rows = noise-label codes z, columns = noise ε (random samples, no optimization)", fontsize=11)
fig.subplots_adjust(left=0.03, right=0.99, bottom=0.01, top=0.96, wspace=0.03, hspace=0.03)
plt.show()

### Check 8 conclusions (8 classes; generator analysis, no HR, no test images)

**The hypothesis is half right: z is global, but it is much more than a "style" code. z is the dominant latent; it selects the whole scene, including layout. ε is a weak, mostly local detail input.**

**A. What each input controls** (share of variation over the 8 × 8 grids, mean over 8 classes):

| Descriptor | z | ε | interaction |
|---|---:|---:|---:|
| colour mean/std (position-independent) | **0.82** | 0.03 | 0.15 |
| radial power spectrum (position-independent) | **0.68** | 0.05 | 0.26 |
| pooled AlexNet features (position-independent) | **0.56** | 0.06 | 0.38 |
| 16×16 thumbnail (layout) | **0.44** | 0.10 | 0.46 |
| edge layout | **0.37** | 0.10 | 0.53 |

- As predicted, z dominates the position-independent descriptors (palette, spectrum, texture/style), in every class.
- **Not as predicted:** z also explains 4× more of the *layout* variation than ε does (0.44 vs 0.10). The rest is interaction: what ε does depends on z.
- The grids show it directly (e.g. crane): each **row** (one z) is a different scene type: a flock in grass, a single bird by water, a close-up head, white cranes in a park, a crowned crane. Each **column** (one ε) only shifts pose and positions within that scene.

**B. Locality** (96 perturbations each):
- Changing **one ε patch**: 5.9% of the change lands in that tile and 19% in its 3×3 neighbourhood (15× and 5× uniform spread). So ε acts locally, but most of the change (81%) still spreads through attention (entropy 0.71). And the change is small: RMS 0.046.
- Changing **one of the 32 noise-label coordinates**: the change is spread over the whole image (entropy 0.90; the largest tile gets only 3%) and is **8.5× larger** (RMS 0.39).

**C. R3 reanalysis:** with random z, the error rises in **84–98% of tiles** for 3 of the 4 in-range targets (spread evenly: entropy 0.86–0.93). On the fourth (8117), known and random z are equally poor (that target wasn't recovered well even with known z). So wrong-z errors are global, as hypothesized.

**Interpretation and implications**
- In this drift generator, **the discrete noise labels carry most of the "what" of an image** (scene composition, viewpoint, palette, texture), and ε mainly adjusts local detail within the scene z selects.
- **Standard CSGM optimizes only the weak half of the latent.** With a randomly drawn z, CSGM must push ε far off-distribution to imitate a different scene (Check 3b: per-patch norms up to 4.8×), which is where tiles and wrong textures appear. This explains Check 5 R3 (wrong z → wrong content everywhere) and why the noise-label search (Check 7) helps, and suggests stronger z search or a continuous relaxation of the conditioning vector as the natural next step for inverting drift generators.
- Caveats: 8 classes, 8 codes and 8 ε per class; the descriptors are coarse; shares depend on the descriptors chosen.

---
# Check 9: optimizing the noise labels z: is it possible, and is joint (ε, z) worth the compute?

**Why.** Check 8 showed z is the dominant latent (scene, layout, palette), but CSGM keeps it fixed at a random draw. z is discrete, but it enters the network only through the conditioning vector
cond = E_class[c] + Σᵢ Eᵢ[zᵢ] + 0.02·cfg term (768-d, summed in bf16, as upstream), which drives the AdaLN scale/shift/gate of every block. So z can be relaxed and optimized by gradient, with the generator frozen.

**What exactly was done** (runs in `scripts/run_z_opt_dev.py`, no `platform` allocator, no θ-gradients; this section loads its cache `~/work/datasets/ImageNetV2/z_opt_dev_v1/`). Development images only, correct label; HR is never used for any choice.
- **Relaxations:**
  - **soft z:** per coordinate i, weights wᵢ = softmax(logitsᵢ) over its 64 embeddings; contribution wᵢ·Eᵢ (a convex combination of real embeddings, so it stays in the embeddings' hull). Logits start at 10 × one-hot of the starting code (weight ≈ 0.997), Adam lr 0.1. **Rounded z** = argmax per coordinate, evaluated with the final ε.
  - **free cond:** cond = cond(z₀) + s·δ with a free 768-d offset δ (s = RMS of the starting noise-label sum), Adam lr 0.01, penalty 10·mean(δ²).
- **Reproduction test:** with a hard code (and with a saturated one-hot soft code), the differentiable conditioning path reproduces the generator **bit for bit** (max difference 0).
- **Fine-tune experiments** (from each image's best saved CSGM solution, +2,000 steps, Adam lr 0.02 on ε as CSGM):
  - **ε only:** the compute-matched control (just more CSGM steps, same code path with z's learning rate set to 0);
  - **soft z only:** ε frozen (is z optimization possible on its own?);
  - **ε + soft z;**
  - **ε + free cond.**
- **From scratch, compute-matched with the CSGM baseline:** **scratch ε + soft z** from `initialize(image, restart)`, 5,000 steps, 2 restarts, keep the lowest final LR loss; compared with the saved CSGM baseline (same starts, same steps, z fixed).
- **Settings were not tuned** (single learning rates, single penalty). This is a feasibility test.
- **Metrics:** as all checks (PSNR, LPIPS, LR loss, band SNR), plus blockiness; raw and + BP. z diagnostics: coordinates whose argmax changed, mean largest weight, normalized weight entropy; ε per-patch norm spread (Check 3b).

In [ ]:
# Check 9: load the z-optimization runs and compare (development images, correct label).
from freq_metrics import blockiness

ZOPT_DIR = DATA_DIR / "z_opt_dev_v1"
ARM9 = {"F0_eps_only": "ε only", "F1_z_only": "soft z only", "F2_eps_softz": "ε + soft z",
        "F3_eps_freecond": "ε + free cond", "S2": "scratch ε + soft z"}  # run name -> label


def zrun(i, name):
    with np.load(ZOPT_DIR / f"img{i:05d}_{name}.npz") as data:
        return json.loads(str(data["metrics_json"])), {k: data[k] for k in data.files if k != "metrics_json"}


def ev(x, i):
    m = evaluate(x, images[i]["hr"], images[i]["y"])
    m["blockiness"] = blockiness(x)
    return m


z9 = {}
for i in DEV_IMAGES:
    e = {"CSGM (saved start)": images[i]["outputs"]["CSGM"]}
    for name, label9 in (("F0_eps_only", "ε only (+2,000)"), ("F1_z_only", "soft z only (+2,000)"),
                         ("F2_eps_softz", "ε + soft z (+2,000)"), ("F3_eps_freecond", "ε + free cond (+2,000)")):
        m, a = zrun(i, name)
        e[label9] = a["x"]
        if "x_rounded" in a and name != "F0_eps_only":
            e[label9.replace("(+2,000)", "rounded z")] = a["x_rounded"]
        z9[(i, name)] = m
    scratch = [zrun(i, f"S2_scratch_softz_r{r}") for r in RESTARTS]
    r_best = min(RESTARTS, key=lambda r: (scratch[r][0]["lr_loss"], r))
    z9[(i, "S2")] = {**scratch[r_best][0], "restart_kept": r_best, "eps": scratch[r_best][1]["eps"]}
    e["scratch ε + soft z (5,000)"] = scratch[r_best][1]["x"]
    e["scratch, rounded z"] = scratch[r_best][1]["x_rounded"]
    e["check 7 noise-label search"] = nls[i]["x"]
    images[i]["z9"] = {n: {"x": x, "m": ev(x, i), "m_bp": ev(backproject(x, images[i]["y"]), i)} for n, x in e.items()}
NAMES9 = list(images[DEV_IMAGES[0]]["z9"])


def table9(names, ref, bp=False):
    key = "m_bp" if bp else "m"
    head = "| Method | PSNR | LPIPS ↓ | blockiness | LR loss | SNR mid | SNR high | Δ PSNR vs ref | PSNR better | Δ LPIPS vs ref | LPIPS better |"
    lines = [head, "|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|"]
    rp = np.array([images[i]["z9"][ref][key]["psnr"] for i in DEV_IMAGES])
    rl = np.array([images[i]["z9"][ref][key]["lpips"] for i in DEV_IMAGES])
    for n in names:
        ms = [images[i]["z9"][n][key] for i in DEV_IMAGES]
        p, l = np.array([m["psnr"] for m in ms]), np.array([m["lpips"] for m in ms])
        lines.append(f"| {n} | {p.mean():.2f} | {l.mean():.3f} | {np.mean([m['blockiness'] for m in ms]):.3f} | "
                     f"{np.mean([m['lr_loss'] for m in ms]):.2f} | {np.mean([m['band_snr_db']['mid'] for m in ms]):.2f} | "
                     f"{np.mean([m['band_snr_db']['high'] for m in ms]):.2f} | {np.mean(p - rp):+.2f} | {int(np.sum(p > rp + 1e-9))}/4 | "
                     f"{np.mean(l - rl):+.3f} | {int(np.sum(l < rl - 1e-9))}/4 |")
    return "\n".join(lines)


FT = [n for n in NAMES9 if not n.startswith(("scratch", "check 7"))]
display(Markdown("**Fine-tune from the CSGM solution (+2,000 steps), raw output; Δ vs the compute-matched ε-only control**\n\n"
                 + table9(FT, "ε only (+2,000)")))
display(Markdown("**Same, after back-projection; Δ vs ε only + BP**\n\n" + table9(FT, "ε only (+2,000)", bp=True)))
SC = ["CSGM (saved start)", "scratch ε + soft z (5,000)", "scratch, rounded z", "check 7 noise-label search"]
display(Markdown("**From scratch at equal steps (5,000, best of 2 restarts by LR loss); Δ vs the saved CSGM baseline** "
                 "(check 7 search: 18,000 steps, for reference)\n\n" + table9(SC, "CSGM (saved start)")))
display(Markdown("**Same, after back-projection**\n\n" + table9(SC, "CSGM (saved start)", bp=True)))

# Compute cost and z / ε diagnostics.
lines = ["| Variant | s per step (mean) | coords changed (of 32) | mean largest weight | weight entropy (0 = discrete) | δ RMS | "
         "ε patch-norm std (Gaussian 0.026) |", "|---|---:|---:|---:|---:|---:|---:|"]
for name in ("F0_eps_only", "F1_z_only", "F2_eps_softz", "F3_eps_freecond", "S2"):
    ms = [z9[(i, name)] for i in DEV_IMAGES]
    sps = np.mean([m["seconds"] / m["steps"] for m in ms])
    if name in ("F1_z_only", "F2_eps_softz", "S2"):
        zcols = (f"{np.mean([m['coords_changed'] for m in ms]):.1f} | {np.mean([m['max_weight_mean'] for m in ms]):.3f} | "
                 f"{np.mean([m['weight_entropy_mean'] for m in ms]):.3f}")
    else:
        zcols = "— | — | —"
    dcol = f"{np.mean([m['delta_rms'] for m in ms]):.3f}" if name == "F3_eps_freecond" else "—"
    eps_list = [z9[(i, "S2")]["eps"] for i in DEV_IMAGES] if name == "S2" else [zrun(i, name)[1]["eps"] for i in DEV_IMAGES]
    eps_std = np.mean([eps_stats(e)[0]["patch_norm_std"] for e in eps_list])
    lines.append(f"| {ARM9[name]} | {sps * 1000:.1f} ms | {zcols} | {dcol} | {eps_std:.3f} |")
display(Markdown("**Compute and diagnostics** (mean over 4 images; saved CSGM ε patch-norm std ≈ 0.41)\n\n" + "\n".join(lines)))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
for name, color in (("F0_eps_only", "k"), ("F1_z_only", "C0"), ("F2_eps_softz", "C1"), ("F3_eps_freecond", "C2")):
    tr = [zrun(i, name)[1] for i in DEV_IMAGES]
    axes[0].plot(tr[0]["trace_steps"], np.mean([t["trace_loss"] for t in tr], axis=0), color=color, label=ARM9[name])
axes[0].set(yscale="log", xlabel="fine-tune step (from CSGM)", ylabel="LR loss (compiled)", title="Fine-tune: LR loss, mean over 4 images")
axes[0].legend(fontsize=8)
for r, ls in zip(RESTARTS, ("-", "--")):
    tr = [zrun(i, f"S2_scratch_softz_r{r}")[1] for i in DEV_IMAGES]
    axes[1].plot(tr[0]["trace_steps"], np.mean([t["trace_loss"] for t in tr], axis=0), color="C1", linestyle=ls, label=f"scratch ε + soft z, r{r}")
    with np.load(CSGM_CACHE / f"img{DEV_IMAGES[0]:05d}_oracle_r{r}.npz") as data:
        steps_c = data["trace_steps"]
    base = []
    for i in DEV_IMAGES:
        with np.load(CSGM_CACHE / f"img{i:05d}_oracle_r{r}.npz") as data:
            base.append(data["trace_loss"])
    axes[1].plot(steps_c, np.mean(base, axis=0), color="k", linestyle=ls, label=f"CSGM (z fixed), r{r}")
axes[1].set(yscale="log", xlabel="step", ylabel="LR loss (compiled)", title="From scratch: joint vs CSGM, mean over 4 images")
axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

for i in (3939, 403):
    shown = [(n, images[i]["z9"][n]["x"], images[i]["z9"][n]["m"]) for n in
             ("CSGM (saved start)", "ε only (+2,000)", "ε + soft z (+2,000)", "ε + free cond (+2,000)",
              "scratch ε + soft z (5,000)")]
    side_by_side(images[i]["hr"], f"Original HR, development image {i}\n{class_name(images[i]['label'])}", shown)

### Check 9 conclusions (4 development images, correct label; untuned settings; small sample)

**1. Is z optimization possible? Yes, through the conditioning vector, but only as a soft (relaxed) code.**
- The differentiable path reproduces the generator bit for bit, and optimizing soft z lowers the LR loss (alone: 50 → 48; with ε from scratch: down to 4–15).
- **The optimum is not a discrete code.** From scratch, the weights spread (mean largest weight 0.80, 13.5 of 32 coordinates change their argmax), and **rounding to the nearest discrete code destroys the reconstruction** (LR loss 11 → 1,834, PSNR 21.7 → 13.8). Soft z finds mixtures of embeddings, not better codes. They stay inside the embeddings' convex hull but outside what the generator saw in training.

**2. Fine-tuning z from the CSGM solution is not worth it.** Starting from CSGM, z hardly moves (largest weight 0.97–0.99, no coordinate changes), and ε + soft z and ε + free cond equal the compute-matched ε-only control within noise (PSNR +0.02 / −0.03 dB; LPIPS −0.002 / −0.001; also after BP). The CSGM solution is a local optimum in which z is locked; extra compute is better spent on more ε steps.

**3. Joint (ε, soft z) from scratch is worth it for the raw output, at no extra cost.**
- Same starts and same 5,000 steps as CSGM, and nearly the same time per step (15.9 ms vs about 15 ms).
- **Measurement fit:** LR loss **10.8 vs 50.5**. **PSNR +0.62 dB** (2/4 images better), **LPIPS −0.045 (4/4 better)**, blockiness 1.32 vs 1.38.
- That matches the Check 7 noise-label search (PSNR +0.55, LPIPS −0.040) at **0.28× its compute** (5,000 vs 18,000 steps).
- **It fixes the scene, as Check 8 predicts:** on image 403 it is the only variant with the crane in the right pose and without CSGM's invented background (+2.7 dB).
- **ε stays much closer to Gaussian:** per-patch norm spread 0.18 vs 0.41 for CSGM. With z absorbing the scene, ε doesn't need to go far off-distribution (Check 3b).

**4. But after back-projection the advantage disappears.** Joint (ε, soft z) from scratch + BP vs CSGM + BP: PSNR +0.05 dB (1/4), LPIPS +0.003 (0/4). The Check 7 search + BP is still slightly ahead (PSNR +0.24, LPIPS −0.013). As in Check 1 and the IADrift test, most of the raw gain is measurement fit, which BP gives CSGM for free; the remaining content differences are mixed.

**5. Restart selection needs care.** With joint optimization both restarts reach LR losses (5–15) below the bf16 noise floor (about 5), so "keep the lowest LR loss" becomes nearly random. On 8117 and 3939 it kept the restart with the lower PSNR (8117: 19.51 vs 20.37 dB). Averaging restarts, or a fixed rule, would be needed.

**Answer to the question.** Optimizing z is possible (as a soft relaxation), and joint (ε, z) from scratch gives clearly better raw reconstructions and more natural ε at the same cost as CSGM. But on these 4 images it doesn't beat CSGM + BP, which is the strongest perceptual baseline we have (test: LPIPS 0.284). Useful as an ablation and an interpretation (conditioning-space adaptation sits between CSGM and IADrift); not, on this evidence, worth a post-hoc test run before the deadline.

---
# Check 10: discrete noise-label swaps: why late swaps fail, and do early swaps help?

**Why.** Check 9: z can only be optimized as a soft mixture; rounding to a real code destroyed the result, and fine-tuning z from the CSGM solution did nothing. Idea: keep z a real code and search over **discrete swaps** (one coordinate zᵢ → k at a time).

**What exactly was done** (runs in `scripts/run_z_swap_dev.py` / `run_z_swap_early_dev.py`, no `platform` allocator, no θ-gradients; cache `~/work/datasets/ImageNetV2/z_swap_dev_v1/`). Development images only, correct label; HR never used for any decision. All screening uses one compiled loss function (bf16 values from different compilations differ by a few %).
- **Smoke test (gradient-guided version):** at the CSGM solution, the first-order prediction from ∂L/∂(one-hot z) proposed 4 swaps per coordinate (128); the best 4 by exact loss plus "no swap" each got 100 ε steps; the lowest loss won. On image 8117, the best of 128 swaps raised the LR loss from about 68 to about 1,200, "no swap" won both rounds, and the gradient's ranking was useless (rank correlation −0.09 and −0.50). This motivated the two experiments below.
- **(2) Exhaustive scan.** At each image's best saved CSGM solution: exact LR loss of **all 32 × 63 = 2,016** single swaps with the current ε (no re-fit), and the first-order prediction for each.
- **(3) Early swaps.** From `initialize(image, restart)`, restarts 0 and 1, Adam lr 0.02 on ε, 5,000 ε steps along the path. After **100, 250, 500 and 1,000** steps, one round: all 2,016 swaps screened exactly; the best 4 and "no swap" each get 100 ε steps (continuing the Adam state); the lowest LR loss wins. **Control:** the same code without swaps (same starts, same compiled functions). Restart rule as CSGM: lowest final LR loss.
- **Settings untuned** (4 swap points, 4 finalists, 100-step re-fit). Early-swap runs take about 2× the control's time (screening 4 × 2,016 forward passes plus 4 × 400 extra re-fit steps).
- Metrics as all checks: PSNR, LPIPS, blockiness, LR loss, band SNR; raw and + BP.

In [ ]:
# Check 10: discrete noise-label swaps (development images, correct label). Loads cached runs only.
SWAP_DIR = DATA_DIR / "z_swap_dev_v1"


def sw_load(part, name):
    with np.load(SWAP_DIR / part / name) as data:
        return json.loads(str(data["metrics_json"])), {k: data[k] for k in data.files if k != "metrics_json"}


# (2) Exhaustive single-swap scan at the CSGM solution.
lines = ["| Image | LR loss at CSGM | swaps that lower the loss | best change | median change | rank corr. gradient vs exact |",
         "|---|---:|---:|---:|---:|---:|"]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for c_, i in enumerate(DEV_IMAGES):
    m, a = sw_load("scan", f"img{i:05d}_scan.npz")
    lines.append(f"| {i} ({class_name(images[i]['label'])}) | {m['base_lr_loss']:.1f} | {m['n_improving']}/{m['n_swaps']} | "
                 f"{m['best_delta']:+.0f} | {m['median_delta']:+.0f} | {m['spearman_pred_vs_exact']:+.2f} |")
    axes[0].hist(a["exact_delta"], bins=60, histtype="step", color=f"C{c_}", label=str(i))
    axes[1].scatter(a["pred_delta"], a["exact_delta"], s=3, alpha=0.3, color=f"C{c_}", label=str(i))
axes[0].axvline(0, color="k", linewidth=1)
axes[0].set(xlabel="change in LR loss from one swap (current ε)", ylabel="swaps", title="All 2,016 single swaps at the CSGM solution")
axes[1].axhline(0, color="k", linewidth=1)
axes[1].set(xlabel="first-order (gradient) prediction", ylabel="exact change", title="Gradient prediction vs exact change")
for ax in axes:
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()
display(Markdown("**(2) Exhaustive single-swap scan at each image's CSGM solution**\n\n" + "\n".join(lines)))

# (3) Early swaps vs the no-swap control.
sw = {}
for i in DEV_IMAGES:
    for tag in ("control", "early"):
        for r in RESTARTS:
            m, a = sw_load("early", f"img{i:05d}_{tag}_r{r}.npz")
            x = a["x"]
            mm, mb = evaluate(x, images[i]["hr"], images[i]["y"]), None
            mm["blockiness"] = blockiness(x)
            x_bp = backproject(x, images[i]["y"])
            mb = evaluate(x_bp, images[i]["hr"], images[i]["y"])
            mb["blockiness"] = blockiness(x_bp)
            sw[(i, tag, r)] = {"meta": m, "x": x, "x_bp": x_bp, "m": mm, "m_bp": mb}

lines = ["| Image | restart | control PSNR | early PSNR | Δ | control LPIPS | early LPIPS | Δ | control LR loss | early LR loss | swaps accepted (steps) |",
         "|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---|"]
for i in DEV_IMAGES:
    for r in RESTARTS:
        c0, e0 = sw[(i, "control", r)], sw[(i, "early", r)]
        steps_acc = ", ".join(str(rr["at_step"]) for rr in e0["meta"]["rounds"] if rr["accepted"])
        lines.append(f"| {i} | {r} | {c0['m']['psnr']:.2f} | {e0['m']['psnr']:.2f} | {e0['m']['psnr'] - c0['m']['psnr']:+.2f} | "
                     f"{c0['m']['lpips']:.3f} | {e0['m']['lpips']:.3f} | {e0['m']['lpips'] - c0['m']['lpips']:+.3f} | "
                     f"{c0['m']['lr_loss']:.1f} | {e0['m']['lr_loss']:.1f} | {e0['meta']['n_accepted']} ({steps_acc}) |")
display(Markdown("**(3) Early swaps vs no-swap control, every run (raw output)**\n\n" + "\n".join(lines)))


def chosen(i, tag):
    r = min(RESTARTS, key=lambda r: (sw[(i, tag, r)]["m"]["lr_loss"], r))  # restart rule: lowest final LR loss
    return sw[(i, tag, r)]


rows10 = {"bicubic + BP": lambda i, bp: images[i]["metrics"]["bicubic + BP"],
          "CSGM (saved baseline)": lambda i, bp: images[i]["metrics"]["CSGM + BP" if bp else "CSGM"],
          "control (no swaps, same code)": lambda i, bp: chosen(i, "control")["m_bp" if bp else "m"],
          "**early swaps**": lambda i, bp: chosen(i, "early")["m_bp" if bp else "m"]}
for bp in (False, True):
    lines = ["| Method | PSNR | LPIPS ↓ | blockiness | LR loss | SNR mid | SNR high | Δ PSNR vs control | better | Δ LPIPS vs control | better |",
             "|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|"]
    ref = [rows10["control (no swaps, same code)"](i, bp) for i in DEV_IMAGES]
    for n, f in rows10.items():
        ms = [f(i, bp) for i in DEV_IMAGES]
        p, l = np.array([m["psnr"] for m in ms]), np.array([m["lpips"] for m in ms])
        rp, rl = np.array([m["psnr"] for m in ref]), np.array([m["lpips"] for m in ref])
        blk = np.mean([m.get("blockiness", np.nan) for m in ms])
        lines.append(f"| {n} | {p.mean():.2f} | {l.mean():.3f} | {blk:.3f} | {np.mean([m['lr_loss'] for m in ms]):.2f} | "
                     f"{np.mean([m['band_snr_db']['mid'] for m in ms]):.2f} | {np.mean([m['band_snr_db']['high'] for m in ms]):.2f} | "
                     f"{np.mean(p - rp):+.2f} | {int(np.sum(p > rp + 1e-9))}/4 | {np.mean(l - rl):+.3f} | {int(np.sum(l < rl - 1e-9))}/4 |")
    display(Markdown(f"**(3) With the restart rule (lowest final LR loss), mean over 4 images, {'after back-projection' if bp else 'raw output'}**"
                     + (" (CSGM row = saved CSGM + BP)" if bp else "") + "\n\n" + "\n".join(lines)))

for i in (8117, 403):
    shown = [("CSGM (saved)", images[i]["outputs"]["CSGM"], images[i]["metrics"]["CSGM"]),
             ("control (no swaps)", chosen(i, "control")["x"], chosen(i, "control")["m"]),
             ("early swaps", chosen(i, "early")["x"], chosen(i, "early")["m"]),
             ("control + BP", chosen(i, "control")["x_bp"], chosen(i, "control")["m_bp"]),
             ("early swaps + BP", chosen(i, "early")["x_bp"], chosen(i, "early")["m_bp"])]
    side_by_side(images[i]["hr"], f"Original HR, development image {i}\n{class_name(images[i]['label'])}", shown)

### Check 10 conclusions (4 development images, correct label; untuned; small sample)

**1. Late swaps are ruled out.** At the CSGM solution, **none of the 2,016 single swaps lowers the LR loss on any of the 4 images** (best change +573 to +1,205; median +1,350 to +3,578). The gradient's first-order prediction ranks swaps poorly (rank correlation −0.21 to +0.43). Reason (consistent with Check 8): one noise-label coordinate changes the whole image, so a swap is a jump to another scene, which neither a linear prediction nor a short ε re-fit can handle once ε has spent 5,000 steps fitting the current scene. This is why z fine-tuning failed in Check 9 too.

**2. Early swaps work as intended: z stays a real code and the scene is chosen while ε is still unlocked.**
- Swaps were accepted in all 8 runs (1–3 each, almost all at steps 100 and 250); at step 100 some starts had many improving swaps even before the re-fit (403 r1: 178 of 2,016), i.e. their random z was a poor scene.
- Against the same code without swaps: **LR loss lower in 7/8 runs, LPIPS lower in 7/8 runs** (−0.003 to −0.100), PSNR higher in 4/8 (+0.40 to +1.36 dB) and slightly lower in 4/8 (−0.06 to −0.32 dB).
- With the restart rule: PSNR **+0.37 dB** (2/4), LPIPS **−0.022** (3/4), blockiness 1.33 vs 1.40.

**3. Unlike Checks 1, 9 and the IADrift test, the gain survives back-projection** (vs the same-code control + BP): **PSNR +0.30 dB (2/4), LPIPS −0.024 (3/4)**, mid- and high-band SNR both higher. Back-projection only fixes the measured bands; early swaps change the *content* (the scene), which BP can't supply. Image 403 shows it: early swaps + BP gets the crane's pose and neck right and drops part of the invented background (22.74 dB / LPIPS 0.483 vs 21.43 / 0.573 for the control + BP).

**4. Important caveat: implementation noise is as large as the effect.** The no-swap control is the same algorithm as the saved CSGM baseline, but compiled differently (bf16), and it ends up worse: after BP 21.62 dB / LPIPS 0.379 vs 21.86 / 0.328 for the saved baseline. So **against the saved CSGM + BP, early swaps + BP tie on PSNR (21.92 vs 21.86) and are worse on LPIPS (0.355 vs 0.328)**. With n = 4 images and 2 restarts, run-to-run variation from numerics and restart choice is about the size of the effect; the fair comparison is the same-code control, but the result needs more images (and restarts) before it can be called real.

**Cost:** about 2× plain CSGM time (4 screening rounds of 2,016 forward passes plus 4 × 400 extra re-fit steps); screening could be cut (fewer rounds; rounds 3–4 rarely mattered).

**Insight.** With a drift generator whose discrete code sets the scene, z must be chosen **early, discretely, and by measurement fit**, the opposite of IAGAN-style late adaptation. Early discrete swaps are the only z method here whose gain on the same-code control survived back-projection and showed in LPIPS; whether it beats the strongest baseline (CSGM + BP) is not established on 4 images.

---
**Check 10b preview (2026-10-05, late): early swaps vs REAL CSGM on 2 new development images** (9780, 8405; never used before). Script `scripts/run_z_swap_confirm_dev.py`; every ε step uses CSGM's own compiled step, so the only difference from CSGM is the swaps; swap rounds after steps 100 and 250 only; restart rule lowest final LR loss. Computed outside this notebook (cache `z_swap_dev_v1/confirm/`).

| Image | Method | restart kept | PSNR | LPIPS | PSNR + BP | LPIPS + BP |
|---|---|---:|---:|---:|---:|---:|
| 9780 | CSGM | r1 | 20.41 | 0.271 | 20.85 | 0.213 |
| 9780 | early swaps | r1 | 20.39 | 0.275 | 20.92 | 0.215 |
| 8405 | CSGM | r0 | 25.21 | 0.383 | 26.40 | 0.293 |
| 8405 | early swaps | r1 | 24.37 | 0.577 | 25.31 | 0.494 |

- **Against genuine CSGM, early swaps did not help:** a tie on 9780 and a clear loss on 8405 (after BP −1.09 dB, LPIPS +0.20). Per run, 1 of 4 better (9780 r0: +0.55 dB, not selected by the restart rule).
- On 8405 the restart rule kept the early-swap run with the lower LR loss (25.1 vs 37.2) but much worse HR quality: below about 30, LR loss doesn't discriminate between restarts.
- Together with Check 10 caveat 4, the most likely reading is that **the gain over the same-code control in Check 10 was mostly run-to-run variation, not an effect of the swaps.** Only 2 images, so this is a preview; but there is no sign that early swaps beat CSGM + BP.

---
# Check 11: better z-start + IADrift @100 (post-hoc, development images)

> Labelled **post-hoc**: "IADrift @100" was preferred because of the test-set outcome (protocol Section 9); everything here is development-only.

**Question.** On the test set, IADrift stopped at 100 steps gave a small, significant LPIPS gain over CSGM after BP. Does it help more when it starts from a better z (noise-label search, Check 7, or early swaps, Check 10)? Pipeline: **start → IADrift 100 steps → back-projection**.

**What exactly was done** (`scripts/run_combo_iadrift100_dev.py`, `platform` allocator for θ-gradients; cache `~/work/datasets/ImageNetV2/combo_iadrift100_dev_v1/`). Correct label; HR never used for any decision.
- **IADrift** exactly as frozen (joint θ + ε, Adam, θ lr 1e-5 on all leaves, ε lr 1e-3, `make_joint_step`), 100 steps, eager evaluation.
- **Starts** (cached; for each image and start type the restart with the lowest final LR loss):
  - **CSGM:** saved CSGM for the 4 development images used so far; Check 10b's CSGM for the 2 new ones (9780, 8405);
  - **search:** Check 7 noise-label search (4 images only);
  - **early swaps:** Check 10 (4 images) and Check 10b (9780, 8405).
- Each output with and without BP; PSNR, LPIPS, blockiness. Reference in every comparison: **CSGM + BP** of the same image (the strongest perceptual baseline). The 4 original images were used to design Checks 7–10; 9780 and 8405 are fresh.

In [ ]:
# Check 11: load the IADrift @100 runs from each start, back-project, score.
COMBO = DATA_DIR / "combo_iadrift100_dev_v1"
IMGS11 = [8117, 3939, 403, 347, 9780, 8405]
c11 = {}
for i in IMGS11:
    hr = images[i]["hr"] if i in images else load_hr256(DATA_DIR / index_rows[i]["relative_path"])
    y = images[i]["y"] if i in images else jnp.asarray(lr_all[i:i + 1], jnp.float32)
    for name in ("CSGM", "search", "early"):
        path = COMBO / f"img{i:05d}_{name}.npz"
        if not path.exists():
            continue
        with np.load(path) as data:
            for stage, x in (("start", data["x_start"]), ("start → IADrift @100", data["x_iad100"])):
                for bp in (False, True):
                    xx = backproject(x, y) if bp else x
                    m = evaluate(xx, hr, y)
                    m["blockiness"] = blockiness(xx)
                    c11[(i, name, stage, bp)] = m

PIPES = [(n, s) for n in ("CSGM", "search", "early") for s in ("start", "start → IADrift @100")]
for subset, title in ((IMGS11[:4], "4 original development images (all starts)"), (IMGS11[4:], "2 new development images (no search start)")):
    for bp in (False, True):
        ref = [c11[(i, "CSGM", "start", True)] for i in subset]
        lines = ["| Start | Stage | PSNR | LPIPS ↓ | blockiness | Δ PSNR vs CSGM + BP | better | Δ LPIPS vs CSGM + BP | better |",
                 "|---|---|---:|---:|---:|---:|---:|---:|---:|"]
        for n, s in PIPES:
            if not all((i, n, s, bp) in c11 for i in subset):
                continue
            ms = [c11[(i, n, s, bp)] for i in subset]
            p, l = np.array([m["psnr"] for m in ms]), np.array([m["lpips"] for m in ms])
            rp, rl = np.array([m["psnr"] for m in ref]), np.array([m["lpips"] for m in ref])
            lines.append(f"| {n} | {s} | {p.mean():.2f} | {l.mean():.3f} | {np.mean([m['blockiness'] for m in ms]):.3f} | "
                         f"{np.mean(p - rp):+.2f} | {int(np.sum(p > rp + 1e-9))}/{len(subset)} | {np.mean(l - rl):+.3f} | "
                         f"{int(np.sum(l < rl - 1e-9))}/{len(subset)} |")
        display(Markdown(f"**{title}, {'after BP' if bp else 'raw output'}** (reference: CSGM + BP)\n\n" + "\n".join(lines)))

lines = ["| Image | CSGM + BP | CSGM → IAD@100 + BP | search + BP | search → IAD@100 + BP | early + BP | early → IAD@100 + BP |",
         "|---|---:|---:|---:|---:|---:|---:|"]
for i in IMGS11:
    cell = lambda n, s: (f"{c11[(i, n, s, True)]['psnr']:.2f} / {c11[(i, n, s, True)]['lpips']:.3f}" if (i, n, s, True) in c11 else "—")
    lines.append(f"| {i} | " + " | ".join(cell(n, s) for n, s in PIPES) + " |")
display(Markdown("**Per image after BP: PSNR dB / LPIPS**\n\n" + "\n".join(lines)))

# Side by side (after BP): CSGM, CSGM -> IADrift @100, search, search -> IADrift @100, early swaps -> IADrift @100.
for i in IMGS11:
    hr = images[i]["hr"] if i in images else load_hr256(DATA_DIR / index_rows[i]["relative_path"])
    y = images[i]["y"] if i in images else jnp.asarray(lr_all[i:i + 1], jnp.float32)
    shown = []
    for n, s_, title in (("CSGM", "start", "CSGM + BP"), ("CSGM", "start → IADrift @100", "CSGM → IAD@100 + BP"),
                         ("search", "start", "search + BP"), ("search", "start → IADrift @100", "search → IAD@100 + BP"),
                         ("early", "start → IADrift @100", "early swaps → IAD@100 + BP")):
        path = COMBO / f"img{i:05d}_{n}.npz"
        if not path.exists():
            continue
        with np.load(path) as data:
            x = data["x_start"] if s_ == "start" else data["x_iad100"]
        shown.append((title, backproject(x, y), c11[(i, n, s_, True)]))
    lbl = images[i]["label"] if i in images else int(labels[i])
    side_by_side(hr, f"Original HR, development image {i}\n{class_name(lbl)}", shown)

### Check 11 conclusions (6 development images; post-hoc; small sample)

**No: a better z-start and IADrift @100 do not add up.** After back-projection, against CSGM + BP of the same image:

| Pipeline (after BP) | 4 original images: Δ PSNR / Δ LPIPS | 2 new images: Δ PSNR / Δ LPIPS |
|---|---|---|
| CSGM → IADrift @100 | −0.10 dB / −0.002 (3/4 better) | −0.74 dB / +0.008 |
| search (Check 7) | **+0.24 dB / −0.013** | — |
| search → IADrift @100 | −0.00 dB / −0.010 | — |
| early swaps (Check 10) | +0.07 dB / +0.028 | −0.51 dB / +0.102 |
| early swaps → IADrift @100 | −0.16 dB / +0.028 | −0.28 dB / +0.076 |

1. **IADrift @100 on its own, after BP, is roughly neutral**, matching the test set's direction and size on the 4 original images (−0.10 dB / −0.002 here vs −0.12 dB / −0.008 on 100 test images), slightly worse on the 2 new ones. Before BP it helps every start a little (LPIPS −0.011 to −0.042), but BP already gives that measurement-fit gain.
2. **On top of the noise-label search, IADrift @100 adds nothing** (LPIPS −0.010 vs −0.013 for search alone) and gives back the search's PSNR gain.
3. **Early swaps stay worse than CSGM on LPIPS** with or without IADrift @100 (2 new images: +0.076 to +0.102), confirming the Check 10b preview.
4. **Best development pipeline remains noise-label search + BP** (+0.24 dB, LPIPS −0.013 on 4 images), and that too is mixed per image.

**Summary.** The z-side methods (Checks 7–11) and the θ-side method (IADrift) each give at most small gains once back-projection is applied, and they don't stack. After BP, what remains between methods is the content above the LR limit, and neither choosing z nor 100 steps of weight adaptation changes it much on average.

---
# Check 12: one-image demos of four improvement ideas (A–D), development image 403

> Exploratory demos on **one development image** (403, crane: the image with the strongest tile and texture problems). They show *how* each idea behaves, not whether it is better in general; a winner would need more images. HR is used only for scoring.

After back-projection, methods differ only in the content above the LR limit (0.125 cycles/pixel) and in the 16-pixel tile seams. Each idea below targets one of those:
- **A** texture-strength knob (post-processing, CPU)
- **B** restart averaging (post-processing, CPU)
- **C** tile-seam penalty inside CSGM (optimization, GPU)
- **D** shift ensemble / cycle spinning (optimization, GPU)

All four start from (or are compared with) the **saved CSGM solution of image 403** (restart 0, the one kept by the lowest-LR-loss rule) and its back-projection, **CSGM + BP**, the strongest perceptual baseline (development LPIPS 0.435 on this image). Bicubic + BP is shown as the PSNR reference.

In [ ]:
# Check 12 setup: image 403, references, cache.
DEMO_I = 403
DEMO_DIR = DATA_DIR / "one_image_demos_v1"
DEMO_DIR.mkdir(exist_ok=True)
d_im = images[DEMO_I]
d_hr, d_y = d_im["hr"], d_im["y"]
d_ref = {"bicubic + BP": (d_im["outputs"]["bicubic + BP"], d_im["metrics"]["bicubic + BP"]),
         "CSGM + BP": (d_im["outputs"]["CSGM + BP"], d_im["metrics"]["CSGM + BP"])}
assert d_im["csgm_restart"] == 0


def d_eval(x):
    m = evaluate(x, d_hr, d_y)
    m["blockiness"] = blockiness(x)
    return m


for n in d_ref:
    d_ref[n][1]["blockiness"] = blockiness(d_ref[n][0])
d_title = f"Original HR, development image {DEMO_I}\n{class_name(d_im['label'])}"
print({n: (round(m["psnr"], 2), round(m["lpips"], 3), round(m["blockiness"], 3)) for n, (_, m) in d_ref.items()})

### A. Texture-strength knob (perception–distortion trade-off)

**What it does.** CSGM's output has two parts: the content **below** the LR limit (0.125 cycles/pixel), which back-projection makes consistent with the measurement, and the content **above** it, which the generator invents. Checks 1 and 4 showed the invented part is mostly *wrong in detail* (negative high-band SNR, so it costs PSNR) but *perceptually useful* (it's why CSGM + BP has the best LPIPS).

The knob scales only the invented part by a factor **α**:
- x_α = BP( low(CSGM + BP) + α · high(CSGM + BP) ), with low/high split at 0.125 cycles/pixel (`band_filter`, mirror-padded FFT), then back-projected again so the result still reproduces y exactly;
- **α = 0:** no invented detail (a smooth, bicubic-like, measurement-consistent image) → expected **high PSNR, worse LPIPS**;
- **α = 1:** plain CSGM + BP → **best LPIPS, low PSNR**.

**What we look for:** whether intermediate α gives a useful middle ground (most of CSGM's perceptual gain at a much smaller PSNR cost), i.e. the shape of the perception–distortion curve. α = 0, 0.1, …, 1 (11 values); no optimization; CPU.

In [ ]:
# A. Texture-strength knob on CSGM + BP (image 403).
ALPHAS = np.round(np.linspace(0, 1, 11), 2)
x_base = d_ref["CSGM + BP"][0]
low, high = band_filter(x_base, 0.0, LR_NYQUIST), band_filter(x_base, LR_NYQUIST, np.inf)
A_out = {}
for a in ALPHAS:
    x = backproject(low + a * high, d_y)
    A_out[a] = (x, d_eval(x))

lines = ["| α | PSNR | LPIPS ↓ | blockiness | SNR high | SNR very high | LR loss |", "|---:|---:|---:|---:|---:|---:|---:|"]
for a, (_, m) in A_out.items():
    lines.append(f"| {a:.1f} | {m['psnr']:.2f} | {m['lpips']:.3f} | {m['blockiness']:.3f} | {m['band_snr_db']['high']:.2f} | "
                 f"{m['band_snr_db']['very high']:.2f} | {m['lr_loss']:.1e} |")
for n, (_, m) in d_ref.items():
    lines.append(f"| {n} | {m['psnr']:.2f} | {m['lpips']:.3f} | {m['blockiness']:.3f} | {m['band_snr_db']['high']:.2f} | "
                 f"{m['band_snr_db']['very high']:.2f} | {m['lr_loss']:.1e} |")
display(Markdown("**A. Texture strength α applied to CSGM + BP (image 403)**\n\n" + "\n".join(lines)))

fig, ax = plt.subplots(figsize=(6.5, 4.5))
p = [m["psnr"] for _, m in A_out.values()]
l = [m["lpips"] for _, m in A_out.values()]
ax.plot(p, l, "-o", color="C0", label="CSGM + BP, texture strength α")
for a, pp, ll in zip(ALPHAS, p, l):
    ax.annotate(f"{a:.1f}", (pp, ll), textcoords="offset points", xytext=(4, 4), fontsize=8)
for n, (_, m), c in zip(d_ref, d_ref.values(), ("k", "C3")):
    ax.scatter(m["psnr"], m["lpips"], color=c, marker="*", s=150, zorder=3, label=n)
ax.set(xlabel="PSNR (dB) →  better", ylabel="LPIPS ↓  better", title="A. Perception–distortion trade-off, image 403")
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
plt.show()

side_by_side(d_hr, d_title, [("bicubic + BP", *d_ref["bicubic + BP"])] +
             [(f"α = {a:.1f}", *A_out[a]) for a in (0.0, 0.3, 0.6, 1.0)])

### B. Restart averaging (an approximate posterior mean)

**What it does.** CSGM runs from different random starts (restarts) give different plausible reconstructions; all reproduce the measurement roughly, but they invent *different* details. Averaging them keeps what they agree on (mostly the measured content) and cancels what they don't (much of the invented detail and part of the tile seams), approximating the **posterior mean**, the PSNR-optimal estimate.

Two versions on image 403, each back-projected:
- **B1:** the mean of the 2 saved CSGM restarts (r0, r1), compared with the restart kept by the usual rule (r0) and with the other restart;
- **B2:** the mean of the 2 full-length runs of the noise-label search (Check 7), whose two finalists differ in z (scene) as well as ε.

Back-projection is affine, so BP(mean) = mean(BP). **Expected:** higher PSNR and lower blockiness, but blurrier, so possibly worse LPIPS. No optimization; CPU.

In [ ]:
# B. Restart averaging (image 403).
def load_x(path, key="x_hat"):
    with np.load(path) as data:
        return data[key]


xr = [load_x(CSGM_CACHE / f"img{DEMO_I:05d}_oracle_r{r}.npz") for r in RESTARTS]
xs = [load_x(NLS_DIR / f) for f in ("final_img00403_r0_nl5.npz", "final_img00403_r1_nl4.npz")]
B_out = {"CSGM r0 + BP (kept by rule)": backproject(xr[0], d_y), "CSGM r1 + BP (other restart)": backproject(xr[1], d_y),
         "B1: mean of 2 CSGM restarts + BP": backproject((xr[0] + xr[1]) / 2, d_y),
         "search r0_nl5 + BP (kept)": backproject(xs[0], d_y), "search r1_nl4 + BP": backproject(xs[1], d_y),
         "B2: mean of 2 search finalists + BP": backproject((xs[0] + xs[1]) / 2, d_y)}
B_out = {n: (x, d_eval(x)) for n, x in B_out.items()}

lines = ["| Method | PSNR | LPIPS ↓ | blockiness | SNR mid | SNR high |", "|---|---:|---:|---:|---:|---:|"]
for n, (_, m) in list(d_ref.items())[:1] + list(B_out.items()):
    lines.append(f"| {n} | {m['psnr']:.2f} | {m['lpips']:.3f} | {m['blockiness']:.3f} | {m['band_snr_db']['mid']:.2f} | "
                 f"{m['band_snr_db']['high']:.2f} |")
display(Markdown("**B. Restart averaging (image 403), all back-projected**\n\n" + "\n".join(lines)))

side_by_side(d_hr, d_title, [(n, *B_out[n]) for n in ("CSGM r0 + BP (kept by rule)", "CSGM r1 + BP (other restart)",
                                                      "B1: mean of 2 CSGM restarts + BP", "B2: mean of 2 search finalists + BP")])

### C. Tile-seam penalty inside CSGM

**What it does.** The generator paints the image as 256 tiles of 16×16 pixels; under CSGM, neighbouring tiles disagree at their borders (the seams you see when zooming in; blockiness ≈ 1.3–1.4 vs 1.0 for real photos). This adds a penalty on those seams to the CSGM loss, so the optimizer is pushed toward ε values whose tiles join smoothly:

  L(ε) = ‖y − A f(ε)‖² + λ · S(f(ε)),  S(x) = Σ over every 16-pixel tile border (both directions) of the squared pixel jump across it.

- It uses no HR; it's a prior on what the output should look like (no seams), not information about the image.
- **Risk:** real edges that happen to lie on a tile border get penalized too.
- Run exactly as CSGM otherwise: `initialize(403, restart 0)` (the restart kept for the saved baseline), Adam lr 0.02, 5,000 steps; **λ = 0.1 and 0.3** (untuned; with these values the penalty starts at roughly 10–60 % of the measurement loss). Compared with the saved CSGM restart 0 (λ = 0), raw and + BP.
- GPU: about 75 s per run; cached in `one_image_demos_v1/`.

In [ ]:
# C. CSGM with a tile-seam penalty (image 403, restart 0).
def seam_energy(x):
    # Sum of squared pixel jumps across all 16-pixel tile borders, both directions. x: [B, 256, 256, 3].
    dv = x[:, 16::16, :, :] - x[:, 15:-1:16, :, :]
    dh = x[:, :, 16::16, :] - x[:, :, 15:-1:16, :]
    return jnp.sum(dv ** 2) + jnp.sum(dh ** 2)


def seam_loss(eps, target, label, noise_labels):
    y_, lam = target
    x = generate(eps, label, noise_labels).astype(jnp.float32)
    return jnp.sum((A_sr4(x) - y_) ** 2) + lam * seam_energy(x)


C_LAMS = (0.1, 0.3)
eps0, nl0 = initialize(DEMO_I, 0)
x_csgm_r0 = load_x(CSGM_CACHE / f"img{DEMO_I:05d}_oracle_r0.npz")
C_out = {"CSGM r0 (λ = 0, saved)": x_csgm_r0}
for lam in C_LAMS:
    m, a = run_csgm(DEMO_DIR / f"C_seam_lam{lam:g}_r0.npz", seam_loss, project_none, (d_y, jnp.float32(lam)),
                    d_im["label"], eps0, nl0)
    C_out[f"seam penalty λ = {lam:g}"] = a["x_hat"]
print("seam energy:", {n: round(float(seam_energy(jnp.asarray(x)[None])), 1) for n, x in C_out.items()})

C_res = {}
for n, x in C_out.items():
    C_res[n] = (x, d_eval(x))
    xb = backproject(x, d_y)
    C_res[n + " + BP"] = (xb, d_eval(xb))
lines = ["| Method | PSNR | LPIPS ↓ | blockiness | LR loss | SNR high |", "|---|---:|---:|---:|---:|---:|"]
for n, (_, m) in C_res.items():
    lines.append(f"| {n} | {m['psnr']:.2f} | {m['lpips']:.3f} | {m['blockiness']:.3f} | {m['lr_loss']:.2f} | {m['band_snr_db']['high']:.2f} |")
display(Markdown("**C. Tile-seam penalty in CSGM (image 403, restart 0)**\n\n" + "\n".join(lines)))

side_by_side(d_hr, d_title, [(n, *C_res[n]) for n in C_res if n.endswith("+ BP")] + [("seam penalty λ = 0.3 (raw)", *C_res["seam penalty λ = 0.3"])])

### D. Shift ensemble (cycle spinning)

**What it does.** The tile grid is fixed in output coordinates (every 16 pixels), so CSGM's seams always land in the same places relative to the image. If we shift the *measurement* by one LR pixel (= 4 HR pixels), run CSGM on that, and shift the result back, the seams land 4 pixels elsewhere on the image. Averaging the four shift combinations (0 or 1 LR pixel in each direction) spreads and averages the seams out, and also averages away part of the invented detail (like B).

- Shifts are circular (`np.roll`); since the operator isn't exactly circular at the image border, a thin strip at one edge may be slightly wrong; back-projection then makes the final average reproduce the unshifted y exactly.
- Each shifted run is CSGM exactly as frozen (`initialize(403, restart 0)`, Adam lr 0.02, 5,000 steps); the (0, 0) shift is the saved CSGM restart 0. Compared with CSGM r0 + BP and with B1 (restart averaging, same number of runs = 2).
- GPU: 3 new runs × about 75 s; cached in `one_image_demos_v1/`.

In [ ]:
# D. Shift ensemble (image 403, restart 0 init for every shift).
SHIFTS = [(0, 0), (0, 1), (1, 0), (1, 1)]  # in LR pixels (1 LR pixel = 4 HR pixels)
D_x = {}
for sy, sx in SHIFTS:
    if (sy, sx) == (0, 0):
        x = x_csgm_r0
    else:
        y_s = jnp.roll(d_y, (sy, sx), axis=(1, 2))
        m, a = run_csgm(DEMO_DIR / f"D_shift{sy}{sx}_r0.npz", measurement_loss, project_none, y_s, d_im["label"], eps0, nl0)
        x = a["x_hat"]
    D_x[(sy, sx)] = np.roll(x, (-4 * sy, -4 * sx), axis=(0, 1))  # back to the original coordinates
D_res = {"CSGM r0 + BP": d_ref["CSGM + BP"],
         "shift (0, 1) alone, shifted back + BP": (backproject(D_x[(0, 1)], d_y), None),
         "D: mean of 4 shifts + BP": (backproject(np.mean(list(D_x.values()), axis=0), d_y), None),
         "B1: mean of 2 restarts + BP (for comparison)": B_out["B1: mean of 2 CSGM restarts + BP"]}
D_res = {n: (x, m if m is not None else d_eval(x)) for n, (x, m) in D_res.items()}
lines = ["| Method | PSNR | LPIPS ↓ | blockiness | SNR mid | SNR high |", "|---|---:|---:|---:|---:|---:|"]
for n, (_, m) in D_res.items():
    lines.append(f"| {n} | {m['psnr']:.2f} | {m['lpips']:.3f} | {m['blockiness']:.3f} | {m['band_snr_db']['mid']:.2f} | "
                 f"{m['band_snr_db']['high']:.2f} |")
display(Markdown("**D. Shift ensemble (image 403)**\n\n" + "\n".join(lines)))
side_by_side(d_hr, d_title, [(n, *v) for n, v in D_res.items()])

### Check 12 notes (one development image, 403; exploratory, not evidence)

On this image **all four ideas beat CSGM + BP on both PSNR and LPIPS** after back-projection:

| Method (all + BP) | PSNR | LPIPS ↓ | blockiness | cost |
|---|---:|---:|---:|---|
| bicubic | 25.55 | 0.593 | 0.99 | — |
| CSGM (baseline) | 21.79 | 0.435 | 1.18 | 1 run |
| **A** texture strength α = 0.5 | 24.26 | 0.387 | 1.17 | free (post-processing) |
| **A** α = 0.3 | 25.02 | 0.421 | 1.14 | free |
| **B1** mean of 2 CSGM restarts | 23.50 | 0.397 | 1.12 | free (both restarts are run anyway) |
| **B2** mean of 2 search finalists | 23.81 | 0.345 | 1.18 | free given the search |
| **C** seam penalty λ = 0.3 | 24.92 | 0.408 | 0.94 | 1 run |
| **D** mean of 4 shifts | **24.47** | **0.338** | 1.09 | 4 runs |

- **A:** for α between about 0.3 and 0.9 the result is better than CSGM + BP on *both* metrics (e.g. α = 0.5: +2.5 dB and −0.048 LPIPS). The full invented detail (α = 1) is past the LPIPS optimum (α ≈ 0.5). Visually, α = 0.3–0.6 keeps the feather structure but loses most of the colour noise and seams.
- **B and D:** averaging reconstructions *improves LPIPS* here, not only PSNR. The removed parts (seams, colour-noise tiles, wrong texture) hurt LPIPS more than the slight blur does. D (4 shifted runs) is the best single result: seams spread out, structure kept.
- **C:** the seam penalty nearly removes the tile grid (blockiness 0.94, slightly *below* HR's 0.99, i.e. borders a bit over-smoothed) and gives +3.1 dB. Before BP the image is flatter and blurrier (raw λ = 0.3: LPIPS 0.572); BP restores the measured detail.
- **Common mechanism:** all four reduce the generator's *wrong* high-frequency content (seams, confetti, painted texture) while keeping its plausible structure. That's what PSNR penalized and what LPIPS partly penalized too.

**Caveats:** one image, and the one where CSGM is worst (most room for improvement); untuned settings (α grid, λ = 0.1/0.3).

### E. Combinations (quick look, image 403)

**Why.** A–D fix the wrong high-frequency content in different ways: **A** shrinks all of it uniformly, **B/D** remove what differs between runs (D also moves the tile seams), **C** prevents seams during each run. Complementary pairs may add up; overlapping ones (B + D, A + C) probably won't.

**What exactly is combined** (same image, same starts; each result back-projected):
- **C + D:** the 4-shift ensemble of D, but every shifted run uses the seam penalty of C (λ = 0.3). The (0, 0) run is C's existing λ = 0.3 run; 3 new GPU runs (about 75 s each).
- **C + B:** the mean of two seam-penalty runs (λ = 0.3) from restarts 0 and 1 (1 new run).
- **A on top:** the texture-strength knob (α = 0, 0.1, …, 1) applied to each base: B2, C, D, C + D, C + B. For averaged bases the best α is expected to be higher than for plain CSGM (0.5), because averaging already removed much of the wrong detail.

**Caveat.** Every setting here (λ, α, number of runs) is chosen on this one image, so a combination that looks best here is a candidate for checking on more images, not a result.

In [ ]:
# E. Combinations on image 403.
LAM_E = 0.3
CD_x = {}
for sy, sx in SHIFTS:
    if (sy, sx) == (0, 0):
        x = C_out[f"seam penalty λ = {LAM_E:g}"]
    else:
        y_s = jnp.roll(d_y, (sy, sx), axis=(1, 2))
        m, a = run_csgm(DEMO_DIR / f"CD_seam{LAM_E:g}_shift{sy}{sx}_r0.npz", seam_loss, project_none, (y_s, jnp.float32(LAM_E)),
                        d_im["label"], eps0, nl0)
        x = a["x_hat"]
    CD_x[(sy, sx)] = np.roll(x, (-4 * sy, -4 * sx), axis=(0, 1))
eps1, nl1 = initialize(DEMO_I, 1)
m, a = run_csgm(DEMO_DIR / f"C_seam_lam{LAM_E:g}_r1.npz", seam_loss, project_none, (d_y, jnp.float32(LAM_E)), d_im["label"], eps1, nl1)
x_c_r1 = a["x_hat"]

bases = {"CSGM + BP": d_ref["CSGM + BP"][0],
         "B2: mean of 2 search finalists + BP": B_out["B2: mean of 2 search finalists + BP"][0],
         "C: seam λ = 0.3 + BP": C_res["seam penalty λ = 0.3 + BP"][0],
         "D: mean of 4 shifts + BP": D_res["D: mean of 4 shifts + BP"][0],
         "C + D: mean of 4 seam-penalty shifts + BP": backproject(np.mean(list(CD_x.values()), axis=0), d_y),
         "C + B: mean of 2 seam-penalty restarts + BP": backproject((C_out[f"seam penalty λ = {LAM_E:g}"] + x_c_r1) / 2, d_y)}

E_curves = {}
for n, xb in bases.items():
    lo_, hi_ = band_filter(xb, 0.0, LR_NYQUIST), band_filter(xb, LR_NYQUIST, np.inf)
    E_curves[n] = {a: d_eval(backproject(lo_ + a * hi_, d_y)) for a in ALPHAS}

lines = ["| Base (all + BP) | PSNR (α = 1) | LPIPS (α = 1) | blockiness (α = 1) | best-LPIPS α | PSNR there | LPIPS there |",
         "|---|---:|---:|---:|---:|---:|---:|"]
for n, cur in E_curves.items():
    a_best = min(cur, key=lambda a: (cur[a]["lpips"], -a))
    m1, mb = cur[1.0], cur[a_best]
    lines.append(f"| {n} | {m1['psnr']:.2f} | {m1['lpips']:.3f} | {m1['blockiness']:.3f} | {a_best:.1f} | {mb['psnr']:.2f} | {mb['lpips']:.3f} |")
lines.append(f"| bicubic + BP (reference) | {d_ref['bicubic + BP'][1]['psnr']:.2f} | {d_ref['bicubic + BP'][1]['lpips']:.3f} | "
             f"{d_ref['bicubic + BP'][1]['blockiness']:.3f} | — | — | — |")
display(Markdown("**E. Combinations (image 403): each base as is (α = 1) and at its best-LPIPS texture strength α**\n\n" + "\n".join(lines)))

fig, ax = plt.subplots(figsize=(8, 5.5))
for c_, (n, cur) in enumerate(E_curves.items()):
    p, l = [cur[a]["psnr"] for a in ALPHAS], [cur[a]["lpips"] for a in ALPHAS]
    ax.plot(p, l, "-o", color=f"C{c_}", markersize=3, label=n)
    ax.scatter(p[-1], l[-1], color=f"C{c_}", s=60, edgecolor="k", zorder=3)
ax.scatter(d_ref["bicubic + BP"][1]["psnr"], d_ref["bicubic + BP"][1]["lpips"], color="k", marker="*", s=150, label="bicubic + BP")
ax.set(xlabel="PSNR (dB) →  better", ylabel="LPIPS ↓  better",
       title="E. Perception–distortion curves (α from 0 to 1; big dot = α = 1), image 403")
ax.grid(alpha=0.3)
ax.legend(fontsize=7)
plt.show()

side_by_side(d_hr, d_title, [("D: 4 shifts + BP", bases["D: mean of 4 shifts + BP"], E_curves["D: mean of 4 shifts + BP"][1.0]),
                             ("C + D: 4 seam shifts + BP", bases["C + D: mean of 4 seam-penalty shifts + BP"],
                              E_curves["C + D: mean of 4 seam-penalty shifts + BP"][1.0]),
                             ("C + B: 2 seam restarts + BP", bases["C + B: mean of 2 seam-penalty restarts + BP"],
                              E_curves["C + B: mean of 2 seam-penalty restarts + BP"][1.0]),
                             ("B2: 2 search finalists + BP", bases["B2: mean of 2 search finalists + BP"],
                              E_curves["B2: mean of 2 search finalists + BP"][1.0])])

### E. Combinations: notes (image 403; quick look)

**The combinations don't add up; the single best method stays D (the 4-shift ensemble).**

| (all + BP) | PSNR | LPIPS ↓ | blockiness |
|---|---:|---:|---:|
| CSGM | 21.79 | 0.435 | 1.18 |
| **D: 4 shifts** | 24.47 | **0.338** | 1.09 |
| B2: 2 search finalists (best α = 0.8) | 24.37 | 0.339 | — |
| C + D: 4 seam-penalty shifts | 25.33 | 0.452 | 1.01 |
| C + B: 2 seam-penalty restarts | 25.30 | 0.429 | 0.96 |
| bicubic | 25.55 | 0.593 | 0.99 |

- **C + averaging over-smooths.** The seam penalty already removes much of the invented detail; averaging removes more, so C + D and C + B move toward bicubic: PSNR almost reaches bicubic (25.3 vs 25.6) but LPIPS gets *worse* than C or D alone (0.43–0.45). The crops show it: C + D/C + B are smooth and blurry, D keeps more feather structure.
- **A on top of an averaged base helps little or not at all.** The best α is 1.0 for D, C, C + D and C + B (no shrinking wanted: the averaging already did it), and 0.8 for B2 (LPIPS 0.345 → 0.339, a tiny gain).
- So the four ideas are mostly **substitutes, not complements**: each is a way of reducing the generator's wrong high-frequency content, and once one has done it, adding another overshoots into blur.

**Is there justification to try more images?** For the *single* methods, yes, especially **D** (best LPIPS here) and the free **A** and **B** (cached outputs, no runs). For the combinations, no: none beats D alone on this image.

---
# Check 13: posterior-mean ensemble of CSGM (6 development images)

> Exploratory, development only (the ensemble idea came from Checks 7–12).

**Idea (from all checks together).** After back-projection, every method differs only in the content above the LR limit, where the generator invents detail that is mostly wrong in place and adds 16-pixel tile seams (Checks 1, 3, 12). Methods that *reduce* that wrong content helped; methods that add freedom hurt. Averaging several plausible CSGM reconstructions keeps what they agree on and cancels what they don't: an approximate **posterior mean** over the generator's latent (ε, z), followed by BP so the measured bands are exact:

  x̂ = BP( Σₖ wₖ · xₖ ),  xₖ = CSGM reconstructions with different starts / noise labels / shifts.

**What exactly was done.** Images: the 4 development images of Checks 1–12 + the 2 fresh ones of Check 10b (9780, 8405). Correct label; HR used only for scoring. Members (CSGM as frozen: Adam lr 0.02, 5,000 steps):
- **2 restarts:** the saved CSGM runs r0, r1 (`initialize(image, r)`);
- **3 shifts:** CSGM on the measurement shifted by (0, 1), (1, 0), (1, 1) LR pixels from the r0 start, shifted back by 4 HR pixels per LR pixel (as Check 12 D; new GPU runs, about 75 s each; image 403 reuses Check 12's);
- **2 search finalists** (4 original images only): the two full-length runs of the Check 7 noise-label search.

Ensembles (all + BP):
- **E1 = mean of the 2 restarts** (free: the frozen pipeline already runs both and keeps one);
- **E2 = mean of the 4 runs from the r0 start** (r0 + 3 shifts; Check 12's D);
- **Ens-U = mean of all CSGM runs** (2 restarts + 3 shifts = 5);
- **E4 = Ens-U + 2 search finalists** (7 runs; 4 original images);
- **Ens-W / E4w:** the same with weights wₖ ∝ exp(−(Lₖ − L_min)/T), Lₖ = eager LR loss of member k (in original coordinates), T = standard deviation of the members' losses (untuned heuristic; down-weights clearly worse fits).

Reference: **CSGM + BP** with the usual restart rule (lowest LR loss), i.e. the method frozen for the test set.

In [ ]:
# Check 13: posterior-mean ensemble of CSGM runs (development images, correct label).
ENS_DIR = DATA_DIR / "ensemble_dev_v1"
ENS_DIR.mkdir(exist_ok=True)
IMGS13 = [8117, 3939, 403, 347, 9780, 8405]
CONFIRM_DIR = DATA_DIR / "z_swap_dev_v1" / "confirm"
SHIFTS3 = [(0, 1), (1, 0), (1, 1)]


def csgm_restart_file(i, r):
    return CSGM_CACHE / f"img{i:05d}_oracle_r{r}.npz" if i in images else CONFIRM_DIR / f"img{i:05d}_csgm_r{r}.npz"


e13 = {}
for i in IMGS13:
    lbl = images[i]["label"] if i in images else int(labels[i])
    hr = images[i]["hr"] if i in images else load_hr256(DATA_DIR / index_rows[i]["relative_path"])
    y = images[i]["y"] if i in images else jnp.asarray(lr_all[i:i + 1], jnp.float32)
    members = {}
    for r in RESTARTS:
        members[f"restart r{r}"] = load_x(csgm_restart_file(i, r))
    e0, n0 = initialize(i, 0)
    for sy, sx in SHIFTS3:
        path = (DEMO_DIR / f"D_shift{sy}{sx}_r0.npz") if i == DEMO_I else (ENS_DIR / f"img{i:05d}_shift{sy}{sx}_r0.npz")
        m, a = run_csgm(path, measurement_loss, project_none, jnp.roll(y, (sy, sx), axis=(1, 2)), lbl, e0, n0)
        members[f"shift ({sy}, {sx})"] = np.roll(a["x_hat"], (-4 * sy, -4 * sx), axis=(0, 1))
    if i in images:
        s = json.loads((NLS_DIR / "summary.json").read_text())[str(i)]
        for f in sorted(NLS_DIR.glob(f"final_img{i:05d}_*.npz")):
            members[f"search {f.stem.split('_', 2)[2]}"] = load_x(f)
    loss = {n: lr_loss(x, y) for n, x in members.items()}
    kept = min(RESTARTS, key=lambda r: (loss[f"restart r{r}"], r))

    def ens(names, weighted=False):
        L = np.array([loss[n] for n in names])
        w = np.exp(-(L - L.min()) / max(L.std(), 1e-6)) if weighted else np.ones(len(L))
        w = w / w.sum()
        return backproject(sum(wk * members[n] for wk, n in zip(w, names)), y), w

    R = [f"restart r{r}" for r in RESTARTS]
    SH = ["restart r0"] + [f"shift ({sy}, {sx})" for sy, sx in SHIFTS3]
    ALL = R + SH[1:]
    SE = [n for n in members if n.startswith("search")]
    outs = {"CSGM + BP (restart rule)": (backproject(members[f"restart r{kept}"], y), None),
            "E1: mean of 2 restarts": ens(R), "E2: mean of r0 + 3 shifts": ens(SH),
            "Ens-U: mean of 5 CSGM runs": ens(ALL), "Ens-W: weighted 5 CSGM runs": ens(ALL, True)}
    if SE:
        outs["E4: Ens-U + 2 search finalists"] = ens(ALL + SE)
        outs["E4w: weighted E4"] = ens(ALL + SE, True)
    e13[i] = {"label": lbl, "hr": hr, "loss": loss, "kept": kept,
              "res": {n: (x, None, w) for n, (x, w) in outs.items()}}
    for n, (x, _, w) in list(e13[i]["res"].items()):
        mm = evaluate(x, hr, y)
        mm["blockiness"] = blockiness(x)
        e13[i]["res"][n] = (x, mm, w)
    print(f"image {i}: members {list(members)}; LR losses " + ", ".join(f"{v:.0f}" for v in loss.values()) + f"; rule keeps r{kept}")

METH13 = ["CSGM + BP (restart rule)", "E1: mean of 2 restarts", "E2: mean of r0 + 3 shifts", "Ens-U: mean of 5 CSGM runs",
          "Ens-W: weighted 5 CSGM runs", "E4: Ens-U + 2 search finalists", "E4w: weighted E4"]
for subset, title in ((IMGS13, "all 6 development images"), (IMGS13[:4], "4 original development images (with search members)")):
    ref = [e13[i]["res"][METH13[0]][1] for i in subset]
    lines = ["| Ensemble (all + BP) | runs | PSNR | LPIPS ↓ | blockiness | Δ PSNR vs CSGM + BP | better | Δ LPIPS vs CSGM + BP | better |",
             "|---|---:|---:|---:|---:|---:|---:|---:|---:|"]
    for n in METH13:
        if not all(n in e13[i]["res"] for i in subset):
            continue
        ms = [e13[i]["res"][n][1] for i in subset]
        p, l = np.array([m["psnr"] for m in ms]), np.array([m["lpips"] for m in ms])
        rp, rl = np.array([m["psnr"] for m in ref]), np.array([m["lpips"] for m in ref])
        runs = {"CSGM": 1, "E1": 2, "E2": 4, "Ens-U:": 5, "Ens-W": 5, "E4:": 7, "E4w": 7}
        nr = next(v for kk, v in runs.items() if n.startswith(kk))
        lines.append(f"| {n} | {nr} | {p.mean():.2f} | {l.mean():.3f} | {np.mean([m['blockiness'] for m in ms]):.3f} | "
                     f"{np.mean(p - rp):+.2f} | {int(np.sum(p > rp + 1e-9))}/{len(subset)} | {np.mean(l - rl):+.3f} | "
                     f"{int(np.sum(l < rl - 1e-9))}/{len(subset)} |")
    bb = [images[i]["metrics"]["bicubic + BP"] if i in images else None for i in subset]
    display(Markdown(f"**Check 13, {title}, mean** (runs = CSGM runs used; reference CSGM + BP with the restart rule = 1 kept run of 2 computed)\n\n"
                     + "\n".join(lines)))

lines = ["| Image | " + " | ".join(n.split(":")[0] for n in METH13) + " |", "|---|" + "---:|" * len(METH13)]
for i in IMGS13:
    lines.append(f"| {i} | " + " | ".join(f"{e13[i]['res'][n][1]['psnr']:.2f} / {e13[i]['res'][n][1]['lpips']:.3f}"
                                         if n in e13[i]["res"] else "—" for n in METH13) + " |")
display(Markdown("**Per image (all + BP): PSNR dB / LPIPS**\n\n" + "\n".join(lines)))

for i in IMGS13:
    names = ["CSGM + BP (restart rule)", "Ens-U: mean of 5 CSGM runs", "Ens-W: weighted 5 CSGM runs"] + \
            ([n for n in ("E4: Ens-U + 2 search finalists", "E4w: weighted E4") if n in e13[i]["res"]])
    side_by_side(e13[i]["hr"], f"Original HR, development image {i}\n{class_name(e13[i]['label'])}",
                 [(n, e13[i]["res"][n][0], e13[i]["res"][n][1]) for n in names])

### Check 13 conclusions (6 development images; exploratory; small sample)

**1. Averaging CSGM runs raises PSNR on every image, by a lot.** Mean over 6 images, all + BP, vs CSGM + BP with the restart rule (22.45 dB / LPIPS 0.303):

| Ensemble | runs | Δ PSNR | Δ LPIPS | LPIPS better |
|---|---:|---:|---:|---:|
| E1: 2 restarts | 2 | +1.27 (6/6) | +0.011 | 3/6 |
| E2: r0 + 3 shifts | 4 | +2.10 (6/6) | +0.015 | 4/6 |
| Ens-U: 5 CSGM runs | 5 | **+2.36** (6/6) | −0.001 | 4/6 |
| **Ens-W: weighted 5 runs** | 5 | +1.60 (6/6) | **−0.029** | **5/6** |

On the 4 original images (with the 2 search finalists as extra members): **E4w** gives +1.98 dB and **LPIPS −0.072** (0.255 vs 0.328), the best LPIPS of any development method so far; E4 (unweighted) +2.67 dB / −0.040.

**2. Plain averaging can hurt LPIPS through ghosting.** When one member fits much worse than the others (e.g. 347: LR loss 66 vs 17; 9780: 141 vs 28), it brings a different scene, and the mean shows doubled outlines and ripple patterns (347: LPIPS 0.203 → 0.44 for Ens-U). **Weighting by measurement fit removes that** (347 Ens-W 0.266, E4w 0.220; 9780 Ens-W 0.202, better than CSGM's 0.213), at the cost of part of the PSNR gain (fewer effective members).

**3. Why it works.** It's the mechanism of Checks 1, 4 and 12: the members agree on the measured, real structure and disagree on the invented detail (wrong in place) and on where the tile seams fall, so the mean keeps the first and cancels the second; BP then makes the measured bands exact. The search members (different z, i.e. different scenes, but with good fits) add useful diversity: E4w > Ens-W on the 4 original images.

**4. Cost and what's free.** E1 is free in the frozen pipeline (both restarts are already computed); it gains +1.27 dB on every image, LPIPS mixed. Ens-U/Ens-W need 3 extra shifted runs (4× → 2.5× of the frozen cost). Weights use only LR losses (no HR).

**5. Caveats.** 6 images (4 used to design everything); the weighting temperature T (std of member losses) is an untuned heuristic; restart averaging is the only variant directly applicable to the existing test arrays.

**Takeaway.** An LR-loss-weighted posterior-mean ensemble of CSGM runs (+ BP) is the first method in this project that improves on CSGM + BP in **both** PSNR and LPIPS across several images (Ens-W: 6/6 PSNR, 5/6 LPIPS). It's a strong candidate to confirm on more development images, then (post-hoc, with a protocol addendum) on the test set.

---
# Check 14: posterior-mean ensemble of IADrift (6 development images)

> Exploratory, development only (IADrift @100 was singled out post-hoc, protocol Section 9).

**Why.** IADrift's failure is wrong high-frequency content: it fits y almost exactly but fills the unmeasured frequencies with its own noise (test set: very-high-band SNR −8 dB at step 1,000; colour speckle in the crops). Check 13 showed that averaging CSGM runs cancels exactly that kind of run-specific detail. If IADrift runs from different starts produce *different* noise, averaging them should cancel much of it, possibly rescuing even IADrift @1000, and keep IADrift @100's small perceptual gain.

**What exactly was done** (`scripts/run_iadrift_ensemble_dev.py`, `platform` allocator for θ-gradients; cache `~/work/datasets/ImageNetV2/iadrift_ensemble_dev_v1/`). Same 6 development images as Check 13, correct label, HR only for scoring.
- **Members:** IADrift started from **each of Check 13's 5 CSGM members** (restarts r0, r1 and the 3 shifted r0-start runs), exactly as frozen: joint θ + ε, Adam, θ lr 1e-5 on all leaves, ε lr 1e-3, fresh pretrained θ per run, against that member's measurement (the shifted y for shift members). Outputs at **steps 100 and 1,000** (eager path). Shift members are shifted back afterwards.
- **For each checkpoint (100, 1,000):**
  - **single:** the member started from the restart kept by the CSGM rule (= IADrift as on the test set) + BP;
  - **ensemble:** uniform mean of the 5 members + BP;
  - **weighted ensemble:** wₖ ∝ exp(−(Lₖ − L_min)/T), with Lₖ the LR loss of the member's **CSGM start** (Check 13's losses; T = their standard deviation). IADrift's own final LR losses (about 1–5) are below the bf16 noise floor and can't rank members, while the CSGM start loss flags a wrong scene.
- **References:** CSGM + BP (restart rule), and Check 13's Ens-U / Ens-W (the same 5 CSGM members, uniform / weighted).
- **Compute per image:** 5 IADrift runs of 1,000 steps on top of the 5 CSGM runs (25,000 ε steps + 5,000 joint steps).

In [ ]:
# Check 14: ensembles of IADrift runs started from every Check 13 CSGM member.
IAD_ENS = DATA_DIR / "iadrift_ensemble_dev_v1"
MEMB14 = {"r0": "restart r0", "r1": "restart r1", "shift01": "shift (0, 1)", "shift10": "shift (1, 0)", "shift11": "shift (1, 1)"}
SH14 = {"r0": (0, 0), "r1": (0, 0), "shift01": (0, 1), "shift10": (1, 0), "shift11": (1, 1)}
e14 = {}
for i in IMGS13:
    hr, lbl = e13[i]["hr"], e13[i]["label"]
    y = images[i]["y"] if i in images else jnp.asarray(lr_all[i:i + 1], jnp.float32)
    xs, info = {100: {}, 1000: {}}, {}
    for mname, (sy, sx) in SH14.items():
        with np.load(IAD_ENS / f"img{i:05d}_{mname}.npz") as data:
            info[mname] = json.loads(str(data["metrics_json"]))
            for ck in (100, 1000):
                xs[ck][mname] = np.roll(data[f"x_step{ck}"], (-4 * sy, -4 * sx), axis=(0, 1))
    assert all(info[m][f"step{ck}"]["finite"] for m in info for ck in (100, 1000))
    L = np.array([e13[i]["loss"][MEMB14[m]] for m in SH14])  # LR loss of each member's CSGM start
    w = np.exp(-(L - L.min()) / max(L.std(), 1e-6))
    w = w / w.sum()
    kept = f"r{e13[i]['kept']}"
    res = {"CSGM + BP (restart rule)": e13[i]["res"]["CSGM + BP (restart rule)"][:2],
           "CSGM Ens-U: mean of 5 + BP": e13[i]["res"]["Ens-U: mean of 5 CSGM runs"][:2],
           "CSGM Ens-W: weighted 5 + BP": e13[i]["res"]["Ens-W: weighted 5 CSGM runs"][:2]}
    for ck in (100, 1000):
        for nm, x in ((f"IADrift @{ck} single + BP", xs[ck][kept]),
                      (f"IADrift @{ck} ensemble of 5 + BP", np.mean(list(xs[ck].values()), axis=0)),
                      (f"IADrift @{ck} weighted ensemble + BP", sum(wk * xs[ck][m] for wk, m in zip(w, SH14)))):
            xb = backproject(x, y)
            mm = evaluate(xb, hr, y)
            mm["blockiness"] = blockiness(xb)
            res[nm] = (xb, mm)
    e14[i] = {"res": res, "w": dict(zip(SH14, np.round(w, 3))), "hr": hr, "label": lbl}
    print(f"image {i}: CSGM-start weights {e14[i]['w']}; kept start {kept}")

METH14 = list(e14[IMGS13[0]]["res"])
ref = [e14[i]["res"]["CSGM + BP (restart rule)"][1] for i in IMGS13]
rp, rl = np.array([m["psnr"] for m in ref]), np.array([m["lpips"] for m in ref])
lines = ["| Method (all + BP) | PSNR | LPIPS ↓ | blockiness | SNR high | SNR very high | Δ PSNR vs CSGM + BP | better | Δ LPIPS vs CSGM + BP | better |",
         "|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|"]
for n in METH14:
    ms = [e14[i]["res"][n][1] for i in IMGS13]
    p, l = np.array([m["psnr"] for m in ms]), np.array([m["lpips"] for m in ms])
    lines.append(f"| {n} | {p.mean():.2f} | {l.mean():.3f} | {np.mean([m['blockiness'] for m in ms]):.3f} | "
                 f"{np.mean([m['band_snr_db']['high'] for m in ms]):.2f} | {np.mean([m['band_snr_db']['very high'] for m in ms]):.2f} | "
                 f"{np.mean(p - rp):+.2f} | {int(np.sum(p > rp + 1e-9))}/6 | {np.mean(l - rl):+.3f} | {int(np.sum(l < rl - 1e-9))}/6 |")
display(Markdown("**Check 14, mean over 6 development images** (reference: CSGM + BP with the restart rule)\n\n" + "\n".join(lines)))

lines = ["| Image | " + " | ".join(METH14) + " |", "|---|" + "---:|" * len(METH14)]
for i in IMGS13:
    lines.append(f"| {i} | " + " | ".join(f"{e14[i]['res'][n][1]['psnr']:.2f} / {e14[i]['res'][n][1]['lpips']:.3f}" for n in METH14) + " |")
display(Markdown("**Per image (all + BP): PSNR dB / LPIPS**\n\n" + "\n".join(lines)))

SHOW14 = ["CSGM + BP (restart rule)", "CSGM Ens-W: weighted 5 + BP", "IADrift @100 weighted ensemble + BP",
          "IADrift @1000 single + BP", "IADrift @1000 ensemble of 5 + BP", "IADrift @1000 weighted ensemble + BP"]
for i in IMGS13:
    side_by_side(e14[i]["hr"], f"Original HR, development image {i}\n{class_name(e14[i]['label'])}",
                 [(n.replace(" + BP", "").replace("weighted ensemble", "w-ensemble") + " + BP", *e14[i]["res"][n]) for n in SHOW14])

### Check 14 conclusions (6 development images; exploratory)

**1. Averaging rescues IADrift @1000.** A single IADrift @1000 + BP is far worse than CSGM + BP (−3.40 dB, LPIPS +0.118, 0/6); the uniform mean of the 5 runs + BP brings it to **+0.55 dB** (4/6) and LPIPS +0.015 (4/6 better): +3.95 dB and −0.103 LPIPS from averaging alone. The very-high-band SNR goes from −9.2 to −4.9 dB: the 5 runs' high-frequency noise is largely independent, so it cancels. Visually (e.g. 403) the dense colour speckle of a single run becomes a much cleaner image. Weighting hurts here (21.65 dB): it concentrates on 1–2 members (weights up to 0.73), so less noise is averaged out.

**2. At 100 steps, IADrift adds nothing on top of the CSGM ensemble.**

| (all + BP, mean of 6) | PSNR | LPIPS ↓ |
|---|---:|---:|
| CSGM + BP | 22.45 | 0.303 |
| CSGM Ens-U (mean of 5) | **24.81** | 0.302 |
| IADrift @100 ensemble of 5 | 24.71 | 0.293 |
| CSGM Ens-W (weighted 5) | 24.04 | **0.273** |
| IADrift @100 weighted ensemble | 23.82 | **0.273** |

IADrift @100 ensembles are within ±0.1–0.2 dB and ±0.01 LPIPS of the corresponding CSGM ensembles (the same members without adaptation). The small perceptual gain of a single IADrift @100 run (test set: −0.008 LPIPS) disappears once runs are averaged.

**3. The ensemble is the ingredient that matters, not the adaptation.** Averaging over members does what IADrift's weight adaptation was hoped to do: remove the generator's wrong high-frequency content. With averaging in place, adapting θ for 100 steps is neutral and for 1,000 steps still harmful (IADrift @1000 ensemble: LPIPS 0.318 vs CSGM Ens-U's 0.302).

**4. Best development pipelines so far:** **CSGM Ens-U + BP** for PSNR (+2.36 dB, 6/6) and **CSGM Ens-W + BP** (or the equivalent IADrift @100 weighted ensemble) for LPIPS (−0.029, 5/6), both needing 5 CSGM runs per image. Caveat as in Check 13: 6 images, untuned weighting.

---
# Check 15: feeding an image instead of noise ε (no optimization)

> Exploratory look at what the generator does when its noise input ε is replaced by (a mixture with) an actual image. Development images only; forward passes only.

**Why.** CSGM spends 5,000 steps searching for an ε that makes the generator reproduce the measurement. ε enters the network patch by patch (one 16×16 patch → one token, Figure 1), so an image placed in ε might steer each output tile toward the content at that location, like SDEdit-style "noise + image" starts in diffusion models. If so, it could be a much better starting point for CSGM / IADrift than random noise.

**What exactly is done.** For 3 development images (403 crane, 3939 bonnet, 8405 thimble), correct label, the noise labels z of CSGM restart 0 (`initialize(image, 0)`), CFG 1.0, one forward pass f(ε_in, z, c):
- **Input images:** (i) the **bicubic upsampling of y** (legitimately available at test time) and (ii) the **HR photo** itself (an oracle diagnostic: never available in practice).
- **Scaling:** the generator was trained with ε ~ N(0, 1) per pixel, while images live in [−1, 1] with much smaller spread and strong spatial correlation. So each image is **standardized** per colour channel (zero mean, unit standard deviation), giving s.
- **Mixtures:** ε_in = α·s + √(1 − α²)·n with fresh Gaussian noise n (the same n for all α), for **α = 0 (pure noise), 0.25, 0.5, 0.75, 1 (image only)**; √(1 − α²) keeps the per-pixel variance at 1. Plus α = 1 with the **raw, unstandardized** image, to see the effect of the scale.
- Each output is scored against HR (PSNR, LPIPS) and against the measurement (LR loss ‖y − A x‖²); no back-projection, no optimization.

In [ ]:
# Check 15: generator outputs with an image (or image + noise) in place of eps.
IMGS15 = [403, 3939, 8405]
ALPHA15 = [0.0, 0.25, 0.5, 0.75, 1.0]


def std_img(x):
    x = np.asarray(x, np.float32)
    return (x - x.mean(axis=(0, 1))) / x.std(axis=(0, 1))


g15 = {}
for i in IMGS15:
    lbl = images[i]["label"] if i in images else int(labels[i])
    hr = images[i]["hr"] if i in images else load_hr256(DATA_DIR / index_rows[i]["relative_path"])
    y = images[i]["y"] if i in images else jnp.asarray(lr_all[i:i + 1], jnp.float32)
    bic = bicubic_up(np.asarray(y[0]))
    _, z0 = initialize(i, 0)
    n = np.asarray(jax.random.normal(jax.random.PRNGKey(2040 + i), (256, 256, 3)), np.float32)
    rows = {}
    for src_name, img in (("bicubic(y)", bic), ("HR (oracle)", hr)):
        s = std_img(img)
        outs = []
        for a in ALPHA15:
            e = a * s + np.sqrt(1 - a ** 2) * n
            x = np.asarray(generate(jnp.asarray(e)[None], lbl, z0).astype(jnp.float32))[0]
            outs.append((f"{src_name}, α = {a:g}", x, evaluate(x, hr, y)))
        x = np.asarray(generate(jnp.asarray(np.asarray(img, np.float32))[None], lbl, z0).astype(jnp.float32))[0]
        outs.append((f"{src_name}, raw (unstandardized)", x, evaluate(x, hr, y)))
        rows[src_name] = (img, outs)
    g15[i] = {"hr": hr, "label": lbl, "rows": rows,
              "csgm": images[i]["metrics"]["CSGM"] if i in images else None}

lines = ["| Image | input | " + " | ".join(f"α = {a:g}" for a in ALPHA15) + " | raw |", "|---|---|" + "---:|" * (len(ALPHA15) + 1)]
for i in IMGS15:
    for src_name, (_, outs) in g15[i]["rows"].items():
        lines.append(f"| {i} | {src_name} | " + " | ".join(f"{m['psnr']:.1f} / {m['lpips']:.2f} / {m['lr_loss']:.0f}" for _, _, m in outs) + " |")
display(Markdown("**PSNR dB / LPIPS / LR loss of one forward pass** (no optimization, no BP). For reference, optimized CSGM "
                 "(5,000 steps): 403: 20.63 / 0.573 / 80; 3939: 22.27 / 0.388 / 35 (raw output)\n\n" + "\n".join(lines)))

for i in IMGS15:
    for src_name, (img, outs) in g15[i]["rows"].items():
        fig, axes = plt.subplots(1, len(outs) + 2, figsize=(3.0 * (len(outs) + 2), 3.6))
        axes[0].imshow(to_img(g15[i]["hr"]))
        axes[0].set_title(f"HR, image {i}\n{class_name(g15[i]['label'])}", fontsize=8)
        axes[1].imshow(to_img(img))
        axes[1].set_title(f"input image: {src_name}", fontsize=8)
        for ax, (t, x, m) in zip(axes[2:], outs):
            ax.imshow(to_img(x))
            ax.set_title(f"{t.split(', ', 1)[1]}\nPSNR {m['psnr']:.1f} | LPIPS {m['lpips']:.2f}\nLR loss {m['lr_loss']:.0f}", fontsize=7.5)
        for ax in axes:
            ax.axis("off")
        fig.suptitle(f"Generator output with ε = α·(standardized {src_name}) + √(1−α²)·noise (image {i})", fontsize=9)
        fig.subplots_adjust(left=0.01, right=0.99, bottom=0.01, top=0.78, wspace=0.04)
        plt.show()

### Check 15 notes (3 development images; one forward pass each)

**The generator essentially ignores an image placed in ε.** Whatever goes into ε (pure noise, the bicubic upsampling of y, or even the true HR photo, standardized or raw, mixed with noise at any α), the output is **a different scene chosen by the class and the noise labels z**:
- 3939 (doll in a bonnet): every input gives the same kind of scene, a girl in a pink bonnet outdoors, for all α and for both the bicubic and the **HR** input.
- 403 (crowned crane on a pink background): every input gives a grey heron-like bird by a fence on grass; with α = 1 the background changes (grass → planks) but the scene stays unrelated to the photo.
- One forward pass scores 4–11 dB PSNR and LPIPS 0.72–0.97 (vs about 20–22 dB / 0.39–0.57 for optimized CSGM), with LR losses in the thousands. The HR input is **not better** than the bicubic one (e.g. 3939: 7.5 vs 7.3 dB at α = 1), and on 8405 the image inputs are even worse than pure noise (4.1 vs 6.9 dB).

**Interpretation.** This is the strongest evidence yet for Check 8's finding: **z (with the class) decides the scene; ε only perturbs details.** The image's structure in ε is not translated into image content; the network treats ε as unstructured noise. Two consequences:
1. Starting CSGM / IADrift from an image-shaped ε (an SDEdit-like start) is **not** a useful shortcut with this generator.
2. It explains why CSGM has to push ε so far off-distribution (Check 3b: per-patch norms up to 4.8×): to reproduce a photo whose scene differs from what z selects, the optimizer must force a weak input to do the work of the strong one. Choosing z well (noise-label search, Check 7) and averaging over z / ε (Checks 13–14) are the levers that matter.

---
# Check 16: two high-frequency constraints, on single CSGM + BP and on the best ensemble (image 403)

> Exploratory, **one development image** (403, crane). Shows how each constraint changes (i) plain CSGM + BP and (ii) the best method so far, the weighted 5-run ensemble Ens-W + BP (Check 13). Settings untuned.

**Why.** The measurement constrains nothing above 0.125 cycles/pixel, and that's where all our artifacts are (16-pixel seams, colour "confetti" tiles, painted texture, IADrift's speckle). Two constraints target that band more selectively than shrinking it (Check 12):

**Idea A: penalize high-frequency colour.** In real photos fine detail is almost all brightness; colour varies slowly. The confetti tiles and much of the speckle are saturated colour noise. So CSGM gets an extra term
  L(ε) = ‖y − A f(ε)‖² + λ_A · ‖HP(chroma(f(ε)))‖²,
chroma = the Cb/Cr channels of YCbCr, HP = keep radial frequencies ≥ 0.125 cycles/pixel (FFT mask). Brightness texture is not penalized, so it shouldn't blur like Check 12's constraints. λ_A = 0.1 (at the CSGM solution the colour term is then about as large as the LR loss: 0.1 × 650 vs 80); 0.3 as a stronger single-run check.

**Idea B: frequency anchor for IADrift.** IADrift's weight adaptation fixes the measured bands well but paints new noise above the LR limit. The anchor lets θ adapt but penalizes changes to the image's high band relative to the start image x₀ (the CSGM output it starts from):
  L(θ, ε) = ‖y − A f_θ(ε)‖² + λ_B · ‖HP(f_θ(ε) − x₀)‖²,  λ_B = 0.02
(the unanchored run's high-band drift grows to about 10,800 at step 1,000; with λ_B = 0.02 that would cost about 220, i.e. more than the starting LR loss, so the anchor binds). IADrift otherwise as frozen (θ lr 1e-5, ε lr 1e-3; outputs at steps 100 and 1,000).

**What exactly was done** (`scripts/run_hf_constraints_dev.py`: part A without the `platform` allocator, like all CSGM runs; part B with it; cache `hf_constraints_dev_v1/`). Image 403, correct label, HR only for scoring. Every output is back-projected (BP) before scoring.
- **Members** as in Check 13: restarts r0, r1 and the r0-start runs on y shifted by (0, 1), (1, 0), (1, 1) LR pixels (shift members are shifted back).
- **Single** = member r0 (the restart kept by the CSGM rule for this image). **Ensemble** = the 5 members averaged, uniformly and weighted by each member's LR fit loss (wₖ ∝ exp(−(Lₖ − L_min)/T), T = std of the losses), then BP; weighted is the "Ens-W" recipe.
- **Variants:** base (no constraint) · **A** (CSGM with the colour penalty) · IADrift without anchor (Check 14's runs) · **B** (anchored IADrift from the base members) · **A + B** (anchored IADrift from the A members).
- Bicubic and bicubic + BP are shown for comparison (also as the first panel of every figure).

In [ ]:
# Check 16: high-frequency constraints A (colour penalty in CSGM) and B (frequency anchor in IADrift), image 403.
HFC = DATA_DIR / "hf_constraints_dev_v1"
I16 = 403
MEM16 = ["r0", "r1", "shift01", "shift10", "shift11"]
hr16, y16 = images[I16]["hr"], images[I16]["y"]


def ld(path, key):
    with np.load(path) as d:
        return d[key], (json.loads(str(d["metrics_json"])) if "metrics_json" in d.files else {})


def unshift(x, m):
    sy, sx = SH14[m]
    return np.roll(x, (-4 * sy, -4 * sx), axis=(0, 1))


def weights(losses):
    L = np.asarray(losses, float)
    w = np.exp(-(L - L.min()) / max(L.std(), 1e-6))
    return w / w.sum()


# Member outputs (original coordinates) and their start fit losses.
base = {m: unshift(ld((CSGM_CACHE / f"img{I16:05d}_oracle_r{1 if m == 'r1' else 0}.npz") if m in ("r0", "r1")
                     else DEMO_DIR / f"D_shift{m[-2]}{m[-1]}_r0.npz", "x_hat")[0], m) for m in MEM16}
base_L = [e13[I16]["loss"][MEMB14[m]] for m in MEM16]
A = {m: unshift(ld(HFC / f"A_lam0.1_{m}.npz", "x_hat")[0], m) for m in MEM16}
A_L = [ld(HFC / f"A_lam0.1_{m}.npz", "x_hat")[1]["fit_loss"] for m in MEM16]
A03 = ld(HFC / "A_lam0.3_r0.npz", "x_hat")[0]
iad = {ck: {m: unshift(ld(IAD_ENS / f"img{I16:05d}_{m}.npz", f"x_step{ck}")[0], m) for m in MEM16} for ck in (100, 1000)}
B = {ck: {m: unshift(ld(HFC / f"B_from_base_{m}.npz", f"x_step{ck}")[0], m) for m in MEM16} for ck in (100, 1000)}
AB = {ck: {m: unshift(ld(HFC / f"B_from_A_{m}.npz", f"x_step{ck}")[0], m) for m in MEM16} for ck in (100, 1000)}
wb, wa = weights(base_L), weights(A_L)

variants = {  # name -> (members dict, weights for the weighted ensemble)
    "CSGM": (base, wb), "CSGM + A": (A, wa),
    "CSGM → IADrift @100": (iad[100], wb), "CSGM → IADrift @1000": (iad[1000], wb),
    "CSGM → IADrift + B @100": (B[100], wb), "CSGM → IADrift + B @1000": (B[1000], wb),
    "CSGM + A → IADrift + B @100": (AB[100], wa), "CSGM + A → IADrift + B @1000": (AB[1000], wa),
}
r16 = {}
for n, (mem, w) in variants.items():
    for kind, x in (("single (r0)", mem["r0"]), ("ensemble, uniform", np.mean([mem[m] for m in MEM16], axis=0)),
                    ("ensemble, weighted (Ens-W recipe)", sum(wk * mem[m] for wk, m in zip(w, MEM16)))):
        xb = backproject(x, y16)
        mm = evaluate(xb, hr16, y16)
        mm["blockiness"] = blockiness(xb)
        mm["chroma_hf"] = float(np.sum(np.asarray(band_filter(np.stack([-0.1687 * xb[..., 0] - 0.3313 * xb[..., 1] + 0.5 * xb[..., 2],
                                                                        0.5 * xb[..., 0] - 0.4187 * xb[..., 1] - 0.0813 * xb[..., 2]], -1),
                                                               LR_NYQUIST, np.inf)) ** 2))
        r16[(n, kind)] = (xb, mm)
xb = backproject(A03, y16)
mm = evaluate(xb, hr16, y16); mm["blockiness"] = blockiness(xb); mm["chroma_hf"] = np.nan
r16[("CSGM + A (λ_A = 0.3)", "single (r0)")] = (xb, mm)
refs16 = {n: (images[I16]["outputs"][n], {**images[I16]["metrics"][n], "blockiness": blockiness(images[I16]["outputs"][n])})
          for n in ("bicubic", "bicubic + BP")}

for kind in ("single (r0)", "ensemble, weighted (Ens-W recipe)", "ensemble, uniform"):
    ref = r16[("CSGM", kind)][1]
    lines = ["| Method (all + BP) | PSNR | LPIPS ↓ | blockiness | HF colour energy | SNR high | SNR very high | Δ PSNR vs CSGM | Δ LPIPS vs CSGM |",
             "|---|---:|---:|---:|---:|---:|---:|---:|---:|"]
    for n, (_, m) in list(refs16.items()) + [(n2, r16[(n2, kd)]) for (n2, kd) in r16 if kd == kind]:
        lines.append(f"| {n} | {m['psnr']:.2f} | {m['lpips']:.3f} | {m['blockiness']:.3f} | {m.get('chroma_hf', np.nan):.0f} | "
                     f"{m['band_snr_db']['high']:.2f} | {m['band_snr_db']['very high']:.2f} | {m['psnr'] - ref['psnr']:+.2f} | "
                     f"{m['lpips'] - ref['lpips']:+.3f} |")
    display(Markdown(f"**Check 16, image 403, {kind}** (Δ vs CSGM + BP of the same kind)\n\n" + "\n".join(lines)))

for kind, title in (("single (r0)", "single run"), ("ensemble, weighted (Ens-W recipe)", "weighted 5-run ensemble (Ens-W)")):
    shown = [(f"{n} + BP", *r16[(n, kind)]) for n in ("CSGM", "CSGM + A", "CSGM → IADrift @1000", "CSGM → IADrift + B @1000",
                                                     "CSGM → IADrift + B @100", "CSGM + A → IADrift + B @100")]
    side_by_side(hr16, f"Original HR, development image {I16}\ncrane: {title}", shown)


### Check 16 notes (image 403 only; untuned; exploratory)

**Idea A (penalize high-frequency colour) helps the single run clearly and the ensemble a little.**
- Single CSGM + BP: **+2.07 dB, LPIPS −0.028** (21.79 / 0.435 → 23.86 / 0.407). High-frequency colour energy drops from 646 to 62 (−90%), the colour confetti tiles disappear (crop), and the fit even improves (LR loss 80 → 61). A stronger λ_A = 0.3 gives more PSNR (+2.68 dB) but loses the LPIPS gain, i.e. it starts to over-smooth.
- Weighted ensemble (Ens-W): +0.70 dB, LPIPS unchanged (0.361); uniform ensemble: +0.40 dB, LPIPS slightly worse (+0.025). Averaging already cancels most colour noise (ensemble colour energy is 63 vs 646 for one run), so A has much less left to remove.
- Side effect: **blockiness goes up** (1.18 → 1.36 single): removing colour noise exposes the brightness tile seams more; A fixes colour artifacts, not the tile grid.

**Idea B (frequency anchor for IADrift) does exactly what it should: it stops IADrift's damage, but adds no gain on its own.**
- Without the anchor, IADrift @1000 + BP collapses (16.16 dB, LPIPS 0.686, colour energy 2,409: the speckle). With the anchor it stays at the CSGM level (**21.71 dB / 0.430**, colour energy 653): the high band is kept, the measured bands adapt.
- In the ensemble, anchored IADrift @1000 gives the **best LPIPS of the plain-CSGM variants** (weighted 0.351 vs 0.361; uniform 0.319 vs 0.329), small but consistent, at about the same PSNR.

**A + B combine well on this image.** Anchored IADrift started from the colour-penalized CSGM:
- single: **23.71–23.96 dB, LPIPS 0.398–0.402** (vs 21.79 / 0.435 for CSGM + BP): the best single-run result in this check, better than A alone on LPIPS;
- weighted ensemble @1000: **24.93 dB, LPIPS 0.348** (vs 24.29 / 0.361 for Ens-W), the best ensemble LPIPS here except the uniform anchored-IADrift ensemble (0.319, 24.64 dB).

**Summary of contributions (image 403, after BP).**

| | single run: Δ PSNR / Δ LPIPS | weighted ensemble: Δ PSNR / Δ LPIPS |
|---|---|---|
| + A (colour penalty) | +2.07 / −0.028 | +0.70 / 0.000 |
| + IADrift @1000 (no anchor) | −5.63 / +0.251 | −3.89 / +0.154 |
| + IADrift @1000 + B (anchor) | −0.09 / −0.004 | −0.06 / −0.009 |
| + A + IADrift @1000 + B | +1.91 / −0.037 | +0.64 / −0.013 |

**Caveats.** One image (the worst-case crane), one λ per idea, chosen by penalty size, not tuned. **Worth confirming on the 6 development images,** where A is about 6 min of GPU per image (5 members) and B about 4 min.

---
# TEST SET: CSGM benchmark with back-projection, LPIPS and blockiness

> ⚠️ **Test data. Read-only.** This section only **loads** results computed in `notebooks/03_csgm_test_analysis.ipynb` (Part 1 = the CSGM benchmark, Part 2 = back-projection and metrics; cache `~/work/datasets/ImageNetV2/csgm_test_postproc_v1/` and the frozen CSGM test arrays). Nothing is recomputed or optimized here. **These numbers must not be used to choose or tune any method**, in particular the IADrift variant, which is frozen on development images in `docs/PROTOCOL.md`.

**What these results are**
- **Images:** the fixed sample of 100 held-out **test** images, 100 distinct classes (`sr4_test_sample_100_seed2028.npz`). Test image 1704 was used for smoke tests earlier (nothing tuned on it).
- **CSGM:** the frozen benchmark (Adam lr 0.02 on ε, 5,000 steps, 2 restarts, lowest final LR loss kept), with the **correct** label and with one random **wrong** label per image.
- **+ BP:** parameter-free back-projection onto {x : A x = y} (conjugate gradients, the same operator as Check 1).
- **Metrics:** PSNR (dB), LPIPS (lower is better), **blockiness** (`freq_metrics.blockiness`: mean pixel jump across the 16-pixel tile borders ÷ mean jump at the other 4-pixel-aligned positions; HR ≈ 1), LR loss, band SNR.
- **Paired statistics:** mean difference with a normal-approximation 95% CI, mean ± 1.96 s/√n (s = sample standard deviation of the per-image differences, divisor n − 1; n = number of images), Wilcoxon p, count of images where A is better; no multiple-comparison correction.

In [ ]:
# TEST SET: load cached results only; nothing is recomputed, nothing is tuned.
from scipy import stats

TEST_POST_DIR = DATA_DIR / "csgm_test_postproc_v1"
TEST_RESULTS = DATA_DIR / "phase1_sr4_csgm_label_test_v2_n100_9e24c78a47ab.jsonl"
test_metrics = {int(k): v for k, v in json.loads((TEST_POST_DIR / "metrics.json").read_text()).items()}
test_images = list(test_metrics)
sample = np.load(DATA_DIR / "sr4_test_sample_100_seed2028.npz")["test_sample_indices"].tolist()
assert test_images == sample, "metrics must cover exactly the fixed test sample, in order"
assert set(test_images) <= set(test_indices.tolist()) and not set(test_images) & set(DEV_IMAGES)

TEST_METHODS = ["bicubic", "bicubic + BP", "CSGM, correct label", "CSGM, correct label + BP",
                "CSGM, wrong label", "CSGM, wrong label + BP"]


def tvals(method, key, band=None):
    return np.array([test_metrics[i][method]["band_snr_db"][band] if band else test_metrics[i][method][key]
                     for i in test_images])


def psnr_of_mean_mse(p):
    return float(-10 * np.log10(np.mean(10 ** (-np.asarray(p) / 10))))


hr_block = np.array([test_metrics[i]["hr_blockiness"] for i in test_images])
lines = ["| Method | Mean PSNR | PSNR (mean MSE) | LPIPS ↓ | Blockiness (≈ 1 = none) | LR loss | "
         + " | ".join(f"SNR {b}" for b in BAND_NAMES) + " |", "|---|---:|---:|---:|---:|---:|" + "---:|" * len(BAND_NAMES)]
for m in TEST_METHODS:
    p = tvals(m, "psnr")
    lines.append(f"| {m} | {p.mean():.2f} | {psnr_of_mean_mse(p):.2f} | {tvals(m, 'lpips').mean():.3f} | "
                 f"{tvals(m, 'blockiness').mean():.3f} | {tvals(m, 'lr_loss').mean():.2f} | "
                 + " | ".join(f"{tvals(m, None, b).mean():.2f}" for b in BAND_NAMES) + " |")
lines.append(f"| HR (reference) | — | — | 0 | {hr_block.mean():.3f} | 0 | " + " | ".join("—" for _ in BAND_NAMES) + " |")
display(Markdown(f"**TEST SET: all methods, mean over {len(test_images)} test images**\n\n" + "\n".join(lines)))

T_PAIRS = [("bicubic + BP", "bicubic"), ("CSGM, correct label + BP", "CSGM, correct label"),
           ("CSGM, correct label + BP", "bicubic + BP"), ("CSGM, correct label", "bicubic"),
           ("CSGM, correct label", "CSGM, wrong label"), ("CSGM, correct label + BP", "CSGM, wrong label + BP")]
for key, title in (("psnr", "HR PSNR (dB; higher is better)"), ("lpips", "LPIPS (lower is better)"),
                   ("blockiness", f"Blockiness (lower is better; HR ≈ {hr_block.mean():.2f})")):
    lower_better = key != "psnr"
    f = "{:+.3f}" if lower_better else "{:+.2f}"
    lines = ["| A − B | Mean diff. | 95% CI | Wilcoxon p | A better |", "|---|---:|---:|---:|---:|"]
    for a, b in T_PAIRS:
        d = tvals(a, key) - tvals(b, key)
        half = 1.96 * d.std(ddof=1) / np.sqrt(len(d))
        better = int(np.sum(d < 0 if lower_better else d > 0))
        lines.append(f"| {a} − {b} | {f.format(d.mean())} | [{f.format(d.mean() - half)}, {f.format(d.mean() + half)}] | "
                     f"{stats.wilcoxon(d).pvalue:.1e} | {better}/{len(d)} |")
    display(Markdown(f"**TEST SET, paired: {title}**\n\n" + "\n".join(lines)))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))
for k, m in enumerate(TEST_METHODS):
    axes[0].scatter(tvals(m, "psnr"), tvals(m, "lpips"), s=10, alpha=0.6, color=f"C{k}", label=m)
axes[0].set(xlabel="HR PSNR (dB)", ylabel="LPIPS (lower is better)", title="TEST: per image, PSNR vs LPIPS")
axes[0].legend(fontsize=7)
axes[1].boxplot([hr_block] + [tvals(m, "blockiness") for m in TEST_METHODS], showfliers=False,
                tick_labels=["HR"] + [m.replace("CSGM, ", "").replace(" label", "") for m in TEST_METHODS])
axes[1].axhline(1.0, color="gray", linestyle="--", linewidth=1)
axes[1].set(ylabel="blockiness (≈ 1 = no tile seams)", title="TEST: tile-seam blockiness")
axes[1].tick_params(axis="x", rotation=30)
fig.tight_layout()
plt.show()

# Visual examples: the first 4 images of the fixed test sample (not chosen by result).
rows = [json.loads(line) for line in TEST_RESULTS.open() if line.strip()]
best_test = {}
for r in rows:
    k = (r["image_index"], r["condition"])
    if k not in best_test or (r["final_loss"], r["restart"]) < (best_test[k]["final_loss"], best_test[k]["restart"]):
        best_test[k] = r
test_arrays = DATA_DIR / (TEST_RESULTS.stem + "_arrays")
for i in test_images[:4]:
    hr = load_hr256(DATA_DIR / index_rows[i]["relative_path"])
    with np.load(test_arrays / best_test[(i, "oracle")]["array_file"]) as data:
        csgm = data["x_hat"]
    with np.load(TEST_POST_DIR / f"img{i:05d}.npz") as data:
        bic_bp, csgm_bp = data["bicubic_BP"], data["CSGM_correct_label_BP"]
    shown = [("bicubic", bicubic_up(lr_all[i]), test_metrics[i]["bicubic"]),
             ("bicubic + BP", bic_bp, test_metrics[i]["bicubic + BP"]),
             ("CSGM, correct label", csgm, test_metrics[i]["CSGM, correct label"]),
             ("CSGM, correct label + BP", csgm_bp, test_metrics[i]["CSGM, correct label + BP"])]
    shown = [(f"{n}\nblockiness {m['blockiness']:.2f}", x, m) for n, x, m in shown]
    side_by_side(hr, f"TEST image {i}\n{class_name(labels[i])}", shown)

### Test-set conclusions (100 test images; details and checks in notebook 03, Part 2)

1. **Back-projection helps every method on all 100 images:** bicubic + BP vs bicubic **+0.56 dB / −0.054 LPIPS**; CSGM + BP vs CSGM **+1.05 dB / −0.080 LPIPS**. The classical baseline to beat is **bicubic + BP (25.57 dB, LPIPS 0.358)**.
2. **PSNR and LPIPS disagree.** CSGM + BP is **2.82 dB below** bicubic + BP (0/100 images) but **better on LPIPS by 0.074** (76/100; p = 2.5e-7). **CSGM + BP has the best LPIPS of all methods (0.284).** This replicates the development result of Check 1.
3. **The 16-pixel tile seams are real on the test set:** blockiness HR 1.01, bicubic (+ BP) 1.01, CSGM 1.37 → 1.26 with BP; CSGM + BP is blockier than bicubic + BP on 93/100 images.
4. **Label:** no PSNR effect (Part 1; also after BP), but a small perceptual one: after BP the correct label gives LPIPS −0.018 (p = 0.03) and blockiness −0.059 (p = 4e-4). Uncorrected for multiple comparisons; treat as tentative.

---
# TEST SET: IADrift, pre-registered analysis

> ⚠️ **Test data. Read-only.** Loads the cached results of `notebooks/03_csgm_test_analysis.ipynb`, Part 3 (`~/work/datasets/ImageNetV2/iadrift_test_postproc_v1/`). The IADrift test run followed `docs/PROTOCOL.md` (frozen 2026-10-05): joint θ + ε from the CSGM test solutions, θ lr 1e-5, ε lr 1e-3, outputs at steps 100 and 1,000, raw and + BP. Nothing here is recomputed or tuned.

In [ ]:
# TEST SET, IADrift: load the Part 3 results of notebook 03 (read-only).
IAD_TEST_DIR = DATA_DIR / "iadrift_test_postproc_v1"
iad_test = {int(k): v for k, v in json.loads((IAD_TEST_DIR / "metrics.json").read_text()).items()}
assert list(iad_test) == test_images, "IADrift test metrics must cover the same fixed test sample, in order"
primary = json.loads((IAD_TEST_DIR / "primary.json").read_text())

lines = ["| # | Comparison (A − B) | Metric | Mean diff. | 95% CI | Wilcoxon p | **Holm p** | A better |",
         "|---:|---|---|---:|---:|---:|---:|---:|"]
for n, r in enumerate(primary, start=1):
    f = "{:+.3f}" if r["metric"] == "lpips" else "{:+.2f}"
    lines.append(f"| {n} | {r['comparison']} | {'LPIPS (lower better)' if r['metric'] == 'lpips' else 'PSNR (dB)'} | "
                 f"{f.format(r['mean'])} | [{f.format(r['lo'])}, {f.format(r['hi'])}] | {r['p']:.1e} | **{r['holm_p']:.1e}** | "
                 f"{int(r['better'])}/{int(r['n'])} |")
display(Markdown("**TEST SET, PRIMARY ANALYSIS (pre-registered): 100 test images, correct label**\n\n" + "\n".join(lines)))


def tv_all(method, key):
    src = iad_test if method.startswith("IADrift") else test_metrics
    return np.array([src[i][method][key] for i in test_images])


rows_ = ["bicubic + BP", "CSGM, correct label", "CSGM, correct label + BP",
         "IADrift @100, correct label", "IADrift @100, correct label + BP",
         "IADrift @1000, correct label", "IADrift @1000, correct label + BP",
         "CSGM, wrong label + BP", "IADrift @100, wrong label + BP", "IADrift @1000, wrong label + BP"]
lines = ["| Method | Mean PSNR | PSNR (mean MSE) | LPIPS ↓ | Blockiness | LR loss |", "|---|---:|---:|---:|---:|---:|"]
for m in rows_:
    p = tv_all(m, "psnr")
    lines.append(f"| {m} | {p.mean():.2f} | {psnr_of_mean_mse(p):.2f} | {tv_all(m, 'lpips').mean():.3f} | "
                 f"{tv_all(m, 'blockiness').mean():.3f} | {tv_all(m, 'lr_loss').mean():.2f} |")
display(Markdown("**TEST SET: CSGM vs IADrift, mean over 100 test images** (full table in notebook 03, Part 3)\n\n" + "\n".join(lines)))

IAD_TEST_ARRAYS = DATA_DIR / "phase2_sr4_iadrift_test_v1_n100_00ca59364e18_arrays"
for i in test_images[:2]:
    hr = load_hr256(DATA_DIR / index_rows[i]["relative_path"])
    with np.load(TEST_POST_DIR / f"img{i:05d}.npz") as data:
        csgm_bp = data["CSGM_correct_label_BP"]
    with np.load(IAD_TEST_DIR / f"img{i:05d}.npz") as data:
        iad_bp = {s: data[f"bp_oracle_step{s}"] for s in (100, 1000)}
    shown = [("CSGM + BP", csgm_bp, test_metrics[i]["CSGM, correct label + BP"]),
             ("IADrift @100 + BP", iad_bp[100], iad_test[i]["IADrift @100, correct label + BP"]),
             ("IADrift @1000 + BP", iad_bp[1000], iad_test[i]["IADrift @1000, correct label + BP"])]
    side_by_side(hr, f"TEST image {i}\n{class_name(labels[i])}", shown)

### Test-set conclusions, IADrift (details: notebook 03, Part 3)

- **Primary (Holm-corrected, correct label):** IADrift @1000 + BP is worse than CSGM + BP on **PSNR (−2.92 dB, 0/100)** and **LPIPS (+0.082)**. IADrift @100 + BP is **−0.12 dB** on PSNR (33/100) but **better on LPIPS (−0.008, 72/100)**. All four tests are significant.
- **IADrift @100 + BP has the best mean LPIPS of all methods (0.276)**, but every IADrift variant stays far below bicubic + BP on PSNR (−2.9 dB or more).
- Without BP, early-stopped IADrift beats CSGM clearly (+0.28 dB, LPIPS −0.028); BP removes most of that, because the gain is mainly measurement fit (as in Check 1).
- Long adaptation removes the 16-pixel tile seams (blockiness 1.04 ≈ HR) but adds high-frequency noise (very-high-band SNR −8.2 dB): the ill-posedness pattern of Checks 2 and 6, now confirmed on 100 test images.

---
# Summary (final, 2026-10-06)

All experiments finished.

**Development images** (Checks 1–7):

| Finding | Evidence |
|---|---|
| Fusion's PSNR gain is measurement consistency, not the prior | Check 1: bicubic + BP + fusion beats IADrift + BP + fusion on 4/4 |
| **Bicubic + BP is the PSNR baseline to beat** (25.43 dB) | Check 1 |
| **On LPIPS the prior helps:** CSGM + BP is best (0.328 vs bicubic + BP 0.429, 4/4) | Check 1 |
| Adapting θ (IADrift / θ-only) fits y better but worsens PSNR **and** LPIPS | Checks 1–2 |
| 16-pixel patch-grid artifacts are already in the generator's own samples | Check 3 |
| CSGM inflates ε far off the Gaussian shell, but forcing it back (sphere projection) hurts | Checks 3–4 |
| Frozen-generator representation ceiling on real images (24.4 dB) is below bicubic | Check 5 R1 |
| In range, CSGM works well if the noise labels are right (25.9 dB, LPIPS 0.12); a wrong draw costs about 4 dB | Check 5 R3 |
| **With HR known, IAGAN-like adaptation reconstructs almost perfectly** (40.1 dB, LPIPS 0.03, no patch grid) vs CSGM-HR 24.4 dB: the SR failure is ill-posedness, not representation | Check 6 |
| **Noise-label search (choice by LR loss only)** improves CSGM: LR loss 50 → 30, PSNR +0.55 dB, LPIPS −0.04 (mixed per image); LR loss predicts HR quality well (Spearman −0.85 to −0.94) | Check 7 |
| **Noise labels z are the dominant latent:** z explains 0.56–0.82 of style/colour/spectrum variation and 4× more layout variation than ε; one z coordinate changes the image 8.5× more than one ε patch, globally | Check 8 |
| **z can be optimized only as a soft relaxation** (rounding destroys it). Joint (ε, soft z) from scratch beats CSGM raw (LR loss 11 vs 50, PSNR +0.62, LPIPS −0.045, ε closer to Gaussian) at equal cost, but not after BP; fine-tuning z from CSGM = more ε steps | Check 9 |
| **Late z swaps impossible** (0/2,016 single swaps help at the CSGM solution, any image); **early discrete swaps** (steps 100–1,000) beat the same-code control on LPIPS in 7/8 runs and **keep the gain after BP** (+0.30 dB, LPIPS −0.024), but tie/lose vs the saved CSGM + BP: n = 4 too small | Check 10 |
| **Better z-start + IADrift @100 don't stack:** after BP, CSGM→IAD@100 ≈ CSGM (−0.10 dB / −0.002); search→IAD@100 no better than search alone; early swaps worse on LPIPS (2 new images). Best dev pipeline stays noise-label search + BP (+0.24 dB / −0.013, mixed) | Check 11 (post-hoc) |
| **One-image demos (403):** texture-strength knob (α ≈ 0.5), restart/search averaging, a tile-seam penalty and a 4-shift ensemble **all beat CSGM + BP on PSNR and LPIPS** (best: shift ensemble 24.47 dB / 0.338 vs 21.79 / 0.435); one image only | Check 12 (exploratory) |
| **Combinations (403):** C + D and C + B over-smooth (PSNR ≈ bicubic, LPIPS worse than D); A on averaged bases ≈ no gain. The ideas are substitutes; D alone stays best (24.47 dB / 0.338) | Check 12 E (exploratory) |
| **Posterior-mean ensemble (6 dev images):** averaging 5 CSGM runs (restarts + shifts) + BP gives +2.36 dB on 6/6; weighting members by LR loss (Ens-W) gives +1.60 dB (6/6) **and** LPIPS −0.029 (5/6); with search members (4 images) E4w LPIPS −0.072. Plain averaging can ghost when one member fits badly | Check 13 (exploratory) |
| **IADrift ensembles (6 dev images):** averaging 5 IADrift @1000 runs rescues them (+3.95 dB / −0.103 LPIPS vs one run; +0.55 dB vs CSGM + BP) but IADrift @100 ensembles ≈ CSGM ensembles (±0.2 dB / ±0.01 LPIPS): the ensemble matters, θ adaptation doesn't | Check 14 (exploratory) |
| **An image in place of ε is ignored:** with bicubic(y) or even the HR photo as ε (any mix with noise), the output is an unrelated scene set by class + z (4–11 dB, LPIPS 0.72–0.97); strongest evidence that z, not ε, decides the scene | Check 15 (exploratory) |
| **High-frequency constraints (403 only):** colour penalty A: single +2.07 dB / LPIPS −0.028 (confetti gone), ensemble +0.70 dB; anchor B stops IADrift's collapse (@1000: 16.2 → 21.7 dB); A + anchored IADrift best single run (23.7–24.0 dB / 0.40) | Check 16 (exploratory) |

**Test set** (100 images; read-only section above):
- CSGM + BP has the best LPIPS (0.284) but is 2.82 dB below bicubic + BP on PSNR (0/100) and shows tile seams (blockiness 1.26 vs 1.01).
- Back-projection helps every method on all 100 images; bicubic + BP (25.57 dB) is the PSNR baseline to beat.
- The label affects LPIPS and blockiness slightly, not PSNR.
- **IADrift (pre-registered, Holm-corrected):** @1000 + BP is worse than CSGM + BP on PSNR (−2.92 dB) and LPIPS (+0.082); @100 + BP is −0.12 dB on PSNR but better on LPIPS (−0.008, 72/100) and has the best LPIPS of all methods (0.276). Long adaptation removes the tile seams but adds high-frequency noise.